<a href="https://colab.research.google.com/github/caldasvictor10/RO-Py/blob/Vers%C3%A3o-01/RO_Py_VSCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""ROPy_PEA_26_27_V3.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/11wP11vbJEPx2dx3L9dkaYHBLkXGR9ory

# MODELO ROPy
"""

# -*- coding: utf-8 -*-
"""ROPy_PEA_26_27.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/10mq5y7sYhXGwQ1k40adA3zAC6VSCrxtH

# MODELO ROPy
"""

# -*- coding: utf-8 -*-
"""VINCERE_OPTIMA_PEA_V1.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1SIZ2k_CoK8D5VassNA6cmeGjxgFLmmSk

# MODELO
"""

# -*- coding: utf-8 -*-
"""VINCERE_OPTIMA_PVV12.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1tQLDh7dQb9f_Rlc4dq4cvOkoVq9_6ej1

# MODELO
"""

# -*- coding: utf-8 -*-
"""VINCERE_OPTIMA_PVV12.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1tQLDh7dQb9f_Rlc4dq4cvOkoVq9_6ej1

# MODELO
"""

# -*- coding: utf-8 -*-
"""VINCERE_OPTIMA_PVV11.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1u7WbuRuIFrQxEmYeIJFnQ26w_6WGBvX4

# MODELO
"""

# -*- coding: utf-8 -*-
"""VINCERE_OPTIMA_PVV6.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1rO7foCZMiO5bKbSIbON0stcB6-aDKWS4

# MODELO
"""

# -*- coding: utf-8 -*-
"""VINCERE_OPTIMA_PVV6.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1rO7foCZMiO5bKbSIbON0stcB6-aDKWS4

# MODELO
"""

# -*- coding: utf-8 -*-
"""VINCERE_OPTIMA_PVV1.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1GTZ14DZE9YEVOGaCTEjFkFcvWRxoowj3

# MODELO
"""

# -*- coding: utf-8 -*-
"""VINCERE_OPTIMA_V8.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1sduN8vH6OIhJ-UAxYG4VBexC3Q7pm4PG

# MODELO
"""

# -*- coding: utf-8 -*-
"""
VINCERE OPTIMA — Otimizador Global de Carga Metálica v9
========================================================
Modelo de programação linear (PuLP/CBC) para otimização simultânea
da mistura de sinterização (MS1E2, MS3) e carga dos Altos Fornos (AF2, AF3).

Escopo do modelo:
  · Qualidade e custo da sinterização (B2, MgO, granulometria, carga energética)
  · Balanço físico do pátio de sínter (estocagem / retomada)
  · Balanço de ferro, basicidade e slag rate dos Altos Fornos
  · Carga direta de nobres (NPO, Pelota, Briquete) e carbonosos (Coque, PCI)
  · Sazonalidade de materiais higroscópicos (PERIODO: CHUVA / SECA)
  · Análise VIU dupla (marginal + por lote) com relatório executivo Excel
  · Exportação de base para Power BI
  · [v9] Penalização da mistura de finos de NPO por degradação de ganga:
         delta em pontos percentuais por óxido (FET, SiO2, CaO, MgO, Al2O3, Mn, P)
         e penalidade de PPC — lidos da aba CALIBRACAO, coluna GLOBAL.

Entrada:  planilha Excel com abas MATERIAS_PRIMAS, PROCESSO_MS, PROCESSO_AF,
          ALFAS_GUSA_AF, PERFORMANCE, KPIS_MS, KPIS_AF, CALIBRACAO.
Saída:    mesma planilha preenchida com os resultados + aba VIU_ANALISE.
"""

# @title ⚙️ PAINEL DE OTIMIZAÇÃO GLOBAL AF/MS { display-mode: "form" }

import sys
import os
import pandas as pd
import pulp as pl
import unicodedata

# ======================================================
# 0. UPLOAD E CARREGAMENTO
# ======================================================
# Detecta o ambiente automaticamente
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import files
    print("Faça o upload da planilha 'INPUT_MASTER_AF.xlsx'")
    uploaded = files.upload()
    nome_arquivo = list(uploaded.keys())[0]
else:
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    nome_arquivo = filedialog.askopenfilename(
        title="Selecione o INPUT_MASTER_AF.xlsx",
        filetypes=[("Excel files", "*.xlsx *.xls")]
    )
    if not nome_arquivo:
        raise FileNotFoundError("Nenhum arquivo selecionado.")
    print(f"✅ Arquivo encontrado: {nome_arquivo}")

def carregar_e_limpar_aba(nome_aba):
    # 1. Carrega a aba
    df = pd.read_excel(nome_arquivo, sheet_name=nome_aba)

    # 2. Limpeza de nomes de colunas (Maiúsculas, sem espaços e sem acentos)
    novas_colunas = []
    for col in df.columns:
        # Remove acentos
        nfd_form = unicodedata.normalize('NFD', str(col))
        col_limpa = "".join([c for c in nfd_form if unicodedata.category(c) != 'Mn'])
        # Formata: Maiúsculo, sem espaço e sem underline
        col_limpa = col_limpa.upper().replace("_", "").replace(" ", "").strip()
        novas_colunas.append(col_limpa)

    df.columns = novas_colunas

    # IMPORTANTE: Não renomeamos a primeira coluna aqui para não estragar a aba MATERIAS_PRIMAS
    return df

# --- FUNÇÕES DE SUPORTE PARA LEITURA ROBUSTA ---
def clean_name(txt):
    if pd.isna(txt): return ""
    # Remove acentos
    nfd_form = unicodedata.normalize('NFD', str(txt))
    txt_limpo = "".join([c for c in nfd_form if unicodedata.category(c) != 'Mn'])
    # Remove underscores, espaços e traços para comparação interna
    return txt_limpo.upper().replace("_", "").replace(" ", "").replace("-", "").strip()

def buscar_valor(df, param_name, col_name):
    target_param = clean_name(param_name)
    target_col = clean_name(col_name)

    # 1. Identifica a coluna real do AF/Máquina (AF2, AF3, etc)
    try:
        col_real = [c for c in df.columns if clean_name(c) == target_col][0]
    except IndexError:
        raise ValueError(f"ERRO: Coluna '{col_name}' não encontrada no Excel!")

    # 2. Procura o parâmetro nas 3 primeiras colunas da aba
    for i in range(min(3, len(df.columns))):
        col_ref = df.columns[i]
        mask = df[col_ref].astype(str).apply(clean_name) == target_param
        if mask.any():
            valor = df.loc[mask, col_real].iloc[0]
            return float(valor)

    raise ValueError(f"ERRO CRÍTICO: Parâmetro '{param_name}' não encontrado. Verifique o Excel!")

def buscar_linha(df, parametro):
    """
    Busca um item de forma persistente:
    1. Tenta na coluna 'PARAMETRO'
    2. Se não achar, tenta na primeira coluna
    3. Se não achar, tenta na segunda coluna
    """
    # Normaliza o que estamos buscando
    def limpar(txt):
        nfd = unicodedata.normalize('NFD', str(txt))
        return "".join([c for c in nfd if unicodedata.category(c) != 'Mn']).upper().replace("_", "").replace(" ", "").strip()

    procurado = limpar(parametro)

    # Lista de colunas onde vale a pena procurar (PARAMETRO, Coluna 0, Coluna 1)
    colunas_candidatas = []
    if 'PARAMETRO' in df.columns:
        colunas_candidatas.append('PARAMETRO')

    colunas_candidatas.append(df.columns[0]) # Geralmente 'APLICACAO' ou 'ELEMENTO'
    if len(df.columns) > 1:
        colunas_candidatas.append(df.columns[1]) # Geralmente 'PARAMETRO' real

    # Tenta em cada coluna até achar
    for col in colunas_candidatas:
        col_limpa = df[col].astype(str).apply(limpar)
        if procurado in col_limpa.values:
            return df[col_limpa == procurado].iloc[0]

    # Se percorreu tudo e não achou
    raise ValueError(f"ERRO: O item '{parametro}' não foi encontrado em nenhuma das colunas iniciais da aba.")


# ======================================================
# 1. NORMALIZAÇÃO DE DADOS
# ======================================================

# --- PRIMEIRO: Carregar as tabelas básicas ---
df_raw = carregar_e_limpar_aba("MATERIAS_PRIMAS")

# Blindagem para vírgula decimal e espaços em branco
df_raw = df_raw.replace(",", ".", regex=True)

# Garante que a coluna MP seja texto limpo para o índice
df_raw["MP"] = df_raw["MP"].astype(str).str.upper().str.strip()

# --- SEGUNDO: Carregar as abas de processo ---
df_ms = carregar_e_limpar_aba("PROCESSO_MS")
df_af = carregar_e_limpar_aba("PROCESSO_AF")
df_perf = carregar_e_limpar_aba("PERFORMANCE")
df_cal = carregar_e_limpar_aba("CALIBRACAO")

# Listas de apoio
mps = df_raw["MP"].tolist()
MAQUINAS = ["MS1E2", "MS3"]
AFS = ["AF2", "AF3"]

# --- TERCEIRO: Definir o índice ---
df_idx = df_raw.set_index("MP")
# Flag de higroscopicidade — coluna HIGRO na aba MATERIAS_PRIMAS
# 1 = higroscópico (restrição ativa no período chuvoso), 0 = normal
HIGRO = df_idx["HIGRO"].fillna(0).astype(int).to_dict()
mps_higro = [mp for mp in mps if HIGRO.get(mp, 0) == 1]

RN_MS_LIMITS = {ms: (float(buscar_linha(df_ms, "RN_MIN")[ms]),
                     float(buscar_linha(df_ms, "RN_MAX")[ms])) for ms in MAQUINAS}

# Dicionários químicos (%)
quimica = {el: df_idx[el].to_dict() for el in ["FET", "SIO2", "CAO", "MGO", "AL2O3", "MN", "P"]}

# Dicionários de granulometria da sinterização
G_635 = (df_idx[">6,35"].fillna(0) / 100).to_dict()
G_100 = (df_idx[">1,00"].fillna(0) / 100).to_dict()
G_105 = (df_idx["<0,105"].fillna(0) / 100).to_dict()

Umid  = (df_idx["UMIDADE"] / 100).to_dict()
PPC   = (df_idx["PPC"] / 100).to_dict()
Finos = (df_idx["<10,00"].fillna(0) / 100).to_dict()
Preco = df_idx["PRECOCIFBU"].to_dict()
DispMin = df_idx["DISPMIN"].to_dict()
DispMax = df_idx["DISPMAX"].to_dict()
Aplicacao = df_idx["APLICACAO"].astype(str).str.upper().str.strip().to_dict()
tipo_mp = df_idx["TIPO"].astype(str).str.upper().str.strip().to_dict()

# ======================================================
# 2. LEITURA DE PARÂMETROS E ALFAS
# ======================================================

df_alfa = carregar_e_limpar_aba("ALFAS_GUSA_AF")

# MS Params ( MgO, Mn, P e Carga Energética) - PADRONIZADO E ROBUSTO
REND_SINTER = {ms: buscar_valor(df_ms, "RENDIMENTO_SINTER", ms) for ms in MAQUINAS}
PROD_FIXA_MS = {ms: buscar_valor(df_ms, "PROD_FIXA", ms) for ms in MAQUINAS}
B2_MS_LIMITS = {ms: (buscar_valor(df_ms, "B2_MIN", ms), buscar_valor(df_ms, "B2_MAX", ms)) for ms in MAQUINAS}
MGO_MS_LIMITS = {ms: (buscar_valor(df_ms, "MGO_MIN", ms), buscar_valor(df_ms, "MGO_MAX", ms)) for ms in MAQUINAS}
P_MS_MAX = {ms: buscar_valor(df_ms, "P_MAX", ms) for ms in MAQUINAS}
MN_MS_MIN = {ms: buscar_valor(df_ms, "MN_MIN", ms) for ms in MAQUINAS}
ALVO_CARGA_ENERGETICA_MS = {ms: buscar_valor(df_ms, "CARGA_ENERGETICA", ms) for ms in MAQUINAS}

PROD_ALVO_GUSA = {af: float(buscar_linha(df_af, "PROD_TARGET")[af]) for af in AFS}

# Basicidades (Garantindo que pegamos Min e Max)
B2_AF_LIMITS = {af: (float(buscar_linha(df_af, "B2_MIN")[af]), float(buscar_linha(df_af, "B2_MAX")[af])) for af in AFS}
B3_AF_LIMITS = {af: (float(buscar_linha(df_af, "B3_MIN")[af]), float(buscar_linha(df_af, "B3_MAX")[af])) for af in AFS}
SLAG_RATE_MAX = {af: buscar_valor(df_af, "SLAG_RATE_MAX", af) for af in AFS}
SLAG_RATE_TARGET = {af: buscar_valor(df_af, "SLAG_RATE_TARGET", af) for af in AFS}
FATOR_OXIDOS = {af: buscar_valor(df_af, "FATOR_OXIDOS", af) for af in AFS}
MN_AF_MIN = {af: float(buscar_linha(df_af, "MN_MIN")[af]) for af in AFS}
P_AF_MAX = {af: float(buscar_linha(df_af, "P_MAX")[af]) for af in AFS}

# Alvos de Carbonosos (Coque e PCI)
COQUE_RATE_TARGET = {af: buscar_valor(df_af, "COQUE_RATE", af) for af in AFS}
PCI_RATE_TARGET = {af: buscar_valor(df_af, "PCI_RATE", af) for af in AFS}

# Alfas e Fatores de Massa
ALFA = {af: {el: float(buscar_linha(df_alfa, el)[f"ALFAREDUCAO{af}"]) for el in ["FE", "SI", "MN", "P"]} for af in AFS}

# Fração >1mm, Limite Carepa, ponto de cal fina e Proporção de Consumo
RN_MS_LIMITS = {ms: (float(buscar_linha(df_ms, "RN_MIN")[ms]), float(buscar_linha(df_ms, "RN_MAX")[ms])) for ms in MAQUINAS}
FRAC_GT1_MIN = {ms: float(buscar_linha(df_ms, "FRAC_GT1_MIN")[ms]) for ms in MAQUINAS}
CAREPA_MAX_KG_T = {ms: float(buscar_linha(df_ms, "CAREPA_MAX_KG_T")[ms]) for ms in MAQUINAS}
CAL_FINA_KG_T = {ms: float(buscar_linha(df_ms, "CAL_FINA")[ms]) for ms in MAQUINAS}
PROP_TOL_VAL = float(buscar_linha(df_ms, "PROP_TOL")["GLOBAL"])
# Sazonalidade — lê PERIODO da aba PERFORMANCE (coluna GLOBAL)
# e seleciona o limite de higroscópicos correspondente de PROCESSO_MS
PERIODO = str(buscar_linha(df_perf, "PERIODO")["GLOBAL"]).strip().upper()
_lim_chuva = float(buscar_linha(df_ms, "HIGRO_LIM_CHUVA")["GLOBAL"])
_lim_seca  = float(buscar_linha(df_ms, "HIGRO_LIM_SECA")["GLOBAL"])
HIGRO_LIM_BU = _lim_chuva if PERIODO == "CHUVA" else _lim_seca
print(f"  ► Período: {PERIODO} | Higroscópicos: {mps_higro} | Limite: {HIGRO_LIM_BU:,.0f} t B.U.")

# Limites de composição da carga metálica dos Altos Fornos
NPO_MAX_PERC    = {af: buscar_valor(df_af, "NPO_MAX", af) / 100 for af in AFS}
NPO_MIN_PERC    = {af: buscar_valor(df_af, "NPO_MIN", af) / 100 for af in AFS}
PELOTA_MIN_PERC = {af: buscar_valor(df_af, "PELOTA_MIN", af) / 100 for af in AFS}
SINTER_MAX_PERC = {af: buscar_valor(df_af, "SINTER_MAX", af) / 100 for af in AFS}

# Quartzo Min e Max (Kg/t)
QZ_LIMITS = {af: (float(buscar_linha(df_af, "QZ_MIN")[af]), float(buscar_linha(df_af, "QZ_MAX")[af])) for af in AFS}

LIM_MUSA_BEMISA = {ms: float(buscar_linha(df_ms, "LIM_MUSA_BEMISA")[ms]) for ms in MAQUINAS}
LIM_BEMISA = {ms: float(buscar_linha(df_ms, "LIM_BEMISA")[ms]) for ms in MAQUINAS}

# CUSTO OPERACIONAL VARIÁVEL (leitura do Excel)
# US$/t BSR por máquina (MS1E2, MS3) — aplica-se a todo sínter produzido
CUSTO_OP_VAR_MS = {ms: buscar_valor(df_ms, "CUSTO_OP_VAR_MS", ms) for ms in MAQUINAS}
# US$/t gusa por forno (AF2, AF3) — aplica-se a toda produção de gusa
CUSTO_OP_VAR_AF = {af: float(buscar_linha(df_af, "CUSTO_OP_VAR_AF")[af]) for af in AFS}

# Perda física de matérias-primas na mistura parcial (por máquina)
# Aplica-se a todos os materiais da MPA, EXCETO Carga Energética (carbonosos)
PERDA_FISICA = {ms: buscar_valor(df_ms, "PERDA_FISICA", ms) for ms in MAQUINAS}

# ======================================================
# 2.6 PENALIDADES DA MISTURA DE FINOS DE NPO (ABA CALIBRACAO)
# ======================================================
# Delta em pontos percentuais aplicado à MISTURA TOTAL de finos de NPO,
# independente do mix de NPOs escolhido pelo solver.
# A penalidade representa a degradação de ganga ao peneirar NPOs:
#   Ex: PENAL_FINOS_SIO2 =  2.0 → fino entra com SiO2_NPO + 2.0 pp
#   Ex: PENAL_FINOS_FET  = -1.5 → fino entra com FET_NPO  - 1.5 pp
# Zero = sem penalidade (retrocompatível com versões anteriores).
# PENAL_PPC = pontos adicionais de PPC nos finos (afeta balanço de massa BSR).

_OXIDOS_PENAL = ["FET", "SIO2", "CAO", "MGO", "AL2O3", "MN", "P"]
_PARAM_PENAL  = {
    "FET":   "PENAL_FINOS_FET",
    "SIO2":  "PENAL_FINOS_SIO2",
    "CAO":   "PENAL_FINOS_CAO",
    "MGO":   "PENAL_FINOS_MGO",
    "AL2O3": "PENAL_FINOS_AL2O3",
    "MN":    "PENAL_FINOS_MN",
    "P":     "PENAL_FINOS_P",
}

PENAL_BLEND = {}   # delta em pontos percentuais (pode ser negativo)
for _ox, _param in _PARAM_PENAL.items():
    try:
        _val = buscar_valor(df_cal, _param, "GLOBAL")
        PENAL_BLEND[_ox] = float(_val) if pd.notnull(_val) else 0.0
    except Exception:
        PENAL_BLEND[_ox] = 0.0   # sem penalidade se linha não encontrada na aba

# Penalidade de PPC dos finos de NPO (pontos adicionais sobre o PPC do NPO pai)
try:
    PENAL_PPC_FINOS = float(buscar_valor(df_cal, "PENAL_PPC", "GLOBAL"))
    if pd.isna(PENAL_PPC_FINOS):
        PENAL_PPC_FINOS = 0.0
except Exception:
    PENAL_PPC_FINOS = 0.0

# PPC efetivo dos finos de NPO: base (da linha "Finos de NPO" em MATERIAS_PRIMAS) + penalidade
_chave_fino_ppc = next((mp for mp in mps if clean_name(mp) == clean_name("Finos de NPO")), None)
_ppc_base_fino  = PPC.get(_chave_fino_ppc, 0.0664) if _chave_fino_ppc else 0.0664
PPC_FINOS_EFETIVO = _ppc_base_fino + (PENAL_PPC_FINOS / 100)

print(f"  ► Penalidades blend finos NPO (pp): " +
      " | ".join(f"{ox}={PENAL_BLEND[ox]:+.3f}" for ox in _OXIDOS_PENAL))
print(f"  ► Penalidade PPC finos NPO: {PENAL_PPC_FINOS:+.3f} pp "
      f"| PPC base={_ppc_base_fino:.4f} → PPC efetivo={PPC_FINOS_EFETIVO:.4f}")

ESTOQUE_INI = float(buscar_linha(df_ms, "ESTOQUE_PP_INICIAL")["GLOBAL"])
ESTOQUE_FIN = float(buscar_linha(df_ms, "ESTOQUE_PP_FINAL")["GLOBAL"])
DELTA_ESTOQUE = ESTOQUE_FIN - ESTOQUE_INI  # Positivo = Estocar | Negativo = Retomar
ESTOQUE_MAX_PATIO = float(buscar_linha(df_ms, "ESTOQUE_MAX_PATIO")["GLOBAL"])
print(f"  ► Capacidade máxima do pátio: {ESTOQUE_MAX_PATIO:,.0f} t BSR")

# Leitura do preço do pátio para validação
_preco_patio_input = float(Preco.get("PATIO RETOMADO", 0) or 0)

# Custo Op.Var. médio ponderado das MSs (referência mínima para o preço do pátio)
_custo_opvar_ms_ref = (
    sum(CUSTO_OP_VAR_MS[ms] * PROD_FIXA_MS[ms] for ms in MAQUINAS)
    / sum(PROD_FIXA_MS.values())
) if sum(PROD_FIXA_MS.values()) > 0 else 0

# Assert de calibração: o preço do pátio deve ser maior que o custo
# operacional puro — se for menor, provavelmente está sem o custo de MP.
if _preco_patio_input > 0 and _preco_patio_input < _custo_opvar_ms_ref:
    print(
        f"\n  ⚠️  ALERTA DE CALIBRAÇÃO: PRECO_CIF_BU do PATIO "
        f"({_preco_patio_input:.2f} US$/t B.U.) está abaixo do "
        f"CUSTO_OP_VAR_MS ponderado ({_custo_opvar_ms_ref:.2f} US$/t BSR). "
        f"\n     Verifique se o custo de matéria-prima do sínter está incluído."
        f"\n     Referência: use o CUSTO_SINTER da aba KPIS_MS da última rodada."
    )
elif _preco_patio_input == 0:
    print(
        f"\n  ⚠️  ATENÇÃO: PRECO_CIF_BU do PATIO está zerado. "
        f"O sínter retomado entrará nos AFs sem custo de MP nem Op.Var. "
        f"\n     Preencha com o custo total do sínter estocado (US$/t B.U.) "
        f"se houver retomada neste período (DELTA_ESTOQUE = {DELTA_ESTOQUE:,.0f} t)."
    )
else:
    print(f"  ► Preço do sínter do pátio: {_preco_patio_input:.2f} US$/t B.U. ✅")

# % de geração de finos de cada NPO (coluna <10,00 da planilha)
FINOS_NPO_PERC = {}
for mp in mps:
    if tipo_mp.get(mp) == "NPO":
        FINOS_NPO_PERC[mp] = Finos.get(mp, 0.0)


# ======================================================
# 3. VARIÁVEIS DE DECISÃO
# ======================================================
modelo = pl.LpProblem("Otimizacao_Global_V16", pl.LpMinimize)

x_ms = pl.LpVariable.dicts("T_MS_DIRETO", (mps, MAQUINAS, AFS), lowBound=0)
y_ms_patio = pl.LpVariable.dicts("T_MS_PATIO", (mps, MAQUINAS), lowBound=0)
# Variável de retomada do pátio: apenas o índice do AF é necessário.
# Antes declarada como (mps × AFs) — criava variáveis ociosas para cada
# MP que nunca eram usadas. Simplificada para (AFs) apenas.
z_patio_af = pl.LpVariable.dicts("T_PATIO_AF", AFS, lowBound=0)
x_dir = pl.LpVariable.dicts("T_DIR", (mps, AFS), lowBound=0)
x_quartzo = pl.LpVariable.dicts("T_QZ", (AFS), lowBound=0)
gusa = pl.LpVariable.dicts("PROD_GUSA", (AFS), lowBound=0)
excesso_escoria = pl.LpVariable.dicts("EXCESSO_ESCORIA", (AFS), lowBound=0)
npos_list = [mp for mp in mps if tipo_mp.get(mp) == "NPO"]
x_fino_npo = pl.LpVariable.dicts("FINOS_NPO_DINAMICO", (npos_list, MAQUINAS), lowBound=0)

# ======================================================
# 4. RESTRIÇÕES
# ======================================================

# ======================================================
# 4.1 QUALIDADE E PROCESSO DO SÍNTER (MS1E2 e MS3) - COM PÁTIO
# ======================================================

# --- GERAÇÃO GLOBAL DE FINOS DE NPO ---
for mp in npos_list:
    gerado_seco = pl.lpSum(x_dir[mp][af] * (1 - Umid[mp]) * FINOS_NPO_PERC[mp] for af in AFS)
    modelo += pl.lpSum(x_fino_npo[mp][ms] for ms in MAQUINAS) == gerado_seco, f"Bal_Finos_Gerados_{clean_name(mp)}"

for ms in MAQUINAS:

# 1. Massa BSR (Produção de sínter bruto)
    # Perda física aplicada a não-carbonosos de pátio externo.
    # Excluídos da perda: Carga Energética (carbonosos) e Finos de NPO
    # (Finos NPO são gerados internamente no AF e transferidos via circuito
    #  interno — não passam pelo pátio de MPs, portanto sem perda física)
    # Sem perda física: carbonosos (Carga Energética) e reciclados internos
    # (Reciclado Industrial e Finos de NPO não passam pelo pátio externo)
    _sem_perda_ms = lambda mp: (
        tipo_mp.get(mp) in ("CARGA_ENERGETICA", "RECICLADO")
    )
    massa_bsr = pl.lpSum(
            x_ms[mp][ms][af] * (1 - PERDA_FISICA[ms]) * (1 - Umid[mp]) * (1 - PPC[mp])
            for af in AFS for mp in mps
            if Aplicacao.get(mp) == "MS" and not _sem_perda_ms(mp)
        ) + pl.lpSum(
            y_ms_patio[mp][ms] * (1 - PERDA_FISICA[ms]) * (1 - Umid[mp]) * (1 - PPC[mp])
            for mp in mps
            if Aplicacao.get(mp) == "MS" and not _sem_perda_ms(mp)
        ) + pl.lpSum(
            x_ms[mp][ms][af] * (1 - Umid[mp]) *
            (1 - (PPC_FINOS_EFETIVO if clean_name(mp) == clean_name("Finos de NPO") else PPC[mp]))
            for af in AFS for mp in mps
            if Aplicacao.get(mp) == "MS" and _sem_perda_ms(mp)
        ) + pl.lpSum(
            y_ms_patio[mp][ms] * (1 - Umid[mp]) *
            (1 - (PPC_FINOS_EFETIVO if clean_name(mp) == clean_name("Finos de NPO") else PPC[mp]))
            for mp in mps
            if Aplicacao.get(mp) == "MS" and _sem_perda_ms(mp)
        )

    # TRAVA DE PRODUÇÃO: Garante que a máquina respeite o plano (ex: 400kt)
    modelo += massa_bsr == PROD_FIXA_MS[ms], f"Trava_Producao_Fixa_{ms}"

    # 1.1 Mistura Parcial (MPA) - Base Úmida para Granulometria
    # Certifique-se de que o 'massa_mpa' comece exatamente na mesma coluna do 'massa_bsr'
    massa_mpa = pl.lpSum(
        (x_ms[mp][ms][af] for af in AFS for mp in mps if Aplicacao.get(mp) == "MS" and tipo_mp.get(mp) != "CARGA_ENERGETICA")
    ) + pl.lpSum(
        (y_ms_patio[mp][ms] for mp in mps if Aplicacao.get(mp) == "MS" and tipo_mp.get(mp) != "CARGA_ENERGETICA")
    )
    # --- NOVO BALANÇO DE ÓXIDOS (COM QUÍMICA DINÂMICA E PÁTIO) ---
    def m_ox_ms(ox):
        # 1. Massa estática (Puxa tudo MENOS a linha "Finos de NPO")
        # Note: y_ms_patio (sínter para o pátio) está mantido aqui!
        # Perda física aplicada apenas a não-carbonosos (Carga Energética excluída)
        m_estatica = pl.lpSum(
            x_ms[mp][ms][af] * (1 - PERDA_FISICA[ms]) * (1 - Umid[mp]) * (quimica[ox][mp]/100)
            for mp in mps for af in AFS
            if clean_name(mp) != clean_name("Finos de NPO") and tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO")
        ) + pl.lpSum(
            y_ms_patio[mp][ms] * (1 - PERDA_FISICA[ms]) * (1 - Umid[mp]) * (quimica[ox][mp]/100)
            for mp in mps
            if clean_name(mp) != clean_name("Finos de NPO") and tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO")
        ) + pl.lpSum(
            x_ms[mp][ms][af] * (1 - Umid[mp]) * (quimica[ox][mp]/100)
            for mp in mps for af in AFS
            if clean_name(mp) != clean_name("Finos de NPO") and tipo_mp.get(mp) in ("CARGA_ENERGETICA", "RECICLADO")
        ) + pl.lpSum(
            y_ms_patio[mp][ms] * (1 - Umid[mp]) * (quimica[ox][mp]/100)
            for mp in mps
            if clean_name(mp) != clean_name("Finos de NPO") and tipo_mp.get(mp) in ("CARGA_ENERGETICA", "RECICLADO")
        )

        # 2. Massa dinâmica (Herda a química real de cada NPO comprado no AF,
        #    com delta de penalidade de degradação de ganga aplicado à mistura total)
        m_dinamica = pl.lpSum(
            x_fino_npo[mp][ms] * ((quimica[ox][mp] + PENAL_BLEND[ox]) / 100)
            for mp in npos_list
        )
        return m_estatica + m_dinamica

    # --- AMARRAÇÃO PARA O RELATÓRIO DO EXCEL ---
    massa_seca_linha_falsa = pl.lpSum(
        x_ms[mp][ms][af] * (1 - Umid[mp])
        for af in AFS for mp in mps if clean_name(mp) == clean_name("Finos de NPO")
    ) + pl.lpSum(
        y_ms_patio[mp][ms] * (1 - Umid[mp])
        for mp in mps if clean_name(mp) == clean_name("Finos de NPO")
    )
    modelo += massa_seca_linha_falsa == pl.lpSum(x_fino_npo[mp][ms] for mp in npos_list), f"Trava_Relat_Finos_{ms}"

    # --- RESTRIÇÃO DE CONSUMO DE CARGA ENERGÉTICA (kg/t BSR) ---
    # Soma de todos os insumos do tipo CARGA_ENERGETICA em Base Seca
    # Note que usamos (1 - Umid) pois o consumo é reportado em massa seca
    m_energeticos_seco = pl.lpSum(
        (x_ms[mp][ms][af] * (1 - Umid[mp]) for af in AFS for mp in mps if tipo_mp.get(mp) == "CARGA_ENERGETICA")
    ) + pl.lpSum(
        (y_ms_patio[mp][ms] * (1 - Umid[mp]) for mp in mps if tipo_mp.get(mp) == "CARGA_ENERGETICA")
    )

    # Trava: Massa seca total (t) = (Alvo [kg/t] / 1000) * Produção BSR [t]
    modelo += m_energeticos_seco == (ALVO_CARGA_ENERGETICA_MS[ms] / 1000) * massa_bsr, f"Trava_Carga_Energetica_{ms}"

    # --- RESTRIÇÕES QUÍMICAS (MANTIDAS) ---
    # O modelo agora garante que a MISTURA que entra na máquina tenha a qualidade certa
    modelo += m_ox_ms("CAO") >= B2_MS_LIMITS[ms][0] * m_ox_ms("SIO2"), f"B2_Min_{ms}"
    modelo += m_ox_ms("CAO") <= B2_MS_LIMITS[ms][1] * m_ox_ms("SIO2"), f"B2_Max_{ms}"

    modelo += m_ox_ms("MGO") >= (MGO_MS_LIMITS[ms][0]/100) * massa_bsr, f"MgO_Min_{ms}"
    modelo += m_ox_ms("MGO") <= (MGO_MS_LIMITS[ms][1]/100) * massa_bsr, f"MgO_Max_{ms}"
    modelo += m_ox_ms("MN")  >= (MN_MS_MIN[ms]/100) * massa_bsr, f"Mn_Min_{ms}"
    modelo += m_ox_ms("P")   <= (P_MS_MAX[ms]/100) * massa_bsr, f"P_Max_{ms}"

    # --- RESTRIÇÕES FÍSICAS (GRANULAÇÃO - MANTIDAS) ---
    # Somamos as frações granulométricas dos dois caminhos (Direto + Pátio)
    m_gt100 = pl.lpSum((x_ms[mp][ms][af] * G_100[mp] for af in AFS for mp in mps if tipo_mp.get(mp) != "CARGA_ENERGETICA")) + \
              pl.lpSum((y_ms_patio[mp][ms] * G_100[mp] for mp in mps if tipo_mp.get(mp) != "CARGA_ENERGETICA"))

    m_gt635 = pl.lpSum((x_ms[mp][ms][af] * G_635[mp] for af in AFS for mp in mps if tipo_mp.get(mp) != "CARGA_ENERGETICA")) + \
              pl.lpSum((y_ms_patio[mp][ms] * G_635[mp] for mp in mps if tipo_mp.get(mp) != "CARGA_ENERGETICA"))

    m_lt105 = pl.lpSum((x_ms[mp][ms][af] * G_105[mp] for af in AFS for mp in mps if tipo_mp.get(mp) != "CARGA_ENERGETICA")) + \
              pl.lpSum((y_ms_patio[mp][ms] * G_105[mp] for mp in mps if tipo_mp.get(mp) != "CARGA_ENERGETICA"))

    m_fn = m_gt100 - m_gt635
    m_fa = m_lt105

    modelo += m_fn >= RN_MS_LIMITS[ms][0] * (m_fn + m_fa), f"RN_Min_{ms}"
    modelo += m_fn <= RN_MS_LIMITS[ms][1] * (m_fn + m_fa), f"RN_Max_{ms}"
    modelo += m_gt100 >= FRAC_GT1_MIN[ms] * massa_mpa, f"GT1_Min_{ms}"

    # --- CONSUMO DE CAREPA (MANTIDA) ---
    m_carepa_seca = pl.lpSum(
        x_ms[mp][ms][af] * (1 - Umid[mp])
        for af in AFS for mp in mps if "CAREPA" in mp.upper()) + pl.lpSum(y_ms_patio[mp][ms] * (1 - Umid[mp]) for mp in mps if "CAREPA" in mp.upper())

    # Agora aplicamos a restrição usando a massa_bsr (que já é seca)
    modelo += m_carepa_seca <= (CAREPA_MAX_KG_T[ms] / 1000) * massa_bsr, f"Max_Carepa_{ms}"

    # Soma toda a massa seca de matérias-primas que tenham "CAL FINA" no nome
    m_cal_fina_seca = pl.lpSum(
        x_ms[mp][ms][af] * (1 - Umid[mp])
        for af in AFS for mp in mps if "CAL FINA" in mp.upper()
    ) + pl.lpSum(
        y_ms_patio[mp][ms] * (1 - Umid[mp])
        for mp in mps if "CAL FINA" in mp.upper()
    )

    # Trava o consumo EXATO (==) de acordo com o input do Excel
    modelo += m_cal_fina_seca == (CAL_FINA_KG_T[ms] / 1000) * massa_bsr, f"Consumo_Fixo_CalFina_{ms}"

    # --- RESTRIÇÕES DE SINTER FEEDS (MANTIDAS) ---
    # Total de MINERIOs na Máquina (Direto + Pátio) - Base Seca
    m_sf_total_dry = pl.lpSum(
        x_ms[mp][ms][af] * (1 - Umid[mp])
        for af in AFS for mp in mps if tipo_mp.get(mp) == "MINERIO") + pl.lpSum(y_ms_patio[mp][ms] * (1 - Umid[mp]) for mp in mps if tipo_mp.get(mp) == "MINERIO")

    # Musa e Bemisa na Máquina (Direto + Pátio) - Base Seca
    m_musa_dry = pl.lpSum(
        x_ms[mp][ms][af] * (1 - Umid[mp])
        for af in AFS for mp in mps if "MUSA" in mp.upper()) + pl.lpSum(y_ms_patio[mp][ms] * (1 - Umid[mp]) for mp in mps if "MUSA" in mp.upper())

    m_bemisa_dry = pl.lpSum(
        x_ms[mp][ms][af] * (1 - Umid[mp])
        for af in AFS for mp in mps if "BEMISA" in mp.upper()) + pl.lpSum(y_ms_patio[mp][ms] * (1 - Umid[mp]) for mp in mps if "BEMISA" in mp.upper())

    modelo += (m_musa_dry + m_bemisa_dry) <= LIM_MUSA_BEMISA[ms] * m_sf_total_dry, f"Lim_Musa_Bemisa_{ms}"
    modelo += m_bemisa_dry <= LIM_BEMISA[ms] * m_sf_total_dry, f"Lim_Bemisa_{ms}"

# --- RESTRIÇÃO SAZONAL: CONSUMO DE HIGROSCÓPICOS (B.U. GLOBAL MS1E2 + MS3) ---
# Aplicada UMA VEZ, FORA do loop de máquinas — é uma restrição global de pátio.
# O limite é selecionado automaticamente pelo PERIODO lido na aba PERFORMANCE.
if mps_higro:
    consumo_higro_bu = pl.lpSum(
        x_ms[mp][ms][af]
        for mp in mps_higro
        for ms in MAQUINAS
        for af in AFS
    ) + pl.lpSum(
        y_ms_patio[mp][ms]
        for mp in mps_higro
        for ms in MAQUINAS
    )
    modelo += consumo_higro_bu <= HIGRO_LIM_BU, "Limite_Higroscopicos_Sazonal"
    print(f"  ► Restrição sazonal aplicada: {len(mps_higro)} MPs | ≤ {HIGRO_LIM_BU:,.0f} t B.U.")

# --- AJUSTE NA TOLERÂNCIA DE PROPORÇÃO (CONTRA-BALANÇO) ---
for mp in mps:
    # Definimos os materiais que DEVEM ser ignorados pela proporção:
    # 1. Carepa (Logística específica)
    # 2. Reciclado Industrial (Exclusivo MS3)
    # 3. Sinter Fino (Material de ajuste/contra-balanço)

    mp_limpo = clean_name(mp)
    excecoes = [
        clean_name("CAREPA"),
        clean_name("RECICLADO INDUSTRIAL"),
        clean_name("SINTER FINO")
    ]

    # Se a MP não estiver na lista de exceções, aplica a proporção
    if not any(exc in mp_limpo for exc in excecoes):

        m_ms1_total = pl.lpSum(x_ms[mp]["MS1E2"][af] for af in AFS) + y_ms_patio[mp]["MS1E2"]
        m_ms3_total = pl.lpSum(x_ms[mp]["MS3"][af] for af in AFS) + y_ms_patio[mp]["MS3"]

        # Restrição de Proporcionalidade entre as máquinas
        modelo += m_ms1_total * PROD_FIXA_MS["MS3"] * (1 - PROP_TOL_VAL) <= m_ms3_total * PROD_FIXA_MS["MS1E2"], f"Tol_Min_{mp}"
        modelo += m_ms1_total * PROD_FIXA_MS["MS3"] * (1 + PROP_TOL_VAL) >= m_ms3_total * PROD_FIXA_MS["MS1E2"], f"Tol_Max_{mp}"

# ======================================================
# 4.2 REGRAS DE FLUXO DE SINTER E NPO - BALANÇO MESTRE (FINALIZADO)
# ======================================================

# --- BLOQUEIO E SEGREGAÇÃO POR APLICAÇÃO (MS vs AF) ---
for mp in mps:
    # 1. Materiais de Sinterização (MINERIOs MS e Carga Energética)
    # NÃO podem ir direto para o Alto Forno
    if Aplicacao.get(mp) == "MS":
        for af in AFS:
            modelo += x_dir[mp][af] == 0, f"Trava_MS_nao_vai_direto_{mp}_{af}"

    # 2. Materiais de Carga Direta (Pelotas, NPO, Briquetes)
    # NÃO podem entrar na mistura da Sinterização
    if Aplicacao.get(mp) == "AF":
        for ms in MAQUINAS:
            for af in AFS:
                modelo += x_ms[mp][ms][af] == 0, f"Trava_AF_nao_entra_na_MS_{mp}_{ms}_{af}"
            modelo += y_ms_patio[mp][ms] == 0, f"Trava_AF_nao_vai_pro_patio_MS_{mp}_{ms}"

# --- RESTRIÇÃO ESPECÍFICA: RECICLADO INDUSTRIAL APENAS NA MS3 ---
for mp in mps:
    if clean_name(mp) == clean_name("RECICLADO INDUSTRIAL"):
        # 1. Bloqueia o uso na MS1E2
        for af in AFS:
            modelo += x_ms[mp]["MS1E2"][af] == 0, f"Bloqueio_Reciclado_MS1_{af}"
        modelo += y_ms_patio[mp]["MS1E2"] == 0, f"Bloqueio_Reciclado_MS1_Patio"

        # 2. Bloqueia o envio como carga direta para os Altos Fornos
        for af in AFS:
            modelo += x_dir[mp][af] == 0, f"Bloqueio_Reciclado_Direto_AF_{af}"

        print(f"Trava de exclusividade aplicada com sucesso para: {mp}")

# --- BLOQUEIO PÁTIO: entrada exclusiva via z_patio_af ---
# A linha PATIO tem APLICACAO="AF" na planilha, o que a tornaria elegível
# como carga direta via x_dir — em paralelo com z_patio_af (variável dedicada).
# Sem esse bloqueio, quando DELTA_ESTOQUE < 0 o DISP_MAX do PATIO fica igual
# a abs(DELTA_ESTOQUE), e o solver usa x_dir[PATIO] como carga direta adicional,
# duplicando a entrada do pátio na carga metálica e distorcendo a PAM.
# Este bloqueio garante que o sínter de pátio entre EXCLUSIVAMENTE via z_patio_af.
for af in AFS:
    modelo += x_dir["PATIO RETOMADO"][af] == 0, f"Blk_Patio_Dir_{af}"

# 1. TRAVA AF2: AF2 não recebe sínter direto da MS1E2
for mp in mps:
    modelo += x_ms[mp]["MS1E2"]["AF2"] == 0, f"Trava_MS1_AF2_{mp}"

# Rendimento médio ponderado entre as duas máquinas (processo de sinterização)
rend_medio = (
    sum(REND_SINTER[ms] * PROD_FIXA_MS[ms] for ms in MAQUINAS)
    / sum(PROD_FIXA_MS.values())
) if sum(PROD_FIXA_MS.values()) > 0 else 0

# Rendimento de RETOMADA do pátio — grandeza distinta do rend_medio.
# rend_medio = fração de sínter grosso que não vira finos no processo de sinterização.
# rend_patio = fração de sínter aproveitável ao retomar a pilha (descontando
#              segregação, finos de queda e umidade superficial da pilha).
# Lido da aba PERFORMANCE, coluna GLOBAL.
rend_patio = float(buscar_linha(df_perf, "REND_PATIO")["GLOBAL"])
print(f"  ► rend_medio (máquinas): {rend_medio:.4f}  |  rend_patio (retomada): {rend_patio:.4f}")

# ------------------------------------------------------------------
# BALANÇO FÍSICO DO PÁTIO (TUDO EM SÍNTER GROSSO - BSR)
# ------------------------------------------------------------------

# 3. Massa de BSR que VAI para o pátio (Base Seca Útil, mas ainda Bruta/Grossa)
# BUG 3 CORRIGIDO: aplica (1-PERDA) para MPs sujeitas à perda física
# Consistente com massa_bsr (seção 4.1) e prod_ms (seção 4.4).
# Sem a correção, o balanço estaria errado em cenários com DELTA_ESTOQUE != 0.
massa_pro_patio_bsr = pl.lpSum(
    y_ms_patio[mp][ms] * (1 - PERDA_FISICA[ms]) * (1 - Umid[mp]) * (1 - PPC[mp])
    for ms in MAQUINAS for mp in mps
    if Aplicacao.get(mp) == "MS" and tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO")
) + pl.lpSum(
    y_ms_patio[mp][ms] * (1 - Umid[mp]) * (1 - PPC[mp])
    for ms in MAQUINAS for mp in mps
    if Aplicacao.get(mp) == "MS" and tipo_mp.get(mp) in ("CARGA_ENERGETICA", "RECICLADO")
)

# 4. Massa de BSR que SAI do pátio (Base Seca da linha "PATIO RETOMADO" do Excel)
massa_do_patio_bsr = pl.lpSum(
    z_patio_af[af] * (1 - Umid["PATIO RETOMADO"]) * (1 - PPC["PATIO RETOMADO"])
    for af in AFS
)

# 5. Balanço do Pátio: O que entra menos o que sai deve ser o Delta do Estoque.
# Obs: Precisamos garantir que o DELTA_ESTOQUE lido do Excel seja interpretado como BSR.
#
# IMPLEMENTAÇÃO COM FOLGA NUMÉRICA (EPS = 0.1%):
# Para delta != 0, a igualdade estrita (==) combinada com as restrições de
# homogeneização e proporcionalidade MS1/MS3 cria um sistema sobredeterminado
# numericamente para deltas pequenos (< ~5000 t BSR). A razão: as variáveis
# y_ms_patio assumem magnitudes muito pequenas (ex: 39 t de Cal Fina para
# delta=1000 t), criando coeficientes de escala ~1e-3 na mesma restrição que
# contém PROD_FIXA_MS (~2e5), o que excede a capacidade numérica do CBC
# com tolerância padrão (1e-7).
# A folga de 0.1% é imperceptível operacionalmente (máx. 2 t BSR em 2000 t delta)
# e resolve o ill-conditioning sem afetar nenhuma outra restrição.
_EPS_PATIO = 1e-3   # 0.1% de folga numérica
# CORREÇÃO: usar delta ± abs(delta)*eps em vez de delta*(1±eps).
# O motivo: quando DELTA_ESTOQUE < 0 (retomada), multiplicar por (1+eps) produz
# um número MAIS negativo que (1-eps), invertendo a banda:
#   delta*(1-eps) > delta*(1+eps)  para delta < 0
# Isso torna o sistema impossível (piso > teto), causando INFEASIBLE.
# A forma correta é sempre: centro ± folga_absoluta = delta ± abs(delta)*eps,
# garantindo que o limite inferior < limite superior independentemente do sinal.
if abs(DELTA_ESTOQUE) > 1e-6:   # delta != 0
    modelo += (massa_pro_patio_bsr - massa_do_patio_bsr) >= DELTA_ESTOQUE - abs(DELTA_ESTOQUE) * _EPS_PATIO, "Balanco_Fisico_Patio_BSR_Min"
    modelo += (massa_pro_patio_bsr - massa_do_patio_bsr) <= DELTA_ESTOQUE + abs(DELTA_ESTOQUE) * _EPS_PATIO, "Balanco_Fisico_Patio_BSR_Max"
else:   # delta == 0: igualdade estrita (sem pátio ativo, sem risco numérico)
    modelo += (massa_pro_patio_bsr - massa_do_patio_bsr) == DELTA_ESTOQUE, "Balanco_Fisico_Patio_BSR"

modelo += massa_pro_patio_bsr <= massa_do_patio_bsr + (ESTOQUE_MAX_PATIO - ESTOQUE_INI), "Cap_Max_Patio_BSR"

# Travas Lógicas de Operação (Não empilhar e retomar ao mesmo tempo)
if DELTA_ESTOQUE > 0:
    for af in AFS:
        modelo += z_patio_af[af] == 0, f"Trava_Retomada_AF_{af}"
elif DELTA_ESTOQUE < 0:
    for ms in MAQUINAS:
        modelo += pl.lpSum(y_ms_patio[mp][ms] for mp in mps) == 0, f"Trava_Envio_Patio_{ms}"
else: # Delta == 0
    for ms in MAQUINAS:
        modelo += pl.lpSum(y_ms_patio[mp][ms] for mp in mps) == 0
    for af in AFS:
        modelo += z_patio_af[af] == 0

# ======================================================
# 4.2.5 PRÉ-CÁLCULO DE ROTAS (Para transferir a Química Dinâmica)
# ======================================================
v_af2 = PROD_ALVO_GUSA.get("AF2", 0)
v_af3 = PROD_ALVO_GUSA.get("AF3", 0)
p_af2 = v_af2 / (v_af2 + v_af3) if (v_af2 + v_af3) > 0 else 0

prod_ms1 = PROD_FIXA_MS.get("MS1E2", 0)
prod_ms3 = PROD_FIXA_MS.get("MS3", 0)

# Simula o destino logístico do BSR
est_bsr = max(0, DELTA_ESTOQUE)
v_ms1_patio = est_bsr * (prod_ms1 / (prod_ms1 + prod_ms3)) if (prod_ms1 + prod_ms3) > 0 else 0
v_ms3_patio = est_bsr * (prod_ms3 / (prod_ms1 + prod_ms3)) if (prod_ms1 + prod_ms3) > 0 else 0

disp_ms1 = prod_ms1 - v_ms1_patio
disp_ms3 = prod_ms3 - v_ms3_patio

# Desconta o pátio retomado da demanda de sínter direto por AF.
# Quando DELTA_ESTOQUE < 0 (retomada), o pátio supre parte da necessidade
# de sínter do AF — o sínter direto das MSs deve ser proporcionalemente menor.
# O pátio é distribuído entre AFs pela mesma proporção p_af2/p_af3 de gusa.
# Unidade: BSR (dividimos por rend_medio para converter útil→BSR, mantendo
# a mesma base do FATOR_ROTA que opera em BSR).
patio_bsr_af2 = (abs(DELTA_ESTOQUE) * p_af2 / rend_medio) if DELTA_ESTOQUE < 0 else 0.0
dem_af2 = max(0.0, (disp_ms1 + disp_ms3) * p_af2 - patio_bsr_af2)
v_ms3_af2 = min(dem_af2, disp_ms3)
v_ms3_af3 = disp_ms3 - v_ms3_af2
v_ms1_af3 = disp_ms1

# Fatores de rota exatos (Qual proporção de cada máquina vai para qual AF)
FATOR_ROTA = {
    "MS1E2": {
        "AF2": 0.0, # MS1 não atende AF2 no seu modelo
        "AF3": v_ms1_af3 / prod_ms1 if prod_ms1 > 0 else 0
    },
    "MS3": {
        "AF2": v_ms3_af2 / prod_ms3 if prod_ms3 > 0 else 0,
        "AF3": v_ms3_af3 / prod_ms3 if prod_ms3 > 0 else 0
    }
}

# 6. BALANÇOS INDIVIDUAIS POR ALTO FORNO
for af in AFS:
    modelo += gusa[af] == PROD_ALVO_GUSA[af], f"Trava_Producao_Fixa_{af}"

    # ======================================================
    # NOVA FUNÇÃO UNIVERSAL DE QUÍMICA DO ALTO FORNO
    # ======================================================
    def massa_oxido_af(ox):
        # 1. Sínter Estático (Minérios normais, ignorando a linha falsa)
        m_est = pl.lpSum(
            x_ms[mp][ms][af]
            * (1 - PERDA_FISICA[ms] if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
            * (1-Umid[mp]) * (quimica[ox][mp]/100) * REND_SINTER[ms]
            for mp in mps for ms in MAQUINAS if clean_name(mp) != clean_name("Finos de NPO")
        )
        # 2. Sínter Dinâmico (Finos NPO com Rota Exata, Química Real e Penalidade de Ganga)
        m_din = pl.lpSum(
            x_fino_npo[mp][ms] * ((quimica[ox][mp] + PENAL_BLEND[ox]) / 100)
            * REND_SINTER[ms] * FATOR_ROTA[ms][af]
            for mp in npos_list for ms in MAQUINAS
        )

        # 3. Sínter Retomado do Pátio
        m_patio = z_patio_af[af] * (1-Umid["PATIO RETOMADO"]) * (quimica[ox]["PATIO RETOMADO"]/100) * rend_medio

        # 4. Carga Direta (Pelotas, Lump, NPO)
        m_dir = pl.lpSum(
            x_dir[mp][af] * (1-Finos[mp]) * (1-Umid[mp]) * (quimica[ox][mp]/100)
            for mp in mps if Aplicacao.get(mp) == "AF"
        )
        # IMPORTANTE: o quartzo NÃO entra mais aqui.
        # Ele é 100% SiO2 puro e vai 100% à escória — não sofre redução pelo alfa_SI.
        # O quartzo é adicionado diretamente em sio2_e na seção 4.3, após aplicar alfa.
        return m_est + m_din + m_patio + m_dir

    # --- BALANÇO DE FERRO (Agora usando a função universal) ---
    fe_total_in = massa_oxido_af("FET")
    modelo += fe_total_in * ALFA[af]["FE"] == 0.945 * gusa[af], f"Balanco_Fe_{af}"

    # --- MASSA EFETIVA E LIMITES DE CARGA (MANTIDOS IGUAIS) ---
    m_sinter_direto_efetivo = pl.lpSum(
    x_ms[mp][ms][af]
    * (1 - PERDA_FISICA[ms] if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
    * (1-Umid[mp])
    * (1 - (PPC_FINOS_EFETIVO if clean_name(mp) == clean_name("Finos de NPO") else PPC[mp]))
    * REND_SINTER[ms]
    for mp in mps for ms in MAQUINAS if Aplicacao.get(mp) == "MS")

    m_sinter_patio_efetivo = z_patio_af[af] * (1-Umid["PATIO RETOMADO"]) * (1-PPC["PATIO RETOMADO"]) * rend_medio

    m_sinter_total_af = m_sinter_direto_efetivo + m_sinter_patio_efetivo

    m_nobres_diretos = pl.lpSum(x_dir[mp][af] * (1-Finos[mp]) * (1-Umid[mp]) for mp in mps if Aplicacao.get(mp) == "AF" and tipo_mp.get(mp) in ["NPO", "PELOTA", "BRIQUETE"])
    carga_met_af = m_sinter_total_af + m_nobres_diretos

    m_npo_carga = pl.lpSum(x_dir[mp][af] * (1-Finos[mp]) * (1-Umid[mp]) for mp in mps if Aplicacao.get(mp) == "AF" and tipo_mp.get(mp) == "NPO")
    modelo += m_npo_carga <= NPO_MAX_PERC[af] * carga_met_af, f"Limite_NPO_Max_{af}"
    modelo += m_npo_carga >= NPO_MIN_PERC[af] * carga_met_af, f"Limite_NPO_Min_{af}"

    m_pelota_carga = pl.lpSum(x_dir[mp][af] * (1-Finos[mp]) * (1-Umid[mp]) for mp in mps if tipo_mp.get(mp) == "PELOTA")
    modelo += m_pelota_carga >= PELOTA_MIN_PERC[af] * carga_met_af, f"Min_Pelota_{af}"
    modelo += m_sinter_total_af <= SINTER_MAX_PERC[af] * carga_met_af, f"Max_Sinter_{af}"

# ======================================================
# 4.3 ESCÓRIA DO ALTO FORNO E CARBONOSOS
# ======================================================
for af in AFS:
    # Usa a nova Função Universal para puxar os Óxidos da Escória
    # SiO2 da escória = SiO2 da carga metálica × (1 - alfa_SI) + quartzo puro
    # O quartzo é 100% SiO2 e vai 100% à escória; nunca vira silício no gusa.
    # Por isso é somado DEPOIS de aplicar o alfa, e não antes.
    sio2_carga_e = massa_oxido_af("SIO2") * (1 - ALFA[af]["SI"])
    sio2_e = sio2_carga_e + x_quartzo[af]
    cao_e   = massa_oxido_af("CAO")
    mgo_e   = massa_oxido_af("MGO")
    al2o3_e = massa_oxido_af("AL2O3")

    # 1. Calculamos a massa real de escória (usando a calibração do Excel)
    massa_total_esc_base = cao_e + sio2_e + mgo_e + al2o3_e
    massa_total_esc_real = massa_total_esc_base / FATOR_OXIDOS[af]

    # 2. Mede o excesso de escória acima do Target
    modelo += excesso_escoria[af] >= massa_total_esc_real - ((SLAG_RATE_TARGET[af] / 1000) * gusa[af]), f"Mede_Excesso_Escoria_{af}"

    # --- RESTRIÇÃO DE CONSUMO: COQUE_RATE (COM MULTA TÉRMICA) ---
    m_coque_seco_af = pl.lpSum(x_dir[mp][af] * (1 - Umid[mp]) for mp in mps if tipo_mp.get(mp) == "CARBONOSO_COQUE")
    modelo += m_coque_seco_af == ((COQUE_RATE_TARGET[af] / 1000) * gusa[af]) + (excesso_escoria[af] * 0.15), f"Trava_Coque_Rate_{af}"

    # --- RESTRIÇÃO DE CONSUMO: PCI_RATE ---
    m_pci_seco_af = pl.lpSum(x_dir[mp][af] * (1 - Umid[mp]) for mp in mps if tipo_mp.get(mp) == "CARBONOSO_PCI")
    modelo += m_pci_seco_af == (PCI_RATE_TARGET[af] / 1000) * gusa[af], f"Trava_PCI_Rate_{af}"

    # --- RESTRIÇÕES DE BASICIDADE B2 E B3 ---
    modelo += cao_e >= B2_AF_LIMITS[af][0] * sio2_e, f"B2_Min_AF_{af}"
    modelo += cao_e <= B2_AF_LIMITS[af][1] * sio2_e, f"B2_Max_AF_{af}"
    modelo += (cao_e + mgo_e) >= B3_AF_LIMITS[af][0] * sio2_e, f"B3_Min_AF_{af}"
    modelo += (cao_e + mgo_e) <= B3_AF_LIMITS[af][1] * sio2_e, f"B3_Max_AF_{af}"

    # --- VOLUME DE ESCÓRIA MÁXIMO (LIMITE FÍSICO DO FORNO) ---
    modelo += massa_total_esc_real * 1000 <= SLAG_RATE_MAX[af] * gusa[af], f"Slag_Rate_Max_Fisico_{af}"

    # --- CONSUMO DE QUARTZO (KG/T GUSA) ---
    modelo += x_quartzo[af] >= (QZ_LIMITS[af][0] / 1000) * gusa[af], f"QZ_Min_AF_{af}"
    modelo += x_quartzo[af] <= (QZ_LIMITS[af][1] / 1000) * gusa[af], f"QZ_Max_AF_{af}"


# ======================================================
# 4.4 PRODUÇÃO MS E DISPONIBILIDADE - REVISADO PÁTIO
# ======================================================

for ms in MAQUINAS:
    # PRODUÇÃO DE SÍNTER BRUTO (BSR) — seção 4.4
    # Consistente com massa_bsr (seção 4.1): perda física aplicada a não-carbonosos
    prod_ms = pl.lpSum(
        x_ms[mp][ms][af] * (1 - PERDA_FISICA[ms]) * (1 - Umid[mp]) * (1 - PPC[mp])
        for af in AFS for mp in mps
        if Aplicacao.get(mp) == "MS" and tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO")
    ) + pl.lpSum(
        y_ms_patio[mp][ms] * (1 - PERDA_FISICA[ms]) * (1 - Umid[mp]) * (1 - PPC[mp])
        for mp in mps
        if Aplicacao.get(mp) == "MS" and tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO")
    ) + pl.lpSum(
        x_ms[mp][ms][af] * (1 - Umid[mp]) * (1 - PPC[mp])
        for af in AFS for mp in mps
        if Aplicacao.get(mp) == "MS" and tipo_mp.get(mp) in ("CARGA_ENERGETICA", "RECICLADO")
    ) + pl.lpSum(
        y_ms_patio[mp][ms] * (1 - Umid[mp]) * (1 - PPC[mp])
        for mp in mps
        if Aplicacao.get(mp) == "MS" and tipo_mp.get(mp) in ("CARGA_ENERGETICA", "RECICLADO")
    )


for mp in mps:
    # Finos de NPO: volume determinado endogenamente pelo modelo (não fixado pelo Excel)
    if clean_name(mp) == clean_name("Finos de NPO"):
        continue

    # DISPONIBILIDADE GLOBAL DE MATÉRIA-PRIMA:
    cons_total = pl.lpSum(x_ms[mp][ms][af] for ms in MAQUINAS for af in AFS) + \
                 pl.lpSum(y_ms_patio[mp][ms] for ms in MAQUINAS) + \
                 pl.lpSum(x_dir[mp][af] for af in AFS)

    modelo += cons_total >= DispMin.get(mp, 0), f"DispMin_{mp}"
    modelo += cons_total <= DispMax.get(mp, 1e9), f"DispMax_{mp}"


# ======================================================
# 4.5 QUALIDADE DO GUSA (MN E P)
# ======================================================
for af in AFS:
    # Agora Mn e P também bebem da fonte dinâmica exata!
    mn_total = massa_oxido_af("MN")
    modelo += mn_total * ALFA[af]["MN"] >= (MN_AF_MIN[af]/100) * gusa[af], f"Mn_Min_Gusa_{af}"

    p_total = massa_oxido_af("P")
    modelo += p_total * ALFA[af]["P"] <= (P_AF_MAX[af]/100) * gusa[af], f"P_Max_Gusa_{af}"

# ======================================================
# 4.6 REGRA DE MIX ÚNICO (PILHA ÚNICA) - NPOs (BANDA DE TOLERÂNCIA)
# ======================================================
npos_no_mix = [mp for mp in mps if tipo_mp.get(mp) == "NPO" and Aplicacao.get(mp) == "AF"]

if len(AFS) > 1 and len(npos_no_mix) > 1:
    # k_proporcao: razão de tamanho entre os AFs baseada na produção de gusa.
    #
    # RACIONAL FÍSICO: AF2 e AF3 retiram da MESMA pilha de NPO por retomadoras
    # independentes. A composição do blend (fração de cada tipo de NPO) deve ser
    # aproximadamente igual nos dois fornos, mas o VOLUME total de NPO por AF é
    # livre — cada forno decide quanto retomar conforme seu próprio custo ótimo.
    #
    # A restrição correta para pilha única é:
    #   x_dir[mp]["AF2"] / Total_NPO_AF2  ≈  x_dir[mp]["AF3"] / Total_NPO_AF3
    #
    # Isso é bilinear (divisão por variável). A aproximação LP usa k fixo:
    #   k = prod_AF2 / prod_AF3
    # que representa a proporção natural de retirada de cada forno dado o seu
    # tamanho, sem introduzir viés pelos limites regulatórios de NPO_MAX.
    #
    # ATENÇÃO: usar k = (NPO_MAX_AF2 × prod_AF2) / (NPO_MAX_AF3 × prod_AF3)
    # é INCORRETO porque embute os limites percentuais como peso, deflacionando
    # artificialmente o k e impedindo AF2 de atingir seu próprio NPO_MAX mesmo
    # quando há espaço físico e de escória para isso.
    v_af2 = PROD_ALVO_GUSA.get("AF2", 0)
    v_af3 = PROD_ALVO_GUSA.get("AF3", 0)
    k_proporcao = v_af2 / v_af3 if v_af3 > 0 else 1

    # Margem de tolerância física: desvio máximo de blend entre as retomadoras.
    # 20% garante que AF2 atinja seu NPO_MAX (13,8%) sem depender do nível de
    # NPO do AF3, respeitando a física de pilha única com folga operacional real.
    margem_mix = 0.20

    for mp in npos_no_mix:
        # Substituímos a igualdade rígida (==) por duas inequações matemáticas (<= e >=)

        # 1. Limite Superior (Teto): AF2 pode puxar até 20% a MAIS do que a proporção exata
        modelo += x_dir[mp]["AF2"] <= (x_dir[mp]["AF3"] * k_proporcao) * (1 + margem_mix), f"Mix_Unico_Max_{clean_name(mp)}"

        # 2. Limite Inferior (Piso): AF2 pode puxar até 20% a MENOS do que a proporção exata
        modelo += x_dir[mp]["AF2"] >= (x_dir[mp]["AF3"] * k_proporcao) * (1 - margem_mix), f"Mix_Unico_Min_{clean_name(mp)}"

    print(f"Trava de Mix Único flexibilizada em ±{margem_mix*100}% para {len(npos_no_mix)} NPOs.")
    print(f"  k_proporcao = {k_proporcao:.6f}  (prod_AF2/prod_AF3 — razão de tamanho dos fornos)")

# ======================================================
# 4.7 REGRA DE HOMOGENEIZAÇÃO DE SÍNTER (ROTEAMENTO INTELIGENTE)
# ======================================================
# Pré-calculamos as rotas de massa para evitar contradições logísticas e
# garantir que a receita da máquina seja idêntica para qualquer destino.

# 1. Parâmetros Macro
v_af2 = PROD_ALVO_GUSA["AF2"]
v_af3 = PROD_ALVO_GUSA["AF3"]
p_af2 = v_af2 / (v_af2 + v_af3) if (v_af2 + v_af3) > 0 else 0

prod_ms1 = PROD_FIXA_MS["MS1E2"]
prod_ms3 = PROD_FIXA_MS["MS3"]

# 2. Distribuição de Pátio (TUDO EM BSR)
estocagem_bsr = max(0, DELTA_ESTOQUE)
vol_ms1_patio = estocagem_bsr * (prod_ms1 / (prod_ms1 + prod_ms3)) if (prod_ms1 + prod_ms3) > 0 else 0
vol_ms3_patio = estocagem_bsr * (prod_ms3 / (prod_ms1 + prod_ms3)) if (prod_ms1 + prod_ms3) > 0 else 0

# 3. Disponibilidade BSR para os Altos Fornos
disp_ms1 = prod_ms1 - vol_ms1_patio
disp_ms3 = prod_ms3 - vol_ms3_patio

# 4. Distribuição da Demanda
# Desconta o pátio retomado da demanda de sínter direto por AF.
# Consistente com o pré-cálculo do FATOR_ROTA (seção 4.2.5):
# o sínter direto das MSs deve ser reduzido pelo volume que o pátio já supre.
# NOTA: z_patio_af está em BU com Umid=0 e PPC=0, portanto BU=BSR.
# NÃO dividir por rend_medio aqui — o desconto deve estar na mesma base (BSR)
# que a variável z_patio_af, caso contrário o desconto fica 1/rend_medio (~16%)
# maior que o sínter que o pátio realmente entrega, criando déficit no AF2.
patio_bsr_af2_homog = (abs(DELTA_ESTOQUE) * p_af2) if DELTA_ESTOQUE < 0 else 0.0
demanda_af2 = max(0.0, (disp_ms1 + disp_ms3) * p_af2 - patio_bsr_af2_homog)

# A MS3 atende o AF2. O restante vai para o AF3.
vol_ms3_af2 = min(demanda_af2, disp_ms3)
vol_ms3_af3 = disp_ms3 - vol_ms3_af2

# A MS1E2 atende exclusivamente o AF3
vol_ms1_af3 = disp_ms1

# --- APLICAÇÃO MATEMÁTICA DA RECEITA ÚNICA ---
for mp in mps:
    if Aplicacao.get(mp) == "MS":

        # Receita MS1E2 (Blindada pelo volume total da máquina)
        m_total_ms1 = x_ms[mp]["MS1E2"]["AF3"] + y_ms_patio[mp]["MS1E2"]
        if vol_ms1_af3 > 0:
            modelo += x_ms[mp]["MS1E2"]["AF3"] == m_total_ms1 * (vol_ms1_af3 / prod_ms1), f"Homog_MS1_AF3_{clean_name(mp)}"
        if vol_ms1_patio > 0:
            modelo += y_ms_patio[mp]["MS1E2"] == m_total_ms1 * (vol_ms1_patio / prod_ms1), f"Homog_MS1_Patio_{clean_name(mp)}"

        # Receita MS3 (Garante que qualquer destino receba a mesma química, sem depender do AF3)
        m_total_ms3 = pl.lpSum(x_ms[mp]["MS3"][af] for af in AFS) + y_ms_patio[mp]["MS3"]

        if vol_ms3_af2 > 0:
            modelo += x_ms[mp]["MS3"]["AF2"] == m_total_ms3 * (vol_ms3_af2 / prod_ms3), f"Homog_MS3_AF2_{clean_name(mp)}"

        if vol_ms3_af3 > 0:
            modelo += x_ms[mp]["MS3"]["AF3"] == m_total_ms3 * (vol_ms3_af3 / prod_ms3), f"Homog_MS3_AF3_{clean_name(mp)}"

        if vol_ms3_patio > 0:
            modelo += y_ms_patio[mp]["MS3"] == m_total_ms3 * (vol_ms3_patio / prod_ms3), f"Homog_MS3_Patio_{clean_name(mp)}"

# --- RETOMADA DO PÁTIO ---
# Garante que o sínter retomado do pátio tenha a mesma distribuição química nos dois AFs
if v_af3 > 0:
    k_retomada = v_af2 / v_af3
    modelo += z_patio_af["AF2"] == z_patio_af["AF3"] * k_retomada, "Homog_Retomada_Patio"

# ======================================================
# 5. EXECUÇÃO E FUNÇÃO OBJETIVO - FINALIZADO E INTEGRADO
# ======================================================

# Trata preços nulos como zero para evitar erros no cálculo da FO
preco_limpo = {mp: (p if pd.notnull(p) else 0.0) for mp, p in Preco.items()}

# Custo de matérias-primas da sinterização (rotas direto + pátio)
custo_ms = pl.lpSum(
    (pl.lpSum(x_ms[mp][ms][af] for af in AFS) + y_ms_patio[mp][ms]) * preco_limpo.get(mp, 0)
    for mp in mps for ms in MAQUINAS if clean_name(mp) != clean_name("Finos de NPO")
)

# Custo de carga direta (pelotas, NPO, briquetes, carbonosos): 100% do CIF
custo_af_direto = pl.lpSum(
    x_dir[mp][af] * preco_limpo.get(mp, 0)
    for mp in mps for af in AFS
)

# Custo de retomada do pátio de sínter
# ──────────────────────────────────────────────────────────────────────────

custo_retomada = pl.lpSum(
    z_patio_af[af] * preco_limpo.get("PATIO RETOMADO", 0)
    for af in AFS
)

print(f"  ► Custo do sínter retomado na FO: {preco_limpo.get('PATIO RETOMADO', 0):.4f} US$/t B.U. (input Excel)")

# Custo do quartzo (US$ 40/t fixo)
custo_aditivos = pl.lpSum(x_quartzo[af] * 40.0 for af in AFS)

# FUNÇÃO OBJETIVO GLOBAL
# Custos operacionais variáveis: constantes (produção travada), mas incluídos
# para que o valor da FO coincida com o CUSTO_GUSA reportado.
# CUSTO_OP_VAR_MS: aplica-se a todo sínter produzido (direto + pátio), NUNCA à retomada
# CUSTO_OP_VAR_AF: aplica-se a toda produção de gusa de cada forno
custo_op_var_ms = pl.lpSum(CUSTO_OP_VAR_MS[ms] * PROD_FIXA_MS[ms] for ms in MAQUINAS)
custo_op_var_af = pl.lpSum(CUSTO_OP_VAR_AF[af] * PROD_ALVO_GUSA[af] for af in AFS)
modelo += custo_ms + custo_af_direto + custo_retomada + custo_aditivos + custo_op_var_ms + custo_op_var_af

# Comando para resolver
status = modelo.solve(pl.PULP_CBC_CMD(msg=True))
print(f"--- Status da Otimização: {pl.LpStatus[status]} ---")

# Verificação de segurança: Só prossegue se a solução for Ótima
if pl.LpStatus[status] != 'Optimal':
    print("⚠️ ATENÇÃO: O modelo não encontrou uma solução viável. Iniciando diagnóstico automático...\n")
    # Diagnóstico automático: remove cada grupo de constraints temporariamente
    # e verifica qual grupo torna o modelo factível (causa raiz da infeasibilidade).
    _grupos_diag = {
        "Producao MS (PROD_FIXA)": ["Trava_Producao_Fixa_MS"],
        "Disponibilidade Maxima": ["DispMax_"],
        "Disponibilidade Minima": ["DispMin_"],
        "Basicidade B2 MS": ["B2_Min_MS", "B2_Max_MS"],
        "MgO MS": ["MgO_Min_MS", "MgO_Max_MS"],
        "Carga Energetica MS": ["Trava_Carga_Energetica_"],
        "Cal Fina MS": ["Consumo_Fixo_CalFina_"],
        "Proporcionalidade MS1/MS3": ["Tol_Min_", "Tol_Max_"],
        "Granulometria MS": ["RN_Min_", "RN_Max_", "GT1_Min_"],
        "Balanco Fe AF": ["Balanco_Fe_"],
        "Basicidade B2 AF": ["B2_Min_AF_", "B2_Max_AF_"],
        "Basicidade B3 AF": ["B3_Min_AF_", "B3_Max_AF_"],
        "Slag Rate Max AF": ["Slag_Rate_Max_Fisico_"],
        "Coque/PCI Rate AF": ["Trava_Coque_Rate_", "Trava_PCI_Rate_"],
        "Limite NPO/Pelota/Sinter AF": ["Limite_NPO_Max_", "Limite_NPO_Min_", "Min_Pelota_", "Max_Sinter_"],
        "Qualidade Gusa (Mn/P)": ["Mn_Min_Gusa_", "P_Max_Gusa_"],
        "Mix Unico NPO": ["Mix_Unico_NPO_"],
        "Homogeneizacao Sinter": ["Homog_"],
        "Finos de NPO": ["Bal_Finos_Gerados_", "Trava_Relat_Finos_"],
        "Balanco Patio BSR": ["Balanco_Fisico_Patio_BSR"],
        "Cap. Maxima Patio BSR": ["Cap_Max_Patio_BSR"],
        "Quartzo AF": ["QZ_Min_AF_", "QZ_Max_AF_"],
    }
    import io as _io
    print(f"  {'Grupo de Restrição':<45} {'Resultado':>15}")
    print("  " + "─"*62)
    _causas = []
    for _nome_g, _prefixos in _grupos_diag.items():
        _m_tmp = modelo.copy()
        for _r in list(_m_tmp.constraints.keys()):
            if any(_r.startswith(_p) or _r == _p.rstrip("_") for _p in _prefixos):
                del _m_tmp.constraints[_r]
        _old = __import__("sys").stdout
        __import__("sys").stdout = _io.StringIO()
        _st = _m_tmp.solve(pl.PULP_CBC_CMD(msg=False))
        __import__("sys").stdout = _old
        _res = pl.LpStatus[_st]
        if _res == "Optimal":
            _mark = "✅ CAUSA RAIZ"
            _causas.append(_nome_g)
        else:
            _mark = "❌ não é causa"
        print(f"  {_nome_g:<45} {_mark:>15}")
    print("  " + "─"*62)
    if _causas:
        print(f"\n  ➡️  Grupos que tornam o modelo factível se removidos:")
        for _c in _causas:
            print(f"      • {_c}")
        print(f"\n  Dica: verifique os parâmetros de entrada (Excel) para esses grupos.")
    else:
        print("\n  ⚠️  Nenhum grupo isolado resolve o problema — conflito entre múltiplos grupos.")
        print("  Tente reduzir progressivamente os limites mais restritivos no Excel.")

# ======================================================
# VALIDAÇÃO DE CONSISTÊNCIA DA FUNÇÃO OBJETIVO
# ======================================================
# SUBSTITUIÇÃO: apague do comentário
# "# Validação de consistência: verifica se CUSTO_GUSA..."
# até (inclusive) o print("─"*50 + "\n")
# e cole este bloco inteiro no lugar.
#
# CORREÇÕES APLICADAS:
#
# [1] REND_SINTER adicionado em _c_sin_chk
#     O custo do sínter deve ser calculado sobre o sínter ÚTIL
#     (BSR × REND_SINTER) — o que fisicamente entrou no forno.
#     Sem isso, o custo incluía os finos (~13,9%), inflando
#     o CUSTO_GUSA em ~17 US$/t.
#
# [2] Tolerância baseada no gap físico esperado + margem fixa
#     A FO cobre o custo total do período, incluindo:
#       A) Custo dos finos de sinterização (sempre presente)
#          = PROD_BSR × (1-REND) × custo_unit_ms
#       B) Custo do sínter desviado ao pátio (Delta+ apenas)
#          = BSR_pátio × custo_unit_ms
#     O CUSTO_GUSA não inclui esses itens — são custo de
#     períodos futuros ou de outros circuitos (finos retornam à MS).
#     A tolerância = gap_esperado + 1,0 US$/t (margem fixa).
#     Qualquer erro REAL acima de 1 US$/t dispara o alerta.
# ======================================================

# Validação de consistência: verifica se CUSTO_GUSA calculado no relatório
# converge com a função objetivo do solver.
if pl.LpStatus[status] == 'Optimal':

    _prod_total_val = sum(gusa[af].value() or 0 for af in AFS)
    _fo_por_t_val   = pl.value(modelo.objective) / _prod_total_val \
                      if _prod_total_val > 0 else 0

    # ── Custo unitário de cada MS ──────────────────────────────────────────
    _custo_unit_ms_chk = {}
    for ms in MAQUINAS:
        _inv_chk = sum(
            (sum(x_ms[mp][ms][af].value() or 0 for af in AFS)
             + (y_ms_patio[mp][ms].value() or 0))
            * preco_limpo.get(mp, 0)
            for mp in mps if clean_name(mp) != clean_name("Finos de NPO")
        )
        _custo_finos_npo_chk = sum(
            sum((x_dir[npo][af].value() or 0) for af in AFS)
            * Finos.get(npo, 0) * preco_limpo.get(npo, 0)
            * ((x_fino_npo[npo][ms].value() or 0)
               / (sum(x_fino_npo[npo][ms2].value() or 0
                      for ms2 in MAQUINAS) + 1e-10))
            for npo in npos_list
        )
        _cu_bruto_chk = (
            (_inv_chk + _custo_finos_npo_chk) / PROD_FIXA_MS[ms]
            if PROD_FIXA_MS[ms] > 0 else 0
        ) + CUSTO_OP_VAR_MS[ms]
        # Crédito do sínter fino como subproduto: divide pelo REND para alocar
        # o custo total apenas ao sínter útil que entra no Alto Forno.
        _custo_unit_ms_chk[ms] = _cu_bruto_chk / REND_SINTER[ms] if REND_SINTER[ms] > 0 else _cu_bruto_chk

    # ── Reconstrução do CUSTO_GUSA por AF ─────────────────────────────────
    _cg_total_chk = 0
    for af in AFS:
        _v_g_chk = gusa[af].value() or 1

        # Carga direta: NPO paga só a porção útil (1-Finos)
        _c_dir_chk = (
            sum((x_dir[mp][af].value() or 0)
                * (1 - Finos.get(mp, 0)) * preco_limpo.get(mp, 0)
                for mp in mps if tipo_mp.get(mp) == "NPO")
            + sum((x_dir[mp][af].value() or 0) * preco_limpo.get(mp, 0)
                  for mp in mps if tipo_mp.get(mp) != "NPO")
        )

        # [CORREÇÃO 1] REND_SINTER converte BSR bruto → sínter útil que entrou no forno
        _c_sin_chk = sum(
            sum((x_ms[mp][ms][af].value() or 0)
                * (1 - PERDA_FISICA[ms]
                   if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO")
                   else 1.0)
                * (1 - Umid[mp])
                * (1 - (PPC_FINOS_EFETIVO
                        if clean_name(mp) == clean_name("Finos de NPO")
                        else PPC[mp]))
                * REND_SINTER[ms]          # ← CORREÇÃO 1: BSR → sínter útil
                * _custo_unit_ms_chk[ms]
                for mp in mps)
            for ms in MAQUINAS
        )

        _c_qz_chk    = (x_quartzo[af].value() or 0) * 40.0
        _c_op_af_chk = CUSTO_OP_VAR_AF[af] * _v_g_chk
        _cg_total_chk += _c_dir_chk + _c_sin_chk + _c_qz_chk + _c_op_af_chk

    # Retomada do pátio (zero no Delta+, relevante no Delta-)
    _c_retomada_chk = (
        sum(z_patio_af[af].value() or 0 for af in AFS)
        * preco_limpo.get("PATIO RETOMADO", 0)
    )
    _cg_total_chk += _c_retomada_chk

    _cg_por_t_chk = _cg_total_chk / _prod_total_val \
                    if _prod_total_val > 0 else 0

    # ── [CORREÇÃO 2] Gap físico esperado e tolerância ─────────────────────
    #
    # A FO > CUSTO_GUSA por dois motivos físicos permanentes:
    #
    # Componente A — Finos de sinterização (sempre presente):
    #   As MPs compradas geram sínter bruto (BSR). Desse total,
    #   ~13,9% viram finos (REND ≈ 86,1%), que retornam à mistura.
    #   A FO paga pelas MPs que geraram TUDO (útil + finos).
    #   O CUSTO_GUSA paga apenas pelo sínter útil (REND aplicado).
    #   Gap_A = PROD_BSR × (1-REND) × custo_unit_ms / prod_gusa
    #
    # Componente B — Sínter desviado ao pátio (apenas Delta > 0):
    #   A FO paga pelas MPs que geraram o sínter estocado hoje.
    #   O CUSTO_GUSA não inclui esse sínter (vai ser custo futuro).
    #   Gap_B = BSR_pátio × custo_unit_ms / prod_gusa
    #   Onde BSR_pátio = sum(y_ms_patio × (1-PERDA)×(1-umid)×(1-PPC))
    #
    # Tolerância = gap_esperado + 1,0 US$/t
    #   A margem de 1 US$/t absorve o ruído numérico do CBC e pequenas
    #   diferenças de arredondamento entre base úmida e seca.
    #   Qualquer discrepância REAL (preço errado, PROD_FIXA incorreto,
    #   custo_unit_ms fora de lugar) gera excesso > 1 US$/t → alerta.

    # Componente A: custo das MPs que geraram finos (sempre presente)
    _gap_A = sum(
        PROD_FIXA_MS[ms] * (1 - REND_SINTER[ms]) * _custo_unit_ms_chk[ms]
        for ms in MAQUINAS
    ) / _prod_total_val if _prod_total_val > 0 else 0

    # Componente B: custo das MPs que geraram sínter desviado ao pátio
    # BSR_pátio = sínter bruto que foi ao pátio (y_ms_patio → BSR)
    _gap_B = sum(
        sum(
            (y_ms_patio[mp][ms].value() or 0)
            * (1 - PERDA_FISICA[ms]
               if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO")
               else 1.0)
            * (1 - Umid[mp])
            * (1 - PPC[mp])
            * _custo_unit_ms_chk[ms]
            for mp in mps if Aplicacao.get(mp) == "MS"
        )
        for ms in MAQUINAS
    ) / _prod_total_val if _prod_total_val > 0 else 0

    _gap_esperado = _gap_A + _gap_B   # gap estrutural calculado pela física
    _TOL_MARGEM   = 1.0               # US$/t — margem fixa para ruído CBC
    _TOL_CHK      = _gap_esperado + _TOL_MARGEM

    # Cenário textual para o console
    if DELTA_ESTOQUE > 0:
        _cenario_txt = f"ESTOCAGEM  Δ=+{DELTA_ESTOQUE:,.0f} t BSR"
    elif DELTA_ESTOQUE < 0:
        _cenario_txt = f"RETOMADA   Δ={DELTA_ESTOQUE:,.0f} t BSR"
    else:
        _cenario_txt = "NEUTRO     Δ=0 t"

    # ── Impressão ──────────────────────────────────────────────────────────
    _gap_chk = abs(_cg_por_t_chk - _fo_por_t_val)

    print(f"\n" + "─" * 66)
    print(f"  VALIDAÇÃO DA FUNÇÃO OBJETIVO  |  Pátio: {_cenario_txt}")
    print(f"─" * 66)
    print(f"  FO solver          : {_fo_por_t_val:>10.4f} US$/t gusa")
    print(f"  CUSTO_GUSA         : {_cg_por_t_chk:>10.4f} US$/t gusa")
    print(f"  Gap encontrado     : {_gap_chk:>10.4f} US$/t gusa")
    print(f"  ┌─ Gap esperado    : {_gap_esperado:>10.4f} US$/t gusa")
    print(f"  │    Finos MS (A)  : {_gap_A:>10.4f} US$/t gusa"
          f"  ← MPs que geraram finos de sinterização")
    print(f"  │    Pátio    (B)  : {_gap_B:>10.4f} US$/t gusa"
          f"  ← MPs que geraram sínter estocado")
    print(f"  └─ Tolerância      : {_TOL_CHK:>10.4f} US$/t gusa"
          f"  ← gap_esp + {_TOL_MARGEM:.1f} US$/t")

    if _gap_chk <= _TOL_CHK:
        print(f"\n  ✅ Consistência confirmada — gap dentro da tolerância.")
    else:
        _excesso = _gap_chk - _TOL_CHK
        print(f"\n  ⚠️  ALERTA: gap excede tolerância em {_excesso:.4f} US$/t")
        print(f"     Este excesso NÃO é efeito estrutural do pátio ou dos finos.")
        print(f"     Verifique: preços das MPs, PROD_FIXA_MS, CUSTO_OP_VAR_MS.")

    print("─" * 66 + "\n")

# ======================================================
# 6. MÓDULO DE ESCRITA REVERSA
# ======================================================

import openpyxl

if pl.LpStatus[status] == 'Optimal':
    print("\nIniciando gravação dos resultados no Excel...")
    wb = openpyxl.load_workbook(nome_arquivo, data_only=True)

# --- PÓS-PROCESSAMENTO: MÉDIAS PONDERADAS PARA A LINHA "FINOS DE NPO" ---
    chave_fino = [m for m in mps if clean_name(m) == clean_name("Finos de NPO")]
    if chave_fino:
        mp_fino = chave_fino[0]
        # Massa seca total (para a química)
        m_f_total_seca = sum(x_fino_npo[m][ms].value() or 0 for m in npos_list for ms in MAQUINAS)
        # Massa úmida total (para o financeiro)
        m_f_total_umida = sum((x_fino_npo[m][ms].value() or 0) / (1 - Umid[m]) for m in npos_list for ms in MAQUINAS if Umid[m] < 1)

        if m_f_total_seca > 0:
            # 1. Preço Ponderado (Base Úmida)
            Preco[mp_fino] = sum(((x_fino_npo[m][ms].value() or 0) / (1 - Umid[m])) * Preco.get(m, 0) for m in npos_list for ms in MAQUINAS if Umid[m] < 1) / m_f_total_umida

            # 2. Química Ponderada (Base Seca) — inclui penalidade de ganga
            # A química reportada aqui é a que EFETIVAMENTE entrou na sinterização:
            # média ponderada dos NPOs + delta de penalidade da aba CALIBRACAO.
            # Isso garante consistência entre o que o solver usou e o que o relatório mostra.
            for ox in ["FET", "SIO2", "CAO", "MGO", "AL2O3", "MN", "P"]:
                m_ox = sum((x_fino_npo[m][ms].value() or 0) * (quimica[ox][m]/100) for m in npos_list for ms in MAQUINAS)
                quimica[ox][mp_fino] = (m_ox / m_f_total_seca) * 100 + PENAL_BLEND[ox]

    # PRÉ-CÁLCULO DE CUSTOS UNITÁRIOS DAS MÁQUINAS
    # Inclui CUSTO_OP_VAR_MS por máquina (custo operacional variável da sinterização)
    # Denominador usa PROD_FIXA_MS (valor exato da trava, evita ruído numérico do solver)
    custo_unit_ms = {}
    for ms in MAQUINAS:
        invest_total = sum(
            (sum(x_ms[mp][ms][af].value() or 0 for af in AFS) + (y_ms_patio[mp][ms].value() or 0)) * Preco.get(mp, 0)
            for mp in mps if clean_name(mp) != clean_name("Finos de NPO")
        )
        # Finos de NPO: custo proporcional alocado à MS (rateio via x_fino_npo)
        custo_finos_npo_ms = sum(
            sum((x_dir[npo][af].value() or 0) for af in AFS) * Finos.get(npo, 0) * Preco.get(npo, 0)
            * ((x_fino_npo[npo][ms].value() or 0) /
               (sum(x_fino_npo[npo][ms2].value() or 0 for ms2 in MAQUINAS) + 1e-10))
            for npo in npos_list
        )
        _cu_bruto = ((invest_total + custo_finos_npo_ms) / PROD_FIXA_MS[ms] if PROD_FIXA_MS[ms] > 0 else 0) + CUSTO_OP_VAR_MS[ms]
        # Crédito do sínter fino como subproduto: divide pelo REND para alocar
        # o custo total apenas ao sínter útil que entra no Alto Forno.
        custo_unit_ms[ms] = _cu_bruto / REND_SINTER[ms] if REND_SINTER[ms] > 0 else _cu_bruto

    # ------------------------------------------------------
    # A. ABA MATERIAS_PRIMAS (Preenchimento de Consumos, Qualidades e Custos)
    # ------------------------------------------------------
    ws_mp = wb["MATERIAS_PRIMAS"]
    h_mp = {str(cell.value).strip().upper(): cell.column for cell in ws_mp[1] if cell.value}

    # Produções de gusa para o cálculo de kg/t (Denominador)
    prod_af2 = gusa["AF2"].value() or 1
    prod_af3 = gusa["AF3"].value() or 1

    for row in range(2, ws_mp.max_row + 1):
        mp_cell = ws_mp.cell(row=row, column=h_mp.get("MP", 1)).value
        if mp_cell is None: continue
        mp = str(mp_cell).upper().strip()

        def gravar_celula(nome_coluna, valor):
            chave = nome_coluna.strip().upper()
            if chave in h_mp:
                ws_mp.cell(row=row, column=h_mp[chave]).value = valor

        # --- PARTE A: MATÉRIAS-PRIMAS REAIS (MINERIOS, COQUE, ETC) ---
        if mp in mps and mp not in ["MS1E2", "MS3", "PATIO RETOMADO", "PATIO RECOMPOSTO"]:

            # --- NOVO: ESCREVENDO A QUÍMICA E PREÇO DINÂMICOS NO EXCEL ---
            if clean_name(mp) == clean_name("Finos de NPO"):
                gravar_celula("PRECO_CIF_BU", Preco.get(mp, 0))
                for ox in ["FET", "SIO2", "CAO", "MGO", "AL2O3", "MN", "P"]:
                    gravar_celula(ox, quimica[ox].get(mp, 0))

            # Consumos MS
            bu_ms1e2 = sum(x_ms[mp]["MS1E2"][af].value() or 0 for af in AFS) + (y_ms_patio[mp]["MS1E2"].value() or 0)
            # Consumo efetivo (base seca, líquido da perda física)
            # Carga Energética excluída da perda; demais MPs da MPA com perda
            _p_ms1 = (1 - PERDA_FISICA["MS1E2"]) if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0
            _p_ms3 = (1 - PERDA_FISICA["MS3"])   if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0
            bs_ms1e2 = bu_ms1e2 * _p_ms1 * (1 - Umid[mp])
            bu_ms3 = sum(x_ms[mp]["MS3"][af].value() or 0 for af in AFS) + (y_ms_patio[mp]["MS3"].value() or 0)
            bs_ms3 = bu_ms3 * _p_ms3 * (1 - Umid[mp])

            gravar_celula("CONSUMO B.S. – MS1E2 (T)", bs_ms1e2)
            gravar_celula("KG/T B.S. – MS1E2", (bs_ms1e2 * 1000) / PROD_FIXA_MS["MS1E2"] if PROD_FIXA_MS["MS1E2"] > 0 else 0)
            gravar_celula("CONSUMO B.S. – MS3 (T)", bs_ms3)
            gravar_celula("KG/T B.S. – MS3", (bs_ms3 * 1000) / PROD_FIXA_MS["MS3"] if PROD_FIXA_MS["MS3"] > 0 else 0)
            gravar_celula("CONSUMO TOTAL B.U. – MS1E2 + MS3 (T)", bu_ms1e2 + bu_ms3)

            # Consumos AF (Carga Direta)
            bu_af2 = x_dir[mp]["AF2"].value() or 0
            bu_af3 = x_dir[mp]["AF3"].value() or 0
            bs_af2, bs_af3 = bu_af2 * (1 - Umid[mp]), bu_af3 * (1 - Umid[mp])

            gravar_celula("CONSUMO B.S. – AF2 (T)", bs_af2)
            gravar_celula("KG/T B.S. – AF2", (bs_af2 * (1-Finos[mp]) * 1000) / prod_af2)
            gravar_celula("CONSUMO B.S. – AF3 (T)", bs_af3)
            gravar_celula("KG/T B.S. – AF3", (bs_af3 * (1-Finos[mp]) * 1000) / prod_af3)
            gravar_celula("CONSUMO TOTAL B.U. – AF2 + AF3 (T)", bu_af2 + bu_af3)

        # --- PARTE B: LINHAS DE PRODUTO SÍNTER (MS1, MS3 E PATIO) ---
        elif mp in ["MS1E2", "MS3"]:
            ms_nome = mp
            # Massa BSR total da máquina para ponderação química
            m_bsr_ms = sum((sum(x_ms[m][ms_nome][af].value() or 0 for af in AFS) + (y_ms_patio[m][ms_nome].value() or 0))
                            * (1 - Umid[m]) * (1 - PPC[m]) for m in mps if Aplicacao.get(m) == "MS")

            # 1. Preencher Qualidades Resultantes (Loop de Elementos)
            for elem in ["FET", "SIO2", "AL2O3", "CAO", "MGO", "MN", "P"]:
                m_elem = sum((sum(x_ms[m][ms_nome][af].value() or 0 for af in AFS) + (y_ms_patio[m][ms_nome].value() or 0))
                             * (1 - Umid[m]) * (quimica[elem][m]/100) for m in mps if Aplicacao.get(m) == "MS")
                gravar_celula(elem, (m_elem / m_bsr_ms * 100) if m_bsr_ms > 0 else 0)

            # 2. Preencher Custo de Produção na coluna PRECO
            # PRECO_CIF_BU = custo do sínter BRUTO por tonelada de BSR produzido
            # Replica EXATAMENTE a fórmula do KPIS_MS:
            #   custo_abs_ms = sum(BU × Preco)  — custo total das MPs em BU
            #   denominador  = PROD_FIXA_MS     — produção BSR travada pelo solver
            # Isso garante que PRECO_CIF_BU (MATERIAS_PRIMAS) = CUSTO_SINTER (KPIS_MS)
            _custo_abs_mp = sum(
                (sum(x_ms[m][ms_nome][af].value() or 0 for af in AFS)
                 + (y_ms_patio[m][ms_nome].value() or 0))
                * Preco.get(m, 0)
                for m in mps
            )
            _cu_bruto_mp = (
                _custo_abs_mp / PROD_FIXA_MS[ms_nome]
                if PROD_FIXA_MS[ms_nome] > 0 else 0
            ) + CUSTO_OP_VAR_MS[ms_nome]
            gravar_celula("PRECO_CIF_BU", _cu_bruto_mp)

            # 3. Consumos de Sínter Útil (B.S.) e Indicadores kg/t
            c_bs_af2 = sum(
                (x_ms[m][ms_nome]["AF2"].value() or 0)
                * (1 - PERDA_FISICA[ms_nome] if tipo_mp.get(m) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
                * (1-Umid[m]) * (1-PPC[m])
                for m in mps if Aplicacao.get(m) == "MS"
            ) * REND_SINTER[ms_nome]
            c_bs_af3 = sum(
                (x_ms[m][ms_nome]["AF3"].value() or 0)
                * (1 - PERDA_FISICA[ms_nome] if tipo_mp.get(m) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
                * (1-Umid[m]) * (1-PPC[m])
                for m in mps if Aplicacao.get(m) == "MS"
            ) * REND_SINTER[ms_nome]

            gravar_celula("CONSUMO B.S. – AF2 (T)", c_bs_af2)
            gravar_celula("KG/T B.S. – AF2", (c_bs_af2 * 1000) / prod_af2)
            gravar_celula("CONSUMO B.S. – AF3 (T)", c_bs_af3)
            gravar_celula("KG/T B.S. – AF3", (c_bs_af3 * 1000) / prod_af3)
            gravar_celula("CONSUMO TOTAL B.U. – AF2 + AF3 (T)", (c_bs_af2 + c_bs_af3))

        elif mp == "PATIO RETOMADO":
            # Consumo da Retomada (Base Seca Útil)
            c_bs_af2 = (z_patio_af["AF2"].value() or 0) * (1-Umid["PATIO RETOMADO"]) * (1-PPC["PATIO RETOMADO"]) * rend_medio
            c_bs_af3 = (z_patio_af["AF3"].value() or 0) * (1-Umid["PATIO RETOMADO"]) * (1-PPC["PATIO RETOMADO"]) * rend_medio

            gravar_celula("CONSUMO B.S. – AF2 (T)", c_bs_af2)
            gravar_celula("KG/T B.S. – AF2", (c_bs_af2 * 1000) / prod_af2)
            gravar_celula("CONSUMO B.S. – AF3 (T)", c_bs_af3)
            gravar_celula("KG/T B.S. – AF3", (c_bs_af3 * 1000) / prod_af3)
            gravar_celula("CONSUMO TOTAL B.U. – AF2 + AF3 (T)", (c_bs_af2 + c_bs_af3))

            # PRECO_CIF_BU do pátio: mantém o valor inputado pelo operador.
            # O operador é responsável por informar o custo total do sínter
            # (MP + Op.Var.) — o modelo não altera esse campo.
            gravar_celula("PRECO_CIF_BU", preco_limpo.get("PATIO RETOMADO", 0))

        elif mp == "PATIO RECOMPOSTO":
            # ------------------------------------------------------
            # PATIO RECOMPOSTO — Qualidade e custo do sínter estocado
            # Só preenchido quando DELTA_ESTOQUE > 0 (estocagem).
            # Em retomada ou neutro, a linha permanece em branco.
            #
            # FÍSICA: o sínter que vai ao pátio sai das duas máquinas
            # (MS1E2 e MS3) na mesma receita que alimentaria os AFs,
            # graças à regra de homogeneização (seção 4.7).
            # A qualidade da pilha é, portanto, a média ponderada em
            # base seca de tudo que y_ms_patio desviou de cada MS.
            #
            # CUSTO: custo bruto de produção (MP + Op.Var.) ponderado
            # pela produção BSR de cada máquina — igual ao PRECO_CIF_BU
            # gravado nas linhas MS1E2 e MS3 da mesma aba.
            # NÃO usa o crédito do fino (÷ REND) porque o sínter ainda
            # está na pilha — não passou pelo peneiramento do AF.
            # ------------------------------------------------------
            if DELTA_ESTOQUE > 0:

                # ── Base seca total desviada ao pátio (denominador) ──────
                # Replica exatamente a fórmula de massa_pro_patio_bsr,
                # mas sem converter para BSR — queremos base seca bruta
                # para ponderar a química corretamente.
                _bsr_patio_total = sum(
                    (y_ms_patio[m][ms].value() or 0)
                    * (1 - PERDA_FISICA[ms] if tipo_mp.get(m) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
                    * (1 - Umid[m]) * (1 - PPC[m])
                    for ms in MAQUINAS for m in mps
                    if Aplicacao.get(m) == "MS"
                )

                # ── Química ponderada da pilha (por óxido) ───────────────
                # Inclui contribuição dinâmica dos finos de NPO,
                # proporcional ao volume desviado ao pátio por cada MS.
                for ox in ["FET", "SIO2", "CAO", "MGO", "AL2O3", "MN", "P"]:
                    _m_ox_patio = sum(
                        (y_ms_patio[m][ms].value() or 0)
                        * (1 - PERDA_FISICA[ms] if tipo_mp.get(m) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
                        * (1 - Umid[m]) * (quimica[ox][m] / 100)
                        for ms in MAQUINAS for m in mps
                        if Aplicacao.get(m) == "MS"
                        and clean_name(m) != clean_name("Finos de NPO")
                    ) + sum(
                        # Finos de NPO: química real com penalidade de ganga,
                        # ponderada pela fração que cada MS enviou ao pátio.
                        (x_fino_npo[m][ms].value() or 0)
                        * ((quimica[ox][m] + PENAL_BLEND[ox]) / 100)
                        * (vol_ms1_patio / prod_ms1 if ms == "MS1E2" and prod_ms1 > 0 else
                           vol_ms3_patio / prod_ms3 if ms == "MS3"   and prod_ms3 > 0 else 0)
                        for m in npos_list for ms in MAQUINAS
                    )
                    _teor_ox = (_m_ox_patio / _bsr_patio_total * 100) if _bsr_patio_total > 0 else 0
                    gravar_celula(ox, _teor_ox)

                # ── Custo bruto ponderado (PRECO_CIF_BU) ─────────────────
                # Média ponderada do custo bruto de cada MS (MP + Op.Var.),
                # pesada pelo BSR desviado ao pátio por cada máquina.
                _bsr_patio_ms = {}
                for ms in MAQUINAS:
                    _bsr_patio_ms[ms] = sum(
                        (y_ms_patio[m][ms].value() or 0)
                        * (1 - PERDA_FISICA[ms] if tipo_mp.get(m) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
                        * (1 - Umid[m]) * (1 - PPC[m])
                        for m in mps if Aplicacao.get(m) == "MS"
                    )

                _custo_bruto_patio = {}
                for ms in MAQUINAS:
                    _inv_pat = sum(
                        (sum(x_ms[m][ms][af].value() or 0 for af in AFS)
                         + (y_ms_patio[m][ms].value() or 0))
                        * Preco.get(m, 0)
                        for m in mps
                    )
                    _custo_bruto_patio[ms] = (
                        _inv_pat / PROD_FIXA_MS[ms] if PROD_FIXA_MS[ms] > 0 else 0
                    ) + CUSTO_OP_VAR_MS[ms]

                _bsr_patio_soma = sum(_bsr_patio_ms.values())
                _preco_recomposto = (
                    sum(_custo_bruto_patio[ms] * _bsr_patio_ms[ms] for ms in MAQUINAS)
                    / _bsr_patio_soma
                ) if _bsr_patio_soma > 0 else 0

                gravar_celula("PRECO_CIF_BU", _preco_recomposto)

    # ------------------------------------------------------
    # B. ABA KPIS_MS (Qualidade Sínter, Granulometria e Custos) - CORRIGIDO
    # ------------------------------------------------------
    print("Preenchendo indicadores de Sinterização (Química, Física e Custo Unitário)...")
    ws_k_ms = wb["KPIS_MS"]

    # 1. MAPEAMENTO DE COLUNAS (MS1E2, MS3 e GLOBAL)
    col_ms1e2 = None; col_ms3 = None; col_global = None
    for c in range(1, ws_k_ms.max_column + 1):
        header = str(ws_k_ms.cell(row=1, column=c).value).strip().upper()
        if header == "MS1E2": col_ms1e2 = c
        if header == "MS3": col_ms3 = c
        if header == "GLOBAL": col_global = c

    # Criamos o dicionário apenas com as máquinas encontradas
    maquinas_alvo = {}
    if col_ms1e2: maquinas_alvo["MS1E2"] = col_ms1e2
    if col_ms3: maquinas_alvo["MS3"] = col_ms3

    for ms_nome, col_dest in maquinas_alvo.items():

            # Denominador usa PROD_FIXA_MS (valor exato da trava de produção),
        # consistente com custo_unit_ms e a função objetivo.
        m_bsr = PROD_FIXA_MS[ms_nome]

        # Mistura Parcial (Base Úmida, p/ Granulometria) - Sem Carga Energética
        m_mpa = sum((sum(x_ms[mp][ms_nome][af].value() or 0 for af in AFS) + (y_ms_patio[mp][ms_nome].value() or 0))
                    for mp in mps if tipo_mp.get(mp) != "CARGA_ENERGETICA")

        # Frações Granulométricas (gt100, gt635, lt105)
        # Aplicamos o coeficiente G sobre a soma total (Direto + Pátio)
        m_gt100 = sum((sum(x_ms[mp][ms_nome][af].value() or 0 for af in AFS) + (y_ms_patio[mp][ms_nome].value() or 0))
                      * G_100[mp] for mp in mps if tipo_mp.get(mp) != "CARGA_ENERGETICA")

        m_gt635 = sum((sum(x_ms[mp][ms_nome][af].value() or 0 for af in AFS) + (y_ms_patio[mp][ms_nome].value() or 0))
                      * G_635[mp] for mp in mps if tipo_mp.get(mp) != "CARGA_ENERGETICA")

        m_lt105 = sum((sum(x_ms[mp][ms_nome][af].value() or 0 for af in AFS) + (y_ms_patio[mp][ms_nome].value() or 0))
                      * G_105[mp] for mp in mps if tipo_mp.get(mp) != "CARGA_ENERGETICA")

        # Relações RN/RA
        m_fn = m_gt100 - m_gt635
        m_fa = m_lt105
        rn_valor = m_fn / (m_fn + m_fa) if (m_fn + m_fa) > 0 else 0

        # Cálculos de Produção Física (Útil vs Finos)
        prod_sinter_af = m_bsr * REND_SINTER[ms_nome]
        geracao_fino = m_bsr * (1 - REND_SINTER[ms_nome])

        # Custo Total
        # Custo MPs primárias da MS (excluindo finos NPO, que têm custo próprio abaixo)
        custo_abs_ms = sum((sum(x_ms[mp][ms_nome][af].value() or 0 for af in AFS) + (y_ms_patio[mp][ms_nome].value() or 0)) * Preco.get(mp, 0) for mp in mps)
        # Custo dos finos de NPO alocados a esta máquina (porção do custo BU do NPO)
        custo_finos_npo_abs_ms = sum(
            sum((x_dir[npo][af].value() or 0) for af in AFS) * Finos.get(npo, 0) * Preco.get(npo, 0)
            * ((x_fino_npo[npo][ms_nome].value() or 0) /
               (sum(x_fino_npo[npo][ms2].value() or 0 for ms2 in MAQUINAS) + 1e-10))
            for npo in npos_list
        )
        custo_abs_ms += custo_finos_npo_abs_ms

        # Recalcula a química considerando a contribuição dinâmica dos finos de NPO
        def calc_oxido_real(ox):
            # 1. Massa Estática (Minérios normais, ignorando a linha genérica de finos)
            # (1-PERDA_FISICA) aplicado para não-energéticos/não-reciclados — consistente
            # com massa_bsr e demais cálculos de massa que chegam ao BSR.
            m_est = sum(
                (sum(x_ms[mp][ms_nome][af].value() or 0 for af in AFS) + (y_ms_patio[mp][ms_nome].value() or 0))
                * (1 - PERDA_FISICA[ms_nome] if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
                * (1-Umid[mp]) * (quimica[ox][mp]/100)
                for mp in mps if clean_name(mp) != clean_name("Finos de NPO")
            )
            # 2. Massa Dinâmica (Química exata de cada NPO com penalidade de degradação de ganga)
            m_din = sum(
                (x_fino_npo[mp][ms_nome].value() or 0) * ((quimica[ox][mp] + PENAL_BLEND[ox]) / 100)
                for mp in npos_list
            )
            return m_est + m_din

        m_cao = calc_oxido_real("CAO")
        m_sio2 = calc_oxido_real("SIO2")

        # --- PREENCHIMENTO DAS LINHAS NO EXCEL (CORRIGIDO E SEGURO) ---
        for r in range(2, 45):
            label_celula = ws_k_ms.cell(row=r, column=1).value
            if label_celula is None: continue

            # 1. DEFINIMOS OS ALVOS PRIMEIRO
            label = str(label_celula).strip().upper()
            target = ws_k_ms.cell(row=r, column=col_dest)

            # 2. AGORA DISTRIBUÍMOS OS VALORES (Sequência Lógica Correta)
            if label == "PROD_BSR":
                target.value = m_bsr
            elif label == "PRODUCAO_SINTER_AF":
                target.value = prod_sinter_af
            elif label == "GERACAO_SINTER_FINO":
                target.value = geracao_fino

            # --- NOVA LINHA: CARGA ENERGÉTICA (Otimizada) ---
            elif label == "CARGA_ENERGETICA":
                # Calcula o consumo específico real obtido (em kg/t BSR)
                m_ener_bs = sum((sum(x_ms[m][ms_nome][af].value() or 0 for af in AFS) + (y_ms_patio[m][ms_nome].value() or 0))
                                * (1 - Umid[m]) for m in mps if tipo_mp.get(m) == "CARGA_ENERGETICA")
                target.value = (m_ener_bs * 1000) / m_bsr if m_bsr > 0 else 0

            # Teores Químicos (Balanço de Massa preservado e corrigido)
            elif label in ["FET", "SIO2", "AL2O3", "CAO", "MGO", "MN", "P"]:
                m_elem = calc_oxido_real(label)
                target.value = (m_elem / m_bsr) * 100 if m_bsr > 0 else 0

            # Indicadores Físicos e Basicidade (Avanços Anteriores)
            elif label == "RN": target.value = rn_valor
            elif label == "RA": target.value = 1 - rn_valor if (m_fn + m_fa) > 0 else 0
            elif label == ">1MM": target.value = (m_gt100 / m_mpa) * 100 if m_mpa > 0 else 0
            elif label == ">6.3MM": target.value = (m_gt635 / m_mpa) * 100 if m_mpa > 0 else 0
            elif label == "<0.105MM": target.value = (m_lt105 / m_mpa) * 100 if m_mpa > 0 else 0
            elif label == "B2": target.value = m_cao / m_sio2 if m_sio2 > 0 else 0

            # Custo Unitário (Avanço Financeiro) — inclui custo operacional variável da MS
            elif label == "CUSTO_SINTER":
                target.value = (custo_abs_ms / m_bsr if m_bsr > 0 else 0) + CUSTO_OP_VAR_MS[ms_nome]

            elif label == "CUSTO_SINTER_AF":
                # Custo do sínter útil pós-manuseio com crédito do sínter fino (US$/t sínter útil)
                # Fórmula: (CUSTO_SINTER / REND) - ((1-REND) × Vol_Fino × CIF_Fino / Vol_Sinter_Útil)
                # Onde:
                #   CUSTO_SINTER   = custo do BSR bruto produzido (~86 US$/t)
                #   REND           = rendimento da sinterização (~86,1%)
                #   Vol_Fino       = sínter fino gerado = m_bsr × (1-REND)
                #   CIF_Fino       = preço CIF B.U. do Sínter Fino (aba MATERIAS_PRIMAS)
                #   Vol_Sinter_Útil= sínter útil para os AFs = m_bsr × REND
                _custo_sinter_bruto = (custo_abs_ms / m_bsr if m_bsr > 0 else 0) + CUSTO_OP_VAR_MS[ms_nome]
                _cif_fino = preco_limpo.get(
                    next((mp for mp in mps if "SINTER FINO" in mp.upper()), ""), 0
                )
                _credito_fino = (
                    (1 - REND_SINTER[ms_nome]) * geracao_fino * _cif_fino / prod_sinter_af
                    if prod_sinter_af > 0 else 0
                )
                target.value = (_custo_sinter_bruto / REND_SINTER[ms_nome] if REND_SINTER[ms_nome] > 0 else _custo_sinter_bruto) - _credito_fino

        # --- COLUNA GLOBAL: calculada UMA VEZ por máquina, fora do for r acima ---
        # Antes estava dentro do for r — executava 43× por máquina desnecessariamente.
        # Movido para fora: executa 1× por máquina, resultado idêntico.
        if col_global:
            sinter_ef_direto = sum(
                (x_ms[mp][ms][af].value() or 0.0)
                * (1 - PERDA_FISICA[ms] if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
                * (1-Umid[mp])
                * (1 - (PPC_FINOS_EFETIVO if clean_name(mp) == clean_name("Finos de NPO") else PPC[mp]))
                * REND_SINTER[ms]
                for mp in mps for ms in MAQUINAS for af in AFS if Aplicacao.get(mp) == "MS"
            )

            sinter_ef_patio = sum(
                (z_patio_af[af].value() or 0.0) * (1-Umid["PATIO RETOMADO"]) * (1-PPC["PATIO RETOMADO"]) * rend_medio
                for af in AFS
            )

            sinter_af_efetivo_real = sinter_ef_direto + sinter_ef_patio

            for r in range(2, 45):
                label = str(ws_k_ms.cell(row=r, column=1).value).strip().upper()
                target = ws_k_ms.cell(row=r, column=col_global)

                if label == "DELTA_ESTOQUE": target.value = DELTA_ESTOQUE
                elif label == "SINTER_AF_EF": target.value = sinter_af_efetivo_real
                elif label == "ESTOQUE_PP_INICIAL": target.value = ESTOQUE_INI
                elif label == "ESTOQUE_PP_FINAL": target.value = ESTOQUE_FIN
                elif label == "ESTOQUE_MAX_PATIO": target.value = ESTOQUE_MAX_PATIO
                elif label == "CUSTO_SINTER_RETOMADO":
                        # Custo total do sínter retomado (US$/t B.U.) — input direto do Excel.
                        # Zero se não houver retomada no período (DELTA_ESTOQUE >= 0).
                        _vol_retomado_bu = sum(
                            z_patio_af[af].value() or 0 for af in AFS
                        )
                        target.value = (
                            preco_limpo.get("PATIO RETOMADO", 0)
                            if _vol_retomado_bu > 0 else 0
                        )

    # ------------------------------------------------------
    # C. ABA KPIS_AF (Qualidade Gusa e Escória - REVISADO)
    # ------------------------------------------------------
    print("Gravando KPIs do Alto Forno na aba KPIS_AF...")
    ws_k_af = wb["KPIS_AF"]

    # 1. Mapeamento de Colunas (Procura AF2 e AF3 na linha 1)
    c_af = {str(cell.value).strip().upper(): cell.column for cell in ws_k_af[1] if cell.value}

    # Coluna B (2) contém os KPIs conforme seu arquivo
    col_kpi_af = 2

    # Pré-cálculo de Custo Unitário do Sínter (US$/t BSR)
    # Custo unitário por máquina: finos de NPO não entram (custo já em custo_af_direto),
    # denominador fixo em PROD_FIXA_MS para evitar ruído do solver.
    # CUSTO_OP_VAR_MS: aplica-se a todo sínter produzido (direto + pátio),
    # não à retomada (cujo preço já contempla o custo original de produção).
    custo_unit_ms = {}
    for ms in MAQUINAS:
        invest_total = sum(
            (sum(x_ms[mp][ms][af].value() or 0 for af in AFS) + (y_ms_patio[mp][ms].value() or 0))
            * Preco.get(mp, 0)
            for mp in mps
            if clean_name(mp) != clean_name("Finos de NPO")
        )
        # Finos NPO alocados a esta MS: custo proporcional ao fluxo físico
        custo_finos_npo_ms = sum(
            sum((x_dir[npo][af].value() or 0) for af in AFS) * Finos.get(npo, 0) * Preco.get(npo, 0)
            * ((x_fino_npo[npo][ms].value() or 0) /
               (sum(x_fino_npo[npo][ms2].value() or 0 for ms2 in MAQUINAS) + 1e-10))
            for npo in npos_list
        )
        _cu_bruto = ((invest_total + custo_finos_npo_ms) / PROD_FIXA_MS[ms] if PROD_FIXA_MS[ms] > 0 else 0) + CUSTO_OP_VAR_MS[ms]
        # Crédito do sínter fino como subproduto: divide pelo REND para alocar
        # o custo total apenas ao sínter útil que entra no Alto Forno.
        custo_unit_ms[ms] = _cu_bruto / REND_SINTER[ms] if REND_SINTER[ms] > 0 else _cu_bruto

    for af in AFS:
        col_dest = c_af.get(af)
        if not col_dest: continue
        v_g = gusa[af].value() or 1

        # CÁLCULO DAS TAXAS DE CARBONOSOS OBTIDAS
        c_coque_s = sum((x_dir[mp][af].value() or 0) * (1 - Umid[mp]) for mp in mps if tipo_mp.get(mp) == "CARBONOSO_COQUE")
        c_pci_s   = sum((x_dir[mp][af].value() or 0) * (1 - Umid[mp]) for mp in mps if tipo_mp.get(mp) == "CARBONOSO_PCI")
        coque_r = (c_coque_s * 1000) / v_g
        pci_r   = (c_pci_s * 1000) / v_g

        # LOOP DE PREENCHIMENTO DAS LINHAS
        for r in range(2, ws_k_af.max_row + 1):
            label = str(ws_k_af.cell(row=r, column=col_kpi_af).value).strip().upper()
            target = ws_k_af.cell(row=r, column=col_dest)

            # Carbonosos (Inseridos no loop correto)
            if label == "COQUE_RATE":   target.value = coque_r
            elif label == "PCI_RATE":   target.value = pci_r
            elif label == "FUEL_RATE":  target.value = coque_r + pci_r

        # 2. Cálculos de Massa de Carga Metálica (Base Seca e Peneirada)
        m_sinter_direto_af = sum(
            (x_ms[mp][ms][af].value() or 0.0)
            * (1 - PERDA_FISICA[ms] if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
            * (1-Umid[mp])
            * (1 - (PPC_FINOS_EFETIVO if clean_name(mp) == clean_name("Finos de NPO") else PPC[mp]))
            * REND_SINTER[ms]
            for mp in mps for ms in MAQUINAS
        )

        m_sinter_patio_af = (z_patio_af[af].value() or 0.0) * (1-Umid["PATIO RETOMADO"]) * (1-PPC["PATIO RETOMADO"]) * rend_medio

        m_sinter_af = m_sinter_direto_af + m_sinter_patio_af

        def m_direta_tipo(tipo):
            return sum((x_dir[mp][af].value() or 0.0) * (1-Finos[mp]) * (1-Umid[mp]) for mp in mps if tipo_mp.get(mp) == tipo)

        m_npo_af    = m_direta_tipo("NPO")
        m_pelota_af = m_direta_tipo("PELOTA")
        m_briq_af   = m_direta_tipo("BRIQUETE")
        carga_met_total = m_sinter_af + m_npo_af + m_pelota_af + m_briq_af

        # 3. Balanço Químico Exato para o Relatório
        def get_massa_input(elem):
            # Carga Direta
            m_dir = sum((x_dir[mp][af].value() or 0.0) * (1-Finos[mp]) * (1-Umid[mp]) * (quimica[elem][mp]/100) for mp in mps if Aplicacao.get(mp) == "AF")

            # Sínter Estático (excluindo a linha falsa)
            m_est = sum(
                (x_ms[mp][ms][af].value() or 0.0)
                * (1 - PERDA_FISICA[ms] if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
                * (1-Umid[mp]) * (quimica[elem][mp]/100) * REND_SINTER[ms]
                for mp in mps for ms in MAQUINAS if clean_name(mp) != clean_name("Finos de NPO")
            )

            # Sínter Dinâmico (NPOs com penalidade de degradação de ganga)
            m_din = sum(
                (x_fino_npo[mp][ms].value() or 0.0) * ((quimica[elem][mp] + PENAL_BLEND[elem]) / 100)
                * REND_SINTER[ms] * FATOR_ROTA[ms][af]
                for mp in npos_list for ms in MAQUINAS
            )

            # Sínter Retomado do Pátio
            m_patio = (z_patio_af[af].value() or 0.0) * (1-Umid["PATIO RETOMADO"]) * (quimica[elem].get("PATIO RETOMADO", 0)/100) * rend_medio

            return m_dir + m_est + m_din + m_patio

        # Escória
        # CORREÇÃO: SiO2 da escória = SiO2 da carga × (1 - alfa_SI) + quartzo puro
        # O quartzo vai 100% à escória; não sofre redução pelo alfa_SI.
        # Consistente com a restrição de basicidade na seção 4.3.
        cao_e   = get_massa_input("CAO")
        mgo_e   = get_massa_input("MGO")
        al2o3_e = get_massa_input("AL2O3")
        sio2_da_carga = get_massa_input("SIO2")
        v_quartzo = x_quartzo[af].value() or 0
        sio2_e = sio2_da_carga * (1 - ALFA[af]["SI"]) + v_quartzo

        # Massa corrigida com o fator de calibração do respectivo AF
        m_slag_total = (cao_e + mgo_e + al2o3_e + sio2_e) / FATOR_OXIDOS[af]

        # Gusa
        # O Si metálico no gusa vem apenas da SiO2 da carga, nunca do quartzo puro
        mn_g = get_massa_input("MN") * ALFA[af]["MN"]
        p_g  = get_massa_input("P") * ALFA[af]["P"]
        si_g = sio2_da_carga * ALFA[af]["SI"] * (28.08 / 60.08)

        # Custos
        # Custo direto: NPOs pagam só pela porção útil (1-Finos); finos já em custo_unit_ms
        custo_dir = (
            sum((x_dir[mp][af].value() or 0.0) * (1 - Finos.get(mp, 0)) * Preco.get(mp, 0)
                for mp in mps if tipo_mp.get(mp) == "NPO")
            + sum((x_dir[mp][af].value() or 0.0) * Preco.get(mp, 0)
                  for mp in mps if tipo_mp.get(mp) != "NPO")
        )

        # CIF do Sínter Fino lido do dicionário de preços (aba MATERIAS_PRIMAS)
        _mp_sinter_fino = next((mp for mp in mps if "SINTER FINO" in mp.upper()), "")
        _cif_fino_af = preco_limpo.get(_mp_sinter_fino, 0)

        # custo_sinter_af: custo do sínter útil pós-manuseio com crédito do fino
        # Fórmula: (cu_bruto / REND) - ((1-REND)/REND) × CIF_fino
        # O segundo termo é o crédito gerado no peneiramento:
        # o sínter fino retorna ao processo valendo CIF_fino por tonelada.
        _custo_sinter_af = {}
        for ms in MAQUINAS:
            _inv_af = sum(
                (sum(x_ms[mp][ms][af2].value() or 0 for af2 in AFS)
                 + (y_ms_patio[mp][ms].value() or 0))
                * preco_limpo.get(mp, 0)
                for mp in mps if clean_name(mp) != clean_name("Finos de NPO")
            )
            _finos_npo_af = sum(
                sum((x_dir[npo][af2].value() or 0) for af2 in AFS)
                * Finos.get(npo, 0) * preco_limpo.get(npo, 0)
                * ((x_fino_npo[npo][ms].value() or 0)
                   / (sum(x_fino_npo[npo][ms2].value() or 0 for ms2 in MAQUINAS) + 1e-10))
                for npo in npos_list
            )
            _cu_bruto_af = (
                (_inv_af + _finos_npo_af) / PROD_FIXA_MS[ms]
                if PROD_FIXA_MS[ms] > 0 else 0
            ) + CUSTO_OP_VAR_MS[ms]
            _credito_af = (
                ((1 - REND_SINTER[ms]) / REND_SINTER[ms]) * _cif_fino_af
                if REND_SINTER[ms] > 0 else 0
            )
            _custo_sinter_af[ms] = (
                _cu_bruto_af / REND_SINTER[ms] - _credito_af
                if REND_SINTER[ms] > 0 else _cu_bruto_af
            )

        custo_sin_direto = sum(
            sum((x_ms[mp][ms][af].value() or 0.0)
                * (1 - PERDA_FISICA[ms] if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
                * (1-Umid[mp])
                * (1 - (PPC_FINOS_EFETIVO if clean_name(mp) == clean_name("Finos de NPO") else PPC[mp]))
                * REND_SINTER[ms]
                * _custo_sinter_af[ms]   # ← usa custo com crédito do fino
                for mp in mps)
            for ms in MAQUINAS
        )

        # Custo do sínter do pátio — média ponderada pela produção de cada máquina
        custo_unit_patio = (
            sum(_custo_sinter_af[ms] * PROD_FIXA_MS[ms] for ms in MAQUINAS)
            / sum(PROD_FIXA_MS.values())
        ) if sum(PROD_FIXA_MS.values()) > 0 else 0

        # Massa do sínter retomado do pátio para custeio
        m_sinter_patio_af_custo = (z_patio_af[af].value() or 0.0) * (1-Umid["PATIO RETOMADO"]) * (1-PPC["PATIO RETOMADO"])
        custo_sin_patio = m_sinter_patio_af_custo * custo_unit_patio

        custo_sin = custo_sin_direto + custo_sin_patio

        custo_qz  = (x_quartzo[af].value() or 0.0) * 40.0
        # CUSTO_OP_VAR_AF: custo operacional variável do forno (US$/t gusa), lido do Excel
        custo_total_af = custo_dir + custo_sin + custo_qz + CUSTO_OP_VAR_AF[af] * v_g

        # 4. Preenchimento (CORREÇÃO DA VARIÁVEL TARGET AQUI)
        for r in range(2, ws_k_af.max_row + 1):
            label = str(ws_k_af.cell(row=r, column=col_kpi_af).value).strip().upper()
            target = ws_k_af.cell(row=r, column=col_dest)

            if label == "SINTER":   target.value = (m_sinter_af / carga_met_total * 100) if carga_met_total > 0 else 0
            elif label == "NPO":    target.value = (m_npo_af / carga_met_total * 100) if carga_met_total > 0 else 0
            elif label == "PELOTA": target.value = (m_pelota_af / carga_met_total * 100) if carga_met_total > 0 else 0
            elif label == "BRIQUETE": target.value = (m_briq_af / carga_met_total * 100) if carga_met_total > 0 else 0
            elif label == "PAM_CM": target.value = (carga_met_total * 1000) / v_g
            elif label == "PROD_GUSA": target.value = v_g
            elif label == "FET":       target.value = 94.50
            elif label == "MN":        target.value = (mn_g / v_g) * 100
            elif label == "P":         target.value = (p_g / v_g) * 100
            elif label == "SI":        target.value = (si_g / v_g) * 100
            elif label == "SLAG_RATE": target.value = (m_slag_total * 1000) / v_g
            elif label == "CAO":       target.value = (cao_e / m_slag_total) * 100 if m_slag_total > 0 else 0
            elif label == "SIO2":      target.value = (sio2_e / m_slag_total) * 100 if m_slag_total > 0 else 0
            elif label == "MGO":       target.value = (mgo_e / m_slag_total) * 100 if m_slag_total > 0 else 0
            elif label == "AL2O3":     target.value = (al2o3_e / m_slag_total) * 100 if m_slag_total > 0 else 0
            elif label == "B2":        target.value = cao_e / sio2_e if sio2_e > 0 else 0
            elif label == "B3":        target.value = (cao_e + mgo_e) / sio2_e if sio2_e > 0 else 0
            elif label == "QUARTZO":   target.value = ((x_quartzo[af].value() or 0) * 1000) / v_g
            elif label == "CUSTO_GUSA": target.value = custo_total_af / v_g

    # ── Custo do gusa global ponderado pela produção ──────────────────────
    # Usa a coluna GLOBAL já existente no Excel (criada pelo usuário),
    # localizando-a pelo cabeçalho — sem criar colunas novas.
    col_global = c_af.get("GLOBAL")

    if col_global:
        _prod_af2  = gusa["AF2"].value() or 0
        _prod_af3  = gusa["AF3"].value() or 0
        _prod_total = _prod_af2 + _prod_af3
        _col_af2   = c_af.get("AF2")
        _col_af3   = c_af.get("AF3")

        for r in range(2, ws_k_af.max_row + 1):
            label = str(ws_k_af.cell(row=r, column=col_kpi_af).value).strip().upper()
            if label == "CUSTO_GUSA" and _prod_total > 0 and _col_af2 and _col_af3:
                _cg_af2    = ws_k_af.cell(row=r, column=_col_af2).value or 0
                _cg_af3    = ws_k_af.cell(row=r, column=_col_af3).value or 0
                _cg_global = (_cg_af2 * _prod_af2 + _cg_af3 * _prod_af3) / _prod_total
                ws_k_af.cell(row=r, column=col_global).value = round(_cg_global, 4)




    # ======================================================
    # 7. ANÁLISE VIU (Value In Use)
    # ======================================================
    import sys as _sys, io as _io, datetime as _dt
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter

    # ── Parâmetros VIU ──────────────────────────────────────────────────────
    _LOTE      = 6000
    _TOL       = 0.05
    _MAX_IT    = 20
    _TIPOS_VIU = {"MINERIO", "NPO", "PELOTA", "BRIQUETE"}
    _FO_BASE   = pl.value(modelo.objective)
    _PROD_VIU  = sum(gusa[af].value() or 0 for af in AFS)
    _CG_BASE   = _FO_BASE / _PROD_VIU if _PROD_VIU > 0 else 0
    _V_ORIG    = {v.name: v.varValue for v in modelo.variables()}
    _viu_lock  = [False]   # guard contra dupla execução
    # Teto da busca binária: garante que VIU real > 500 US$/t não seja
    # classificado como ESTRUTURAL incorretamente. Dinâmico por material
    # (veja uso em _viu_marginal). Valor fixo de referência = 1000 US$/t.
    _VIU_TETO  = 1000.0

    # ── Helpers internos ────────────────────────────────────────────────────
    def _copia_sem_disp(mp):
        m = modelo.copy()
        chaves = {f"DispMin_{mp}", f"DispMax_{mp}",
                  f"DispMin_{clean_name(mp)}", f"DispMax_{clean_name(mp)}"}
        for k in list(m.constraints.keys()):
            if k in chaves:
                del m.constraints[k]
        return m

    def _expr_cons(mp):
        if Aplicacao.get(mp) == "MS":
            return (pl.lpSum(x_ms[mp][ms][af] for ms in MAQUINAS for af in AFS)
                    + pl.lpSum(y_ms_patio[mp][ms] for ms in MAQUINAS))
        return pl.lpSum(x_dir[mp][af] for af in AFS)

    def _val_cons(mp):
        if Aplicacao.get(mp) == "MS":
            return sum(x_ms[mp][ms][af].varValue or 0 for ms in MAQUINAS for af in AFS)
        return sum(x_dir[mp][af].varValue or 0 for af in AFS)

    def _solve_s(m_tmp):
        old = _sys.stdout; _sys.stdout = _io.StringIO()
        st  = m_tmp.solve(pl.PULP_CBC_CMD(msg=False))
        _sys.stdout = old
        if pl.LpStatus[st] != "Optimal": return None
        return pl.value(m_tmp.objective)

    def _ajusta_preco(m_tmp, mp, delta):
        if abs(delta) < 1e-9: return
        if Aplicacao.get(mp) == "MS":
            adj = pl.lpSum((pl.lpSum(x_ms[mp][ms][af] for af in AFS)
                            + y_ms_patio[mp][ms]) * delta for ms in MAQUINAS)
        else:
            adj = pl.lpSum(x_dir[mp][af] * delta for af in AFS)
        m_tmp.objective = m_tmp.objective + adj

    def _restaura():
        for v in modelo.variables():
            v.varValue = _V_ORIG.get(v.name)

    # ── VIU Marginal (busca binária) ─────────────────────────────────────────
    def _viu_marginal(mp, cb):
        cif  = preco_limpo.get(mp, 0)
        dmax = DispMax.get(mp, 0) if pd.notnull(DispMax.get(mp, 0)) else 0
        if pd.isna(dmax) or dmax < _LOTE:
            return None, None, "INDISPONIVEL", 0

        # Teto dinâmico por material: max entre _VIU_TETO e 5× o CIF.
        # Garante que materiais com CIF alto tenham espaço de busca suficiente
        # e que VIU real > 500 US$/t não seja classificado como ESTRUTURAL.
        _teto = max(_VIU_TETO, cif * 5.0)

        if cb < _LOTE - 1:
            # Método A: entrada espontânea [0, _teto]
            lo, hi = 0.0, _teto; viu = None; lres = 0
            for _ in range(_MAX_IT):
                mid = (lo + hi) / 2.0
                mt  = _copia_sem_disp(mp); _ajusta_preco(mt, mp, mid - cif)
                fo  = _solve_s(mt)
                if fo is None: hi = mid; continue
                cv  = _val_cons(mp)
                if cv > 0.1: lo = mid; viu = mid; lres = cv
                else:        hi = mid
                if (hi - lo) < _TOL: break
            _restaura()
            if viu is None: return None, None, "NAO_CONV", 0
            esp = viu - cif
            sta = "VIAVEL" if esp > 1 else "MARGINAL" if esp >= -1 else "PEDIR_DESCONTO"
            return round(viu, 2), round(esp, 2), sta, round(lres)
        else:
            # Método B: saída espontânea [CIF, _teto]
            # ESTRUTURAL apenas se não reduzir nem ao teto dinâmico.
            # Threshold adaptativo: evita falso positivo por ruído numérico
            # do CBC em materiais de grande volume.
            _threshold = max(0.1, cb * 0.001)
            lo, hi = cif, _teto; viu = None; lres = 0
            for _ in range(_MAX_IT):
                mid = (lo + hi) / 2.0
                mt  = _copia_sem_disp(mp); _ajusta_preco(mt, mp, mid - cif)
                fo  = _solve_s(mt)
                if fo is None: lo = mid; continue
                cv  = _val_cons(mp)
                if cv < cb - _threshold: hi = mid; viu = mid; lres = cb - cv
                else:                    lo = mid
                if (hi - lo) < _TOL: break
            _restaura()
            if viu is None:
                return None, None, "ESTRUTURAL", 0
            esp = viu - cif
            sta = "VIAVEL" if esp > 1 else "MARGINAL" if esp >= -1 else "PEDIR_DESCONTO"
            return round(viu, 2), round(esp, 2), sta, max(1, round(lres))

    # ── VIU por Lote (2 runs) ───────────────────────────────────────────────
    def _viu_lote(mp, cb):
        cif  = preco_limpo.get(mp, 0)
        dmax = DispMax.get(mp, 0) if pd.notnull(DispMax.get(mp, 0)) else 0

        # Disponibilidade real limpa (0 se nula)
        dmax_real = dmax if not pd.isna(dmax) else 0

        # Hipotético: quando a margem disponível (dmax - consumo_base) é inferior
        # ao lote forçado. Cobre todos os casos sem disponibilidade:
        #   · dmax = 0 ou NaN     → sem contrato
        #   · dmax < _LOTE        → contrato menor que o lote
        #   · dmax - cb < _LOTE   → contrato saturado (esgotado total ou parcialmente)
        # Nesses casos, criamos _LOTE t artificialmente para avaliar o potencial
        # técnico do material — o resultado é sinalizado como HIPOTÉTICO.
        hipotetico = (dmax_real - cb) < _LOTE
        dmax_efetivo = max(dmax_real, cb + _LOTE)

        mc1  = _copia_sem_disp(mp)
        expr = _expr_cons(mp)
        if cb < _LOTE - 1:
            mc1 += expr >= _LOTE,                      f"VL_Min_{clean_name(mp)}"
            mc1 += expr <= max(dmax_efetivo, _LOTE),   f"VL_Max_{clean_name(mp)}"
        else:
            mc1 += expr >= cb + _LOTE,                 f"VL_Min_{clean_name(mp)}"
            mc1 += expr <= dmax_efetivo + _LOTE,       f"VL_Max_{clean_name(mp)}"
        fo_c1 = _solve_s(mc1); _restaura()
        if fo_c1 is None:
            # Infeasível por restrição técnica (química, escória, etc.) — não por disponibilidade
            return None, None, "INFEASIVEL", False
        viu = (_FO_BASE - fo_c1) / _LOTE + cif
        esp = viu - cif
        if hipotetico:
            sta = "HIPOTETICO"
        else:
            sta = "VIAVEL" if esp > 1 else "MARGINAL" if esp >= -1 else "PEDIR_DESCONTO"
        return round(viu, 2), round(esp, 2), sta, True

    # ── Observação executiva ────────────────────────────────────────────────
    def _obs_exec(mp, sb, sl, ok_lot, cb):
        dmax = DispMax.get(mp, 0) if pd.notnull(DispMax.get(mp, 0)) else 0
        if sb == "INDISPONIVEL":
            return "Volume máximo contratado abaixo do lote mínimo de análise (6.000 t)"
        if sb == "ESTRUTURAL":
            return "Indispensável — mantém consumo mesmo acima de 500 US$/t. Sem substituto identificado"
        if not ok_lot and sb not in ("INDISPONIVEL", "ESTRUTURAL"):
            return "Teto físico: basicidade/escória impedem absorção de mais um lote nas condições atuais"
        if sl == "HIPOTETICO":
            esp_l = None
            # Recuperar o espaço do lote dos dados (não disponível aqui diretamente,
            # então usamos a observação genérica com nota hipotética)
            if sb == "INDISPONIVEL":
                return "Sem contrato ativo — análise hipotética indica potencial técnico: avaliar negociação"
            return "⚠️ Volume hipotético (sem disponibilidade contratada) — VIU por Lote calculado com 6.000 t artificiais"
        if sb == "PEDIR_DESCONTO" and sl in ("PEDIR_DESCONTO", "INFEASIVEL", None):
            return "Inviável em ambas as métricas — solicitar desconto urgente ou revisar participação"
        if sb == "VIAVEL" and sl == "PEDIR_DESCONTO":
            if dmax > 0 and cb >= dmax * 0.95:
                return "Processo saturado — manter volume atual, não expandir"
            return "Volume atual competitivo; próxima composição não se paga — manter contrato sem expansão"
        if sb == "VIAVEL" and sl == "VIAVEL":
            return "Competitivo em ambas as métricas — pode negociar volume adicional"
        if sb == "VIAVEL" and sl == "MARGINAL":
            return "Volume atual viável; próxima composição limítrofe — não expandir sem renegociação"
        if sb == "MARGINAL":
            return "Margem estreita — monitorar variações de preço"
        return "—"

    # ── Engine: calcula VIU duplo ────────────────────────────────────────────
    def _engine_viu_duplo():
        dados = []
        for mp in mps:
            tipo = str(tipo_mp.get(mp, "")).upper().strip()
            if tipo not in _TIPOS_VIU or clean_name(mp) == clean_name("Finos de NPO"):
                continue
            cif  = preco_limpo.get(mp, 0)
            dmax = DispMax.get(mp, 0) if pd.notnull(DispMax.get(mp, 0)) else 0
            cb   = _val_cons(mp)
            vb, eb, sb, _ = _viu_marginal(mp, cb)
            vl, el, sl, ok_l = _viu_lote(mp, cb)
            if sb == "INDISPONIVEL":   sinal = "INDISPONIVEL"
            elif sb == "ESTRUTURAL":   sinal = "ESTRUTURAL"
            elif sl == "HIPOTETICO":   sinal = "HIPOTETICO"
            elif not ok_l:             sinal = "TETO_FISICO"
            elif sb == sl:             sinal = "CONVERGEM"
            else:                      sinal = "DIVERGEM"
            obs = _obs_exec(mp, sb, sl, ok_l, cb)
            dados.append({"MP": mp, "Tipo": tipo,
                          "Base": round(cb), "DispMax": round(dmax),
                          "CIF": round(cif, 2),
                          "VIU_B": vb, "Esp_B": eb, "Sta_B": sb,
                          "VIU_L": vl, "Esp_L": el, "Sta_L": sl,
                          "Sinal": sinal, "Obs": obs})
        return dados

    # ── Relatório Excel executivo ────────────────────────────────────────────
    def _escrever_relatorio_viu(wb_dst, dados):
        if "VIU_ANALISE" in wb_dst.sheetnames:
            del wb_dst["VIU_ANALISE"]
        ws = wb_dst.create_sheet("VIU_ANALISE")

        H1,H2,H3 = "0D3D6E","1565C0","1976D2"; BGL = "EBF4FC"
        VDF,VD2  = "C8E6C9","2E7D32"; AM,AMF = "BF360C","FFF3E0"
        VM,VMF   = "B71C1C","FFCDD2"; RX,RXF = "4A148C","EDE7F6"
        GY,GYF   = "546E7A","ECEFF1"; INF,INFF = "37474F","CFD8DC"
        SEP      = "BBDEFB"; ZB = "F5FAFF"; WH = "FFFFFF"

        def fi(h):  return PatternFill("solid", start_color=h, end_color=h)
        def fo(b=False,c="212121",s=9,i=False):
            return Font(bold=b,color=c,size=s,italic=i,name="Calibri")
        def bd(c=SEP):
            t=Side(style="thin",color=c); return Border(left=t,right=t,top=t,bottom=t)
        def al(h="center",v="center",w=True,ind=0):
            return Alignment(horizontal=h,vertical=v,wrap_text=w,indent=ind)
        def sc(r,c,v,f=None,fn=None,br=None,a=None,fmt=None):
            try:
                cell=ws.cell(row=r,column=c,value=v)
                if f:   cell.fill=f
                if fn:  cell.font=fn
                if br:  cell.border=br
                if a:   cell.alignment=a
                if fmt: cell.number_format=fmt
            except AttributeError: pass
        def mg(r1,c1,r2,c2,v,f=None,fn=None,br=None,a=None):
            ws.merge_cells(start_row=r1,start_column=c1,end_row=r2,end_column=c2)
            sc(r1,c1,v,f,fn,br,a)

        WDS=[6,21,10,10,9,1.5,9,9,13,1.5,9,9,13,1.5,13,33]
        for i,w in enumerate(WDS,1): ws.column_dimensions[get_column_letter(i)].width=w
        NCOL=16; SEPS={6,10,14}

        ws.row_dimensions[1].height=26
        mg(1,1,1,NCOL,
           f"  VIU (Value In Use) — RELATÓRIO EXECUTIVO   ·   "
           f"Custo base: {_CG_BASE:.2f} US$/t gusa   ·   "
           f"Produção: {_PROD_VIU/1000:.1f} kt   ·   "
           f"Lote: {_LOTE:,} t   ·   Teto busca: {_VIU_TETO:.0f} US$/t   ·   {_dt.date.today().strftime('%d/%m/%Y')}",
           fi(H1),fo(True,"FFFFFF",11),bd(H1),al("left",ind=1))
        ws.row_dimensions[2].height=4

        ws.row_dimensions[3].height=15
        mg(3,1,3,5,"  MATERIAL",fi(H1),fo(True,"FFFFFF",8),bd(H1),al("center"))
        sc(3,6,"",fi(SEP))
        mg(3,7,3,9,"  VIU MARGINAL",fi(H2),fo(True,"FFFFFF",8),bd(H2),al("center"))
        sc(3,10,"",fi(SEP))
        mg(3,11,3,13,"  VIU POR LOTE",fi(H3),fo(True,"FFFFFF",8),bd(H3),al("center"))
        sc(3,14,"",fi(SEP))
        mg(3,15,3,16,"  ANÁLISE",fi(H1),fo(True,"FFFFFF",8),bd(H1),al("center"))

        ws.row_dimensions[4].height=26
        HDRS=[(1,"Tipo",H1),(2,"Material",H1),(3,"Base (t)",H1),(4,"Disp.Max (t)",H1),
              (5,"CIF\n(US$/t)",H1),(7,"VIU\n(US$/t)",H2),(8,"Espaço\n(US$/t)",H2),
              (9,"Status",H2),(11,"VIU\n(US$/t)",H3),(12,"Espaço\n(US$/t)",H3),
              (13,"Status",H3),(15,"Sinal",H1),(16,"Observação Executiva",H1)]
        for col,lbl,cor in HDRS:
            sc(4,col,lbl,fi(cor),fo(True,"FFFFFF",8),bd(SEP),al("center","center",True))
        for s in SEPS: sc(4,s,"",fi(SEP))

        HP,HPF = "E65100","FFF8E1"   # laranja escuro / fundo âmbar claro — sinaliza hipotético
        def sta_fi(sta): return {"VIAVEL":fi(VDF),"MARGINAL":fi(AMF),"PEDIR_DESCONTO":fi(VMF),
            "ESTRUTURAL":fi(RXF),"INFEASIVEL":fi(INFF),"INDISPONIVEL":fi(GYF),
            "HIPOTETICO":fi(HPF),"NAO_CONV":fi(GYF)}.get(sta,fi(WH))
        def sta_fo(sta): return {"VIAVEL":fo(True,VD2,9),"MARGINAL":fo(True,AM,9),
            "PEDIR_DESCONTO":fo(True,VM,9),"ESTRUTURAL":fo(True,RX,9),
            "INFEASIVEL":fo(True,INF,9),"INDISPONIVEL":fo(True,GY,9),
            "HIPOTETICO":fo(True,HP,9),"NAO_CONV":fo(True,GY,9)}.get(sta,fo(s=9))
        def sta_lbl(sta): return {"VIAVEL":"VIÁVEL","MARGINAL":"MARGINAL",
            "PEDIR_DESCONTO":"PEDIR DESCONTO","ESTRUTURAL":"ESTRUTURAL",
            "INFEASIVEL":"INFEASÍVEL","INDISPONIVEL":"INDISPONÍVEL",
            "HIPOTETICO":"HIPOTÉTICO","NAO_CONV":"N/A"}.get(sta,sta)
        def sinal_fmt(sn): return {
            "CONVERGEM":   ("✅  CONVERGEM",   fi(VDF),  fo(True,VD2,9)),
            "DIVERGEM":    ("⚠️  DIVERGEM",    fi(AMF),  fo(True,AM,9)),
            "TETO_FISICO": ("🔒  TETO FÍSICO", fi(INFF), fo(True,INF,9)),
            "ESTRUTURAL":  ("🔷  ESTRUTURAL",  fi(RXF),  fo(True,RX,9)),
            "HIPOTETICO":  ("🔬  HIPOTÉTICO",  fi(HPF),  fo(True,HP,9)),
            "INDISPONIVEL":("—",               fi(GYF),  fo(False,GY,9)),
        }.get(sn,("—",fi(WH),fo(s=9)))
        def esp_cor(v):
            if v is None: return fo(False,GY,9)
            if v>1:   return fo(True,VD2,9)
            if v>=-1: return fo(True,AM,9)
            return          fo(True,VM,9)

        ORD={"VIAVEL":0,"MARGINAL":1,"HIPOTETICO":2,"ESTRUTURAL":3,"PEDIR_DESCONTO":4,
             "INFEASIVEL":5,"INDISPONIVEL":6,"NAO_CONV":7}
        df_viu=pd.DataFrame(dados)
        if not df_viu.empty:
            df_viu["_o"]=df_viu["Sta_B"].map(ORD).fillna(9)
            df_viu["_e"]=df_viu["Esp_B"].fillna(-999)
            df_viu=df_viu.sort_values(["_o","_e"],ascending=[True,False]).reset_index(drop=True)

        FN="#,##0"; FU="#,##0.00"; FS='+#,##0.00;-#,##0.00;"-"'
        for ri,row in df_viu.iterrows():
            r=5+ri; ws.row_dimensions[r].height=17
            bg=fi(WH) if ri%2==0 else fi(ZB); bdr=bd()
            sc(r,1,row["Tipo"],bg,fo(False,GY,8),bdr,al("center","center",False))
            sc(r,2,row["MP"],bg,fo(True,"212121",9),bdr,al("left","center",False,1))
            sc(r,3,row["Base"],bg,fo(False,"424242",9),bdr,al("center","center",False),FN)
            dv=row["DispMax"] if row["DispMax"]>0 else "—"
            sc(r,4,dv,bg,fo(False,"424242",9),bdr,al("center","center",False),
               FN if isinstance(dv,(int,float)) else None)
            sc(r,5,row["CIF"],bg,fo(False,"424242",9),bdr,al("center","center",False),FU)
            sc(r,6,"",fi(SEP))
            vb=row["VIU_B"]
            sc(r,7,vb if vb is not None else "—",sta_fi(row["Sta_B"]),
               fo(True,VD2,9) if vb is not None else fo(False,GY,9),
               bdr,al("center","center",False),FU if vb is not None else None)
            eb=row["Esp_B"]
            sc(r,8,eb if eb is not None else "—",sta_fi(row["Sta_B"]),
               esp_cor(eb),bdr,al("center","center",False),FS if eb is not None else None)
            sc(r,9,sta_lbl(row["Sta_B"]),sta_fi(row["Sta_B"]),sta_fo(row["Sta_B"]),
               bdr,al("center","center",False))
            sc(r,10,"",fi(SEP))
            vl=row["VIU_L"]
            sc(r,11,vl if vl is not None else "—",sta_fi(row["Sta_L"]),
               fo(True,VD2,9) if vl is not None else fo(False,GY,9),
               bdr,al("center","center",False),FU if vl is not None else None)
            el=row["Esp_L"]
            sc(r,12,el if el is not None else "—",sta_fi(row["Sta_L"]),
               esp_cor(el),bdr,al("center","center",False),FS if el is not None else None)
            sc(r,13,sta_lbl(row["Sta_L"]),sta_fi(row["Sta_L"]),sta_fo(row["Sta_L"]),
               bdr,al("center","center",False))
            sc(r,14,"",fi(SEP))
            s_lbl,s_fi_,s_fo_=sinal_fmt(row["Sinal"])
            sc(r,15,s_lbl,s_fi_,s_fo_,bdr,al("center","center",True))
            sc(r,16,row["Obs"],bg,fo(False,"37474F",8,True),bdr,al("left","center",True,1))

        R_A=5+len(df_viu)+2; ws.row_dimensions[R_A].height=18
        mg(R_A,1,R_A,NCOL,"  ALERTAS  ·  Ações imediatas identificadas pela análise",
           fi(H1),fo(True,"FFFFFF",10),bd(H1),al("left",ind=1))
        r_curr=R_A+1
        desc2=df_viu[(df_viu["Sta_B"]=="PEDIR_DESCONTO")&(df_viu["Sta_L"]=="PEDIR_DESCONTO")]
        teto=df_viu[df_viu["Sinal"]=="TETO_FISICO"]
        divg=df_viu[df_viu["Sinal"]=="DIVERGEM"]
        estr=df_viu[df_viu["Sinal"]=="ESTRUTURAL"]
        def _al(r,icon,txt,cf,cft):
            ws.row_dimensions[r].height=14
            mg(r,1,r,NCOL,f"  {icon}  {txt}",fi(cf),fo(True,cft,8),bd(SEP),al("left",ind=1))
        if not estr.empty:
            _al(r_curr,"🔷",f"ESTRUTURAIS ({len(estr)}): "+", ".join(estr["MP"].tolist())+
                " — mantêm consumo acima de 500 US$/t. Indispensáveis: garantir fornecimento.",
                RXF,RX); r_curr+=1
        if not desc2.empty:
            _al(r_curr,"🔴",f"DESCONTO URGENTE ({len(desc2)}): "+", ".join(desc2["MP"].tolist())+
                " — inviáveis em ambas as métricas ao CIF atual. Negociar ou substituir.",
                VMF,VM); r_curr+=1
        if not teto.empty:
            _al(r_curr,"🔒",f"TETO FÍSICO ({len(teto)}): "+", ".join(teto["MP"].tolist())+
                " — processo não absorve lote extra nas restrições atuais de qualidade.",
                INFF,INF); r_curr+=1
        if not divg.empty:
            _al(r_curr,"⚠️",f"DIVERGÊNCIA ({len(divg)}): "+", ".join(divg["MP"].tolist())+
                " — viáveis no mix atual, mas próxima composição não se paga. Não expandir.",
                AMF,AM); r_curr+=1
        if desc2.empty and teto.empty and divg.empty and estr.empty:
            _al(r_curr,"✅","Nenhum alerta crítico. Todos os materiais convergem nas duas métricas.",
                VDF,VD2); r_curr+=1

        R_L=r_curr+1; ws.row_dimensions[R_L].height=5; R_L+=1
        ws.row_dimensions[R_L].height=16
        mg(R_L,1,R_L,8,"  LEGENDA DE STATUS",fi(H1),fo(True,"FFFFFF",8),bd(H1),al("left",ind=1))
        for col,lbl,cf,cft in [(2,"VIÁVEL",VDF,VD2),(3,"MARGINAL",AMF,AM),
                                (4,"PEDIR DESC.",VMF,VM),(5,"ESTRUTURAL",RXF,RX),
                                (6,"INFEASÍVEL",INFF,INF),(7,"INDISPONÍVEL",GYF,GY)]:
            sc(R_L,col,lbl,fi(cf),fo(True,cft,8),bd(SEP),al("center","center",False))
        sc(R_L,9,"",fi(SEP))
        sc(R_L,10,"  SINAL",fi(H1),fo(True,"FFFFFF",8),bd(H1),al("left",ind=1))
        for col,lbl,cf,cft in [(11,"✅  CONVERGEM",VDF,VD2),(12,"⚠️  DIVERGEM",AMF,AM),
                                (13,"🔒  TETO FÍSICO",INFF,INF),(14,"🔷  ESTRUTURAL",RXF,RX)]:
            sc(R_L,col,lbl,fi(cf),fo(True,cft,8),bd(SEP),al("center","center",False))
        for cc in (15,16): sc(R_L,cc,"",fi(H1))

        R_E=R_L+2; ws.row_dimensions[R_E-1].height=5
        ws.row_dimensions[R_E].height=14
        ws.row_dimensions[R_E+1].height=28
        ws.row_dimensions[R_E+2].height=22
        mg(R_E,1,R_E,8,"  VIU MARGINAL  ·  Como funciona",
           fi(H2),fo(True,"FFFFFF",9),bd(H2),al("left",ind=1))
        mg(R_E+1,1,R_E+2,8,
           f"Busca binária com teto dinâmico [0 / CIF, max(1000, 5×CIF) US$/t]: encontra o preço exato em que o "
           "solver coloca (Método A) ou retira (Método B) a primeira tonelada voluntariamente.\n"
           f"Teto desta rodada: {_VIU_TETO:.0f} US$/t. Material que não muda nem ao teto → ESTRUTURAL (genuinamente indispensável).",
           fi(BGL),fo(False,"0D3D6E",8),bd(SEP),al("left","top",ind=1))
        sc(R_E,9,"",fi(SEP)); sc(R_E+1,9,"",fi(SEP)); sc(R_E+2,9,"",fi(SEP))
        mg(R_E,10,R_E,NCOL,"  VIU POR LOTE  ·  Como funciona",
           fi(H3),fo(True,"FFFFFF",9),bd(H3),al("left",ind=1))
        mg(R_E+1,10,R_E+2,NCOL,
           "2 runs de solver: compara custo com e sem uma composição ferroviária extra (6.000 t) forçada.\n"
           "Responde: 'Vale comprar a PRÓXIMA composição deste material a este preço?'",
           fi(BGL),fo(False,"1A237E",8),bd(SEP),al("left","top",ind=1))

        ws.freeze_panes=ws.cell(row=5,column=3)
        ws.sheet_view.showGridLines=False
        ws.sheet_properties.tabColor="1565C0"

    # ── Widget Power BI ───────────────────────────────────────────────────────
    def _mostrar_widget_pbi(_out_area):
        with _out_area:
            clear_output()
            print("\n" + "="*52)
            print("  DESEJA GERAR A BASE DE DADOS PARA O POWER BI?")
            _b_sim = widgets.Button(description="SIM", button_style="success")
            _b_nao = widgets.Button(description="NÃO", button_style="danger")
            _pbi_lock = [False]

            def _pbi_sim(b):
                if _pbi_lock[0]: return
                _pbi_lock[0] = True
                b.disabled = True; _b_nao.disabled = True
                with _out_area:
                    clear_output()
                    _tn = widgets.Text(placeholder="Ex: AGO_26", description="Nome Estudo:")
                    _bg = widgets.Button(description="Gerar Arquivo PBI",
                                         button_style="primary", icon="cogs")
                    def _gerar(b2):
                        with _out_area:
                            nome = _tn.value.strip()
                            if not nome:
                                print("⚠️ Digite um nome antes de gerar."); return
                            _bg.disabled=True; _bg.description="A preparar base..."
                            df_pbi=gerar_dados_pbi(nome)
                            _nc=f"PBI_{nome}.csv"
                            _cs=df_pbi.to_csv(index=False,sep=";",decimal=",",encoding="utf-8-sig")
                            import base64 as _b64
                            _b64s=_b64.b64encode(_cs.encode()).decode()
                            clear_output(); print("✅ Dados compilados com sucesso!")
                            display(HTML(f"""
                            <div style="margin-top:15px;">
                              <a href="data:file/csv;base64,{_b64s}" download="{_nc}"
                                 style="display:inline-block;padding:12px 24px;
                                        background-color:#28a745;color:white;
                                        text-decoration:none;border-radius:5px;
                                        font-weight:bold;font-family:sans-serif;
                                        box-shadow:0px 4px 6px rgba(0,0,0,0.1);">
                                 📥 CLIQUE AQUI PARA BAIXAR: {_nc}
                              </a>
                            </div>
                            <p style="margin-top:10px;font-size:12px;color:#666;">
                              <em>(clique uma vez para iniciar o download)</em>
                            </p>"""))
                    _bg.on_click(_gerar)
                    display(widgets.VBox([
                        widgets.Label("1. Digite o nome do cenário e clique em Gerar:"),
                        _tn, _bg]))

            def _pbi_nao(b):
                if _pbi_lock[0]: return
                _pbi_lock[0] = True
                b.disabled=True; _b_sim.disabled=True
                with _out_area:
                    clear_output()
                    print("✅ Processo finalizado. O Excel principal já foi baixado.")

            _b_sim.on_click(_pbi_sim); _b_nao.on_click(_pbi_nao)
            display(widgets.HBox([_b_sim, _b_nao]))
            print("="*52)

    # ── WIDGET PRINCIPAL ─────────────────────────────────────────────────────
    _out_viu = widgets.Output()

    display(HTML("""
    <div style="margin:16px 0;padding:20px 24px;background:#EBF4FC;
                border-left:5px solid #1565C0;border-radius:6px;
                font-family:'Calibri',sans-serif;max-width:700px;">
      <div style="display:flex;align-items:center;gap:10px;margin-bottom:10px;">
        <span style="font-size:22px;">📊</span>
        <span style="font-size:16px;font-weight:bold;color:#0D3D6E;">
          Análise de VIU — <em>Value In Use</em>
        </span>
      </div>
      <p style="margin:0 0 10px;color:#1A237E;font-size:13px;line-height:1.6;">
        O modelo pode calcular o <b>VIU (Value In Use)</b> para todos os minérios,
        NPOs e pelotas, gerando um relatório executivo comparativo na aba
        <b>VIU_ANALISE</b> do Excel.
      </p>
      <div style="display:flex;gap:12px;margin:12px 0;">
        <div style="flex:1;background:#FFFFFF;border:1px solid #BBDEFB;
                    border-radius:4px;padding:10px 14px;">
          <div style="font-size:11px;font-weight:bold;color:#1565C0;margin-bottom:5px;">
            VIU MARGINAL</div>
          <div style="font-size:11px;color:#37474F;line-height:1.5;">
            Busca binária [0–500 US$/t] · Ponto exato de indiferença do solver<br>
            <i style="color:#546E7A;">"A que preço o solver coloca ou retira a primeira tonelada?"</i>
          </div>
        </div>
        <div style="flex:1;background:#FFFFFF;border:1px solid #BBDEFB;
                    border-radius:4px;padding:10px 14px;">
          <div style="font-size:11px;font-weight:bold;color:#1976D2;margin-bottom:5px;">
            VIU POR LOTE</div>
          <div style="font-size:11px;color:#37474F;line-height:1.5;">
            2 runs de solver · Break-even por composição ferroviária (6.000 t)<br>
            <i style="color:#546E7A;">"Vale comprar a próxima composição?"</i>
          </div>
        </div>
      </div>
      <p style="margin:8px 0 0;color:#546E7A;font-size:11px;">
        ⏱️ Tempo estimado: <b>~1 minuto</b> &nbsp;·&nbsp;
        📄 Resultado: aba <b>VIU_ANALISE</b> no Excel gerado
      </p>
    </div>
    """))

    _btn_sim = widgets.Button(description=" ✅  Calcular VIU", button_style="success",
                              layout=widgets.Layout(width="190px", height="40px"))
    _btn_nao = widgets.Button(description=" ⏭️  Pular", button_style="warning",
                              layout=widgets.Layout(width="120px", height="40px"))

    def _on_sim(b):
        if _viu_lock[0]: return
        _viu_lock[0] = True
        _btn_sim.disabled = True; _btn_nao.disabled = True
        # ── Processamento dentro do widget output ────────────────────────────
        with _out_viu:
            clear_output()
            print("⏳ Calculando VIU Marginal + VIU por Lote para todos os materiais...")
            print("   (este processo leva aproximadamente 1 minuto)\n")
            _dados = _engine_viu_duplo()
            print("✅ Cálculo concluído. Gerando relatório executivo no Excel...")
            _escrever_relatorio_viu(wb, _dados)
            _fname = "RESULTADO_OTIMIZACAO.xlsx"
            wb.save(_fname)
            print(f"✅ Excel com VIU_ANALISE pronto.\n")
        # ── Download FORA do widget output: evita re-execução pelo clear_output ──
        from google.colab import files as _gf
        _gf.download(_fname)
        _mostrar_widget_pbi(_out_viu)

    def _on_nao(b):
        if _viu_lock[0]: return
        _viu_lock[0] = True
        _btn_sim.disabled = True; _btn_nao.disabled = True
        # ── Salva ANTES do with, download FORA do widget output ──────────────
        _fname = "RESULTADO_OTIMIZACAO.xlsx"
        wb.save(_fname)
        from google.colab import files as _gf
        _gf.download(_fname)
        with _out_viu:
            clear_output()
            print("✅ Excel gerado e baixado sem análise de VIU.\n")
        _mostrar_widget_pbi(_out_viu)

    _btn_sim.on_click(_on_sim)
    _btn_nao.on_click(_on_nao)
    display(widgets.HBox([_btn_sim, _btn_nao]), _out_viu)



# ======================================================
# 8. EXPORTAÇÃO POWER BI
# ======================================================
def gerar_dados_pbi(nome_personalizado):
    """Agrupa todos os indicadores (Massa, Físicos, Químicos, Estoque, Custos) e retorna o DataFrame PBI."""
    pbi_data = []

    # Custo unitário por máquina para o PBI (consistente com KPIS_AF)
    custo_unit_ms = {}
    for ms in MAQUINAS:
        invest_total = sum(
            (sum(x_ms[mp][ms][af].value() or 0 for af in AFS) + (y_ms_patio[mp][ms].value() or 0)) * Preco.get(mp, 0)
            for mp in mps if clean_name(mp) != clean_name("Finos de NPO")
        )
        # Finos NPO alocados a esta MS (consistente com Seção C e Fix3)
        custo_finos_npo_ms = sum(
            sum((x_dir[npo][af].value() or 0) for af in AFS) * Finos.get(npo, 0) * Preco.get(npo, 0)
            * ((x_fino_npo[npo][ms].value() or 0) /
               (sum(x_fino_npo[npo][ms2].value() or 0 for ms2 in MAQUINAS) + 1e-10))
            for npo in npos_list
        )
        _cu_bruto = ((invest_total + custo_finos_npo_ms) / PROD_FIXA_MS[ms] if PROD_FIXA_MS[ms] > 0 else 0) + CUSTO_OP_VAR_MS[ms]
        # Crédito do sínter fino como subproduto: divide pelo REND para alocar
        # o custo total apenas ao sínter útil que entra no Alto Forno.
        custo_unit_ms[ms] = _cu_bruto / REND_SINTER[ms] if REND_SINTER[ms] > 0 else _cu_bruto

    # --- 0. BALANÇO DE ESTOQUE (PÁTIO GLOBAL) ---
    pbi_data.append([nome_personalizado, "GLOBAL", "ESTOQUE", "Pátio de Sínter", "Estoque Inicial", "Toneladas (t)", ESTOQUE_INI])
    pbi_data.append([nome_personalizado, "GLOBAL", "ESTOQUE", "Pátio de Sínter", "Estoque Final", "Toneladas (t)", ESTOQUE_FIN])
    pbi_data.append([nome_personalizado, "GLOBAL", "ESTOQUE", "Pátio de Sínter", "Delta Estoque", "Toneladas (t)", DELTA_ESTOQUE])

    # --- 1. KPIs DOS ALTOS FORNOS (AF2 e AF3) ---
    for af in AFS:
        v_gusa = gusa[af].value() or 1

        # Produção Absoluta
        pbi_data.append([nome_personalizado, af, "PRODUÇÃO", "Operacional AF", "Produção Gusa", "Toneladas (t)", v_gusa])

        # Massas base
        m_sinter_dir = sum(
            (x_ms[mp][ms][af].value() or 0)
            * (1 - PERDA_FISICA[ms] if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
            * (1-Umid[mp])
            * (1 - (PPC_FINOS_EFETIVO if clean_name(mp) == clean_name("Finos de NPO") else PPC[mp]))
            * REND_SINTER[ms]
            for mp in mps for ms in MAQUINAS if Aplicacao.get(mp) == "MS"
        )
        m_sinter_ret = (z_patio_af[af].value() or 0) * (1-Umid["PATIO RETOMADO"]) * (1-PPC["PATIO RETOMADO"]) * rend_medio
        m_sinter_total = m_sinter_dir + m_sinter_ret

        # --- A. CARGA METÁLICA E ENERGIA FÍSICA (Base para o Gráfico de Mix - Sem Duplicidade) ---
        # Sínter (Como não tem quebra de fornecedor, enviamos o Total físico uma única vez)
        pbi_data.append([nome_personalizado, af, "SINTER", "Carga AF", "Sinter Produzido", "Consumo (kg/t gusa)", (m_sinter_total/v_gusa)*1000])
        # Quartzo
        pbi_data.append([nome_personalizado, af, "FUNDENTE", "Carga AF", "Quartzo", "Consumo (kg/t gusa)", ((x_quartzo[af].value() or 0)*1000)/v_gusa])

        # Nobres e Combustíveis DETALHADOS (Por fornecedor, para não duplicar massas)
        for mp in mps:
            tipo_atual = tipo_mp.get(mp, "OUTROS")
            if tipo_atual in ["NPO", "PELOTA", "BRIQUETE"]:
                m_dir_mp = (x_dir[mp][af].value() or 0) * (1-Finos[mp]) * (1-Umid[mp])
                if m_dir_mp > 0:
                    pbi_data.append([nome_personalizado, af, tipo_atual, "Carga AF", mp, "Consumo (kg/t gusa)", (m_dir_mp/v_gusa)*1000])
            elif "CARBONOSO" in tipo_atual:
                m_ener_mp = (x_dir[mp][af].value() or 0) * (1-Umid[mp])
                if m_ener_mp > 0:
                    pbi_data.append([nome_personalizado, af, "ENERGÉTICO", "Energia AF", mp, "Consumo (kg/t gusa)", (m_ener_mp/v_gusa)*1000])

        # --- B. MACROS DE OPERAÇÃO (Isolados na Dimensão 'KPI') ---
        m_met_af = m_sinter_total + sum((x_dir[mp][af].value() or 0) * (1-Finos[mp]) * (1-Umid[mp]) for mp in mps if tipo_mp.get(mp) in ["NPO", "PELOTA", "BRIQUETE"])
        c_coque_s = sum((x_dir[mp][af].value() or 0) * (1 - Umid[mp]) for mp in mps if tipo_mp.get(mp) == "CARBONOSO_COQUE")
        c_pci_s   = sum((x_dir[mp][af].value() or 0) * (1 - Umid[mp]) for mp in mps if tipo_mp.get(mp) == "CARBONOSO_PCI")

        pbi_data.append([nome_personalizado, af, "KPI", "KPIs Operacionais", "PAM Metálica Total", "kg/t gusa", (m_met_af*1000)/v_gusa])
        pbi_data.append([nome_personalizado, af, "KPI", "KPIs Operacionais", "Coque Rate Total", "kg/t gusa", (c_coque_s * 1000) / v_gusa])
        pbi_data.append([nome_personalizado, af, "KPI", "KPIs Operacionais", "PCI Rate Total", "kg/t gusa", (c_pci_s * 1000) / v_gusa])
        pbi_data.append([nome_personalizado, af, "KPI", "KPIs Operacionais", "Fuel Rate Total", "kg/t gusa", ((c_coque_s + c_pci_s) * 1000) / v_gusa])

        # --- C. Balanço Químico (Gusa e Escória) ---
        def get_massa_input(elem):
            m_dir = sum((x_dir[mp][af].value() or 0) * (1-Finos[mp]) * (1-Umid[mp]) * (quimica[elem][mp]/100) for mp in mps)
            m_via_ms = sum(
                (x_ms[mp][ms][af].value() or 0)
                * (1 - PERDA_FISICA[ms] if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
                * (1-Umid[mp]) * (quimica[elem][mp]/100) * REND_SINTER[ms]
                for mp in mps for ms in MAQUINAS
                if clean_name(mp) != clean_name("Finos de NPO")
            )
            # Sínter Dinâmico (finos NPO com penalidade de ganga — consistente com KPIS_AF)
            m_din_pbi = sum(
                (x_fino_npo[mp][ms].value() or 0.0) * ((quimica[elem][mp] + PENAL_BLEND[elem]) / 100)
                * REND_SINTER[ms] * FATOR_ROTA[ms][af]
                for mp in npos_list for ms in MAQUINAS
            )
            m_via_patio = z_patio_af[af].value() * (1-Umid["PATIO RETOMADO"]) * (quimica[elem].get("PATIO RETOMADO", 0)/100) * rend_medio
            return m_dir + m_via_ms + m_din_pbi + m_via_patio

        mn_g = get_massa_input("MN") * ALFA[af]["MN"]
        p_g  = get_massa_input("P") * ALFA[af]["P"]
        sio2_total = get_massa_input("SIO2") + (x_quartzo[af].value() or 0)
        si_g = sio2_total * ALFA[af]["SI"] * (28.08 / 60.08)

        pbi_data.append([nome_personalizado, af, "QUALIDADE", "Qualidade Gusa", "Fe", "%", 94.50])
        pbi_data.append([nome_personalizado, af, "QUALIDADE", "Qualidade Gusa", "Si", "%", (si_g / v_gusa) * 100])
        pbi_data.append([nome_personalizado, af, "QUALIDADE", "Qualidade Gusa", "Mn", "%", (mn_g / v_gusa) * 100])
        pbi_data.append([nome_personalizado, af, "QUALIDADE", "Qualidade Gusa", "P", "%", (p_g / v_gusa) * 100])

        cao_e = get_massa_input("CAO")
        mgo_e = get_massa_input("MGO")
        al2o3_e = get_massa_input("AL2O3")
        sio2_e = sio2_total * (1 - ALFA[af]["SI"])

        # Massa corrigida para o PBI
        m_slag_total = (cao_e + mgo_e + al2o3_e + sio2_e) / FATOR_OXIDOS[af]

        if m_slag_total > 0:
            pbi_data.append([nome_personalizado, af, "QUALIDADE", "Qualidade Escória", "Slag Rate", "kg/t gusa", (m_slag_total * 1000) / v_gusa])
            pbi_data.append([nome_personalizado, af, "QUALIDADE", "Qualidade Escória", "CaO", "%", (cao_e / m_slag_total) * 100])
            pbi_data.append([nome_personalizado, af, "QUALIDADE", "Qualidade Escória", "SiO2", "%", (sio2_e / m_slag_total) * 100])
            pbi_data.append([nome_personalizado, af, "QUALIDADE", "Qualidade Escória", "MgO", "%", (mgo_e / m_slag_total) * 100])
            pbi_data.append([nome_personalizado, af, "QUALIDADE", "Qualidade Escória", "Al2O3", "%", (al2o3_e / m_slag_total) * 100])
            pbi_data.append([nome_personalizado, af, "QUALIDADE", "Qualidade Escória", "B2", "Índice", cao_e / sio2_e if sio2_e > 0 else 0])
            pbi_data.append([nome_personalizado, af, "QUALIDADE", "Qualidade Escória", "B3", "Índice", (cao_e + mgo_e) / sio2_e if sio2_e > 0 else 0])

        # --- D. Custo Gusa ---
        # Custo direto PBI: NPOs só porção útil; finos em custo_unit_ms
        custo_dir = (
            sum((x_dir[mp][af].value() or 0) * (1 - Finos.get(mp, 0)) * Preco.get(mp, 0)
                for mp in mps if tipo_mp.get(mp) == "NPO")
            + sum((x_dir[mp][af].value() or 0) * Preco.get(mp, 0)
                  for mp in mps if tipo_mp.get(mp) != "NPO")
        )
        # CIF do Sínter Fino — lido do mesmo dicionário de preços
        _cif_fino_pbi = preco_limpo.get(
            next((mp for mp in mps if "SINTER FINO" in mp.upper()), ""), 0
        )
        # custo_sinter_af para o PBI: mesmo cálculo da Seção C
        _custo_sinter_af_pbi = {
            ms: (custo_unit_ms[ms] / REND_SINTER[ms]
                 - (1 - REND_SINTER[ms]) / REND_SINTER[ms] * _cif_fino_pbi)
            if REND_SINTER[ms] > 0 else custo_unit_ms[ms]
            for ms in MAQUINAS
        }

        custo_sin_direto = sum(
            sum((x_ms[mp][ms][af].value() or 0.0)
                * (1 - PERDA_FISICA[ms] if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
                * (1-Umid[mp])
                * (1 - (PPC_FINOS_EFETIVO if clean_name(mp) == clean_name("Finos de NPO") else PPC[mp]))
                * REND_SINTER[ms]
                * _custo_sinter_af_pbi[ms]   # ← usa custo com crédito do fino
                for mp in mps)
            for ms in MAQUINAS
        )
        custo_unit_patio = (
            sum(_custo_sinter_af_pbi[ms] * PROD_FIXA_MS[ms] for ms in MAQUINAS)
            / sum(PROD_FIXA_MS.values())
        ) if sum(PROD_FIXA_MS.values()) > 0 else 0
        m_sinter_patio_af = (z_patio_af[af].value() or 0) * (1-Umid["PATIO RETOMADO"]) * (1-PPC["PATIO RETOMADO"]) * rend_medio
        custo_sin_patio = m_sinter_patio_af * custo_unit_patio
        custo_qz = (x_quartzo[af].value() or 0) * 40.0
        custo_total_af = custo_dir + custo_sin_direto + custo_sin_patio + custo_qz + CUSTO_OP_VAR_AF[af] * v_gusa

        pbi_data.append([nome_personalizado, af, "CUSTO", "Financeiro", "Custo Metálico AF", "US$/t gusa", custo_total_af / v_gusa])

    # --- 2. KPIs DAS SINTERIZAÇÕES (MS1E2 e MS3) ---
    for ms in MAQUINAS:
        prod_bsr = PROD_FIXA_MS[ms]
        m_sf = 0; m_rec = 0; m_fun = 0; m_mp = 0; m_ener = 0

        # Produção Absoluta
        pbi_data.append([nome_personalizado, ms, "PRODUÇÃO", "Operacional MS", "Produção BSR", "Toneladas (t)", prod_bsr])
        pbi_data.append([nome_personalizado, ms, "PRODUÇÃO", "Operacional MS", "Sínter Útil", "Toneladas (t)", prod_bsr * REND_SINTER[ms]])
        pbi_data.append([nome_personalizado, ms, "PRODUÇÃO", "Operacional MS", "Geração Fino", "Toneladas (t)", prod_bsr * (1 - REND_SINTER[ms])])

        # Custo Sínter
        pbi_data.append([nome_personalizado, ms, "CUSTO", "Financeiro", "Custo Produção", "US$/t BSR", custo_unit_ms[ms]])

        # --- A. Consumos Físicos da Sinterização (Sem Duplicidades) ---
        for mp in mps:
            cons_bu = (sum((x_ms[mp][ms][af].value() or 0) for af in AFS) + (y_ms_patio[mp][ms].value() or 0))
            if cons_bu > 0:
                tipo_atual = tipo_mp.get(mp, "OUTROS")
                pbi_data.append([nome_personalizado, ms, tipo_atual, "PAM Sinterização", mp, "Consumo (kg/t BSR)", (cons_bu*1000)/prod_bsr])

                if tipo_atual == "MINERIO": m_sf += cons_bu
                if tipo_atual == "RECICLADO": m_rec += cons_bu
                if tipo_atual == "FUNDENTE": m_fun += cons_bu

        # --- B. MACROS DE OPERAÇÃO MS (Isolados na Dimensão 'KPI') ---
        pbi_data.append([nome_personalizado, ms, "KPI", "KPIs Operacionais", "Total Reciclados", "kg/t BSR", (m_rec*1000)/prod_bsr])
        pbi_data.append([nome_personalizado, ms, "KPI", "KPIs Operacionais", "Total Fundentes", "kg/t BSR", (m_fun*1000)/prod_bsr])
        pbi_data.append([nome_personalizado, ms, "KPI", "KPIs Operacionais", "Total Sinter Feed", "kg/t BSR", (m_sf*1000)/prod_bsr])

        # --- C. Qualidade Química do Sínter (Blindado contra Finos Artificiais) ---
        m_bsr_quim = sum((sum((x_ms[m][ms][af].value() or 0) for af in AFS) + (y_ms_patio[m][ms].value() or 0))
                         * (1 - Umid[m]) * (1 - PPC[m]) for m in mps)

        if m_bsr_quim > 0:
            # Função interna para garantir que o PBI leia a química dinâmica
            def calc_ox_pbi(ox):
                m_est = sum((sum((x_ms[m][ms][af].value() or 0) for af in AFS) + (y_ms_patio[m][ms].value() or 0))
                            * (1-Umid[m]) * (quimica[ox][m]/100)
                            for m in mps if clean_name(m) != clean_name("Finos de NPO"))
                m_din = sum((x_fino_npo[m][ms].value() or 0) * (quimica[ox][m]/100) for m in npos_list)
                return m_est + m_din

            for elem in ["FET", "SIO2", "AL2O3", "CAO", "MGO", "MN", "P"]:
                m_elem = calc_ox_pbi(elem)
                pbi_data.append([nome_personalizado, ms, "QUALIDADE", "Qualidade Sínter", elem, "%", (m_elem / m_bsr_quim) * 100])

            m_cao_s = calc_ox_pbi("CAO")
            m_sio2_s = calc_ox_pbi("SIO2")
            pbi_data.append([nome_personalizado, ms, "QUALIDADE", "Qualidade Sínter", "B2", "Índice", m_cao_s / m_sio2_s if m_sio2_s > 0 else 0])

        # --- D. Granulometria e Umidade (Mistura Parcial) ---
        m_mpa_bu = sum((sum((x_ms[m][ms][af].value() or 0) for af in AFS) + (y_ms_patio[m][ms].value() or 0)) for m in mps if tipo_mp.get(m) != "CARGA_ENERGETICA")
        m_mpa_bs = sum((sum((x_ms[m][ms][af].value() or 0) for af in AFS) + (y_ms_patio[m][ms].value() or 0)) * (1 - Umid[m]) for m in mps if tipo_mp.get(m) != "CARGA_ENERGETICA")

        if m_mpa_bu > 0:
            pbi_data.append([nome_personalizado, ms, "QUALIDADE", "Qualidade Sínter", "Umidade", "%", ((m_mpa_bu - m_mpa_bs) / m_mpa_bu) * 100])

            m_gt100 = sum((sum((x_ms[m][ms][af].value() or 0) for af in AFS) + (y_ms_patio[m][ms].value() or 0)) * G_100[m] for m in mps if tipo_mp.get(m) != "CARGA_ENERGETICA")
            m_gt635 = sum((sum((x_ms[m][ms][af].value() or 0) for af in AFS) + (y_ms_patio[m][ms].value() or 0)) * G_635[m] for m in mps if tipo_mp.get(m) != "CARGA_ENERGETICA")
            m_lt105 = sum((sum((x_ms[m][ms][af].value() or 0) for af in AFS) + (y_ms_patio[m][ms].value() or 0)) * G_105[m] for m in mps if tipo_mp.get(m) != "CARGA_ENERGETICA")

            pbi_data.append([nome_personalizado, ms, "QUALIDADE", "Granulometria MS", "> 1.0mm", "%", (m_gt100 / m_mpa_bu) * 100])
            pbi_data.append([nome_personalizado, ms, "QUALIDADE", "Granulometria MS", "> 6.35mm", "%", (m_gt635 / m_mpa_bu) * 100])
            pbi_data.append([nome_personalizado, ms, "QUALIDADE", "Granulometria MS", "< 0.105mm", "%", (m_lt105 / m_mpa_bu) * 100])

            m_fn = m_gt100 - m_gt635
            m_fa = m_lt105
            rn_valor = m_fn / (m_fn + m_fa) if (m_fn + m_fa) > 0 else 0

            pbi_data.append([nome_personalizado, ms, "QUALIDADE", "Granulometria MS", "RN", "Índice", rn_valor])
            pbi_data.append([nome_personalizado, ms, "QUALIDADE", "Granulometria MS", "RA", "Índice", 1 - rn_valor if (m_fn + m_fa) > 0 else 0])

    return pd.DataFrame(pbi_data, columns=["Cenario", "Unidade", "Dimensao", "Categoria", "Item", "Indicador", "Valor"])



    # -*- coding: utf-8 -*-
# ======================================================
# MÓDULO 5.5 — HOMOLOGAÇÃO DO PÁTIO DE SÍNTER
# ======================================================
# Onde colar: logo após o bloco de validação da FO existente
# (após o print("─"*50 + "\n")) e ANTES do comentário
# "# 6. MÓDULO DE ESCRITA REVERSA".
#
# O que este módulo faz:
#   Executa 5 verificações independentes sobre o cenário
#   de Delta Pátio, cobrindo balanço físico, travas lógicas,
#   consistência química, calibração de preço e capacidade
#   do pátio. Cada verificação retorna ✅ PASSOU ou ❌ FALHOU,
#   com o valor encontrado e o valor esperado lado a lado.
#
# Funciona para os três cenários:
#   DELTA > 0  → Estocagem (tema desta homologação)
#   DELTA < 0  → Retomada
#   DELTA = 0  → Neutro
# ======================================================

if pl.LpStatus[status] == 'Optimal':

    # ── Cabeçalho ─────────────────────────────────────────────────────────
    SEP  = "═" * 68
    SEP2 = "─" * 68
    print(f"\n{SEP}")
    print(f"  🏗️  HOMOLOGAÇÃO DO PÁTIO DE SÍNTER")

    # Identifica o cenário ativamente
    if DELTA_ESTOQUE > 0:
        _cen = f"ESTOCAGEM  (+{DELTA_ESTOQUE:,.0f} t BSR desviados para a pilha)"
    elif DELTA_ESTOQUE < 0:
        _cen = f"RETOMADA   ({DELTA_ESTOQUE:,.0f} t BSR retomados da pilha)"
    else:
        _cen = "NEUTRO     (sem movimentação do pátio)"

    print(f"  Cenário  : {_cen}")
    print(f"  Estoque  : {ESTOQUE_INI:,.0f} t → {ESTOQUE_FIN:,.0f} t  "
          f"(Cap. máx.: {ESTOQUE_MAX_PATIO:,.0f} t BSR)")
    print(SEP)

    # ── Tolerâncias ────────────────────────────────────────────────────────
    # Todas as verificações de massa usam 1 t como tolerância absoluta.
    # Verificações de qualidade (óxidos) usam 0.001 pp (< 0.1 décimo de pp).
    # Verificação de preço usa 0.01 US$/t.
    _TOL_MASSA  = 1.0      # tonelada
    _TOL_OXIDO  = 0.001    # ponto percentual
    _TOL_PRECO  = 0.01     # US$/t

    # Contador geral de resultados
    _passou = 0
    _falhou = 0

    def _resultado(nome, passou, encontrado, esperado, unidade, detalhe=""):
        """Imprime uma linha de resultado formatada e atualiza os contadores."""
        global _passou, _falhou
        icone = "✅ PASSOU" if passou else "❌ FALHOU"
        if passou:
            _passou += 1
        else:
            _falhou += 1
        print(f"\n  {icone}  [{nome}]")
        print(f"    Encontrado : {encontrado}  {unidade}")
        print(f"    Esperado   : {esperado}  {unidade}")
        if detalhe:
            print(f"    Detalhe    : {detalhe}")

    # ══════════════════════════════════════════════════════════════════════
    # VERIFICAÇÃO 1 — BALANÇO FÍSICO DO PÁTIO
    # Garante que a massa desviada para a pilha (y_ms_patio) ou retomada
    # (z_patio_af) corresponde exatamente ao DELTA_ESTOQUE declarado.
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{SEP2}")
    print(f"  VERIFICAÇÃO 1 — Balanço Físico da Pilha (BSR)")
    print(SEP2)

    # Recalcula a massa que foi para o pátio em BSR (mesma fórmula da restrição)
    _massa_entrou_bsr = sum(
        y_ms_patio[mp][ms].value() or 0
        for ms in MAQUINAS for mp in mps
        if Aplicacao.get(mp) == "MS"
        and tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO")
    )
    # Aplica a mesma transformação B.U. → B.S. → BSR usada na restrição algébrica
    _massa_entrou_bsr_corr = sum(
        (y_ms_patio[mp][ms].value() or 0)
        * (1 - PERDA_FISICA[ms]) * (1 - Umid[mp]) * (1 - PPC[mp])
        for ms in MAQUINAS for mp in mps
        if Aplicacao.get(mp) == "MS"
        and tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO")
    ) + sum(
        (y_ms_patio[mp][ms].value() or 0)
        * (1 - Umid[mp]) * (1 - PPC[mp])
        for ms in MAQUINAS for mp in mps
        if Aplicacao.get(mp) == "MS"
        and tipo_mp.get(mp) in ("CARGA_ENERGETICA", "RECICLADO")
    )

    # Massa retomada do pátio em BSR
    _massa_saiu_bsr = sum(
        (z_patio_af[af].value() or 0) * (1 - Umid["PATIO RETOMADO"]) * (1 - PPC["PATIO RETOMADO"])
        for af in AFS
    )

    _delta_calc = _massa_entrou_bsr_corr - _massa_saiu_bsr
    _erro_v1    = abs(_delta_calc - DELTA_ESTOQUE)
    _passou_v1  = _erro_v1 <= _TOL_MASSA

    _resultado(
        "BALANÇO FÍSICO — Entrada - Saída = Delta",
        _passou_v1,
        f"{_delta_calc:,.1f}",
        f"{DELTA_ESTOQUE:,.1f}",
        "t BSR",
        f"Erro absoluto = {_erro_v1:.2f} t  (tolerância = {_TOL_MASSA:.0f} t)"
    )

    # Detalha as contribuições por máquina para facilitar diagnóstico
    print()
    for ms in MAQUINAS:
        _v_ms_patio = sum(
            (y_ms_patio[mp][ms].value() or 0)
            * (1 - PERDA_FISICA[ms]) * (1 - Umid[mp]) * (1 - PPC[mp])
            for mp in mps
            if Aplicacao.get(mp) == "MS"
            and tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO")
        ) + sum(
            (y_ms_patio[mp][ms].value() or 0)
            * (1 - Umid[mp]) * (1 - PPC[mp])
            for mp in mps
            if Aplicacao.get(mp) == "MS"
            and tipo_mp.get(mp) in ("CARGA_ENERGETICA", "RECICLADO")
        )
        print(f"    {ms}: {_v_ms_patio:>12,.1f} t BSR desviados para a pilha")

    for af in AFS:
        _v_ret = (z_patio_af[af].value() or 0) * (1 - Umid["PATIO RETOMADO"]) * (1 - PPC["PATIO RETOMADO"])
        print(f"    Retomada → {af}: {_v_ret:>10,.1f} t BSR")

    # ══════════════════════════════════════════════════════════════════════
    # VERIFICAÇÃO 2 — TRAVAS LÓGICAS (NÃO EMPILHAR E RETOMAR AO MESMO TEMPO)
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{SEP2}")
    print(f"  VERIFICAÇÃO 2 — Travas Lógicas de Operação")
    print(SEP2)

    if DELTA_ESTOQUE > 0:
        # Cenário de estocagem: z_patio_af deve ser zero para todos os AFs
        _z_total = sum(z_patio_af[af].value() or 0 for af in AFS)
        _passou_v2 = _z_total <= _TOL_MASSA
        _resultado(
            "ESTOCAGEM — Retomada travada (z_patio_af = 0)",
            _passou_v2,
            f"{_z_total:,.2f}",
            "0",
            "t B.U. retomados",
            "Em estocagem, não deve haver retomada simultânea."
        )
        # Detalha por AF
        for af in AFS:
            _z = z_patio_af[af].value() or 0
            _ok = "✓" if _z <= _TOL_MASSA else "✗"
            print(f"    {_ok} z_patio_af[{af}] = {_z:,.4f} t B.U.")

    elif DELTA_ESTOQUE < 0:
        # Cenário de retomada: y_ms_patio deve ser zero para todas as MPs e MSs
        _y_total = sum(
            y_ms_patio[mp][ms].value() or 0
            for mp in mps for ms in MAQUINAS
        )
        _passou_v2 = _y_total <= _TOL_MASSA
        _resultado(
            "RETOMADA — Envio ao pátio travado (y_ms_patio = 0)",
            _passou_v2,
            f"{_y_total:,.2f}",
            "0",
            "t B.U. enviados ao pátio",
            "Em retomada, não deve haver estocagem simultânea."
        )

    else:
        # Cenário neutro: ambas as variáveis devem ser zero
        _z_total = sum(z_patio_af[af].value() or 0 for af in AFS)
        _y_total = sum(
            y_ms_patio[mp][ms].value() or 0
            for mp in mps for ms in MAQUINAS
        )
        _passou_v2 = (_z_total <= _TOL_MASSA) and (_y_total <= _TOL_MASSA)
        _resultado(
            "NEUTRO — Estocagem e retomada travadas",
            _passou_v2,
            f"z_total={_z_total:,.2f}  y_total={_y_total:,.2f}",
            "Ambos = 0",
            "t B.U.",
            "Delta zero: nenhuma movimentação deve ocorrer."
        )

    # ══════════════════════════════════════════════════════════════════════
    # VERIFICAÇÃO 3 — CONSISTÊNCIA QUÍMICA DO SÍNTER ESTOCADO
    # A pilha deve ter a mesma composição química do sínter produzido.
    # Compara a média ponderada do que foi ao pátio com a do que foi ao AF.
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{SEP2}")
    print(f"  VERIFICAÇÃO 3 — Consistência Química da Pilha")
    print(SEP2)

    if DELTA_ESTOQUE > 0:

        _OXIDOS_CHK = ["FET", "SIO2", "CAO", "MGO", "AL2O3", "MN", "P"]
        _quim_ok = True

        # Base seca total que foi ao pátio (para calcular média ponderada de química)
        _bsr_patio_total = _massa_entrou_bsr_corr

        # Base seca total que foi direto ao AF via MS (para comparar)
        _bsr_af_total = sum(
            (x_ms[mp][ms][af].value() or 0)
            * (1 - PERDA_FISICA[ms] if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
            * (1 - Umid[mp]) * (1 - PPC[mp])
            for mp in mps for ms in MAQUINAS for af in AFS
            if Aplicacao.get(mp) == "MS"
        )

        print(f"\n    {'Óxido':<8}  {'Pátio (%)':>12}  {'AF direto (%)':>14}  {'Δ (pp)':>10}  {'Status':>8}")
        print(f"    {'─'*8}  {'─'*12}  {'─'*14}  {'─'*10}  {'─'*8}")

        for ox in _OXIDOS_CHK:

            # Química do que foi ao pátio (média ponderada em base seca)
            if _bsr_patio_total > 0:
                _m_ox_patio = sum(
                    (y_ms_patio[mp][ms].value() or 0)
                    * (1 - PERDA_FISICA[ms] if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
                    * (1 - Umid[mp]) * (quimica[ox][mp] / 100)
                    for ms in MAQUINAS for mp in mps
                    if Aplicacao.get(mp) == "MS"
                    and clean_name(mp) != clean_name("Finos de NPO")
                ) + sum(
                    (x_fino_npo[mp][ms].value() or 0)
                    * ((quimica[ox][mp] + PENAL_BLEND[ox]) / 100)
                    * (vol_ms1_patio / prod_ms1 if ms == "MS1E2" and prod_ms1 > 0 else
                       vol_ms3_patio / prod_ms3 if ms == "MS3" and prod_ms3 > 0 else 0)
                    for mp in npos_list for ms in MAQUINAS
                )
                _teor_patio = (_m_ox_patio / _bsr_patio_total) * 100
            else:
                _teor_patio = 0.0

            # Química do que foi ao AF direto via MS (média ponderada em base seca)
            if _bsr_af_total > 0:
                _m_ox_af = sum(
                    (x_ms[mp][ms][af].value() or 0)
                    * (1 - PERDA_FISICA[ms] if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO") else 1.0)
                    * (1 - Umid[mp]) * (quimica[ox][mp] / 100)
                    for mp in mps for ms in MAQUINAS for af in AFS
                    if Aplicacao.get(mp) == "MS"
                    and clean_name(mp) != clean_name("Finos de NPO")
                ) + sum(
                    (x_fino_npo[mp][ms].value() or 0)
                    * ((quimica[ox][mp] + PENAL_BLEND[ox]) / 100)
                    * FATOR_ROTA[ms][af]
                    for mp in npos_list for ms in MAQUINAS for af in AFS
                )
                _teor_af = (_m_ox_af / _bsr_af_total) * 100
            else:
                _teor_af = 0.0

            _delta_ox = abs(_teor_patio - _teor_af)
            _ox_ok    = _delta_ox <= _TOL_OXIDO
            if not _ox_ok:
                _quim_ok = False

            _st = "✓ ok" if _ox_ok else "✗ ERRO"
            print(f"    {ox:<8}  {_teor_patio:>12.4f}  {_teor_af:>14.4f}  "
                  f"{(_teor_patio - _teor_af):>+10.4f}  {_st:>8}")

        print()
        _resultado(
            "QUÍMICA — Pátio idêntico ao sínter dos AFs",
            _quim_ok,
            "Ver tabela acima",
            f"Δ ≤ {_TOL_OXIDO} pp para todos os óxidos",
            "",
            "Divergência indica quebra da regra de homogeneização."
        )

    else:
        print(f"\n    ⏭️  Verificação química do pátio não aplicável "
              f"(DELTA = {DELTA_ESTOQUE:,.0f} t — sem estocagem neste período).")
        _passou += 1   # não penaliza verificações inaplicáveis

    # ══════════════════════════════════════════════════════════════════════
    # VERIFICAÇÃO 4 — CALIBRAÇÃO DO PREÇO DO PÁTIO
    # O PRECO_CIF_BU da linha PATIO deve estar preenchido e coerente.
    # Regra: deve ser >= ao custo operacional variável médio das MSs,
    #        pois o sínter carrega pelo menos esse custo fixo.
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{SEP2}")
    print(f"  VERIFICAÇÃO 4 — Calibração do Preço do Pátio")
    print(SEP2)

    _preco_patio_v4 = float(preco_limpo.get("PATIO RETOMADO", 0) or 0)

    # Custo operacional variável médio ponderado das MSs (piso mínimo)
    _custo_op_ms_medio = (
        sum(CUSTO_OP_VAR_MS[ms] * PROD_FIXA_MS[ms] for ms in MAQUINAS)
        / sum(PROD_FIXA_MS.values())
    ) if sum(PROD_FIXA_MS.values()) > 0 else 0

    # Custo de MP médio ponderado das MSs (recalculado pós-solver)
    _custo_mp_ms_medio_total = 0.0
    _prod_total_ms = sum(PROD_FIXA_MS.values())
    for ms in MAQUINAS:
        _custo_mp = sum(
            (sum(x_ms[mp][ms][af].value() or 0 for af in AFS)
             + (y_ms_patio[mp][ms].value() or 0))
            * preco_limpo.get(mp, 0)
            for mp in mps
            if clean_name(mp) != clean_name("Finos de NPO")
        )
        _custo_finos = sum(
            sum((x_dir[npo][af].value() or 0) for af in AFS)
            * Finos.get(npo, 0) * preco_limpo.get(npo, 0)
            * ((x_fino_npo[npo][ms].value() or 0)
               / (sum(x_fino_npo[npo][ms2].value() or 0 for ms2 in MAQUINAS) + 1e-10))
            for npo in npos_list
        )
        _custo_mp_ms_medio_total += (_custo_mp + _custo_finos)

    _custo_sinter_referencia = (
        (_custo_mp_ms_medio_total / _prod_total_ms) + _custo_op_ms_medio
        if _prod_total_ms > 0 else 0
    )

    # Critérios de aprovação:
    # 4a) Preço não pode ser zero se há retomada prevista
    # 4b) Preço deve ser >= custo operacional puro (piso físico)
    # 4c) Preço não pode ser absurdamente maior que o custo de sínter (teto = 2×)
    _v4a = not (_preco_patio_v4 == 0 and DELTA_ESTOQUE < 0)
    _v4b = _preco_patio_v4 >= _custo_op_ms_medio or _preco_patio_v4 == 0
    _v4c = _preco_patio_v4 <= 2.0 * _custo_sinter_referencia or _preco_patio_v4 == 0
    _passou_v4 = _v4a and _v4b

    print(f"\n    Preço do pátio declarado   : {_preco_patio_v4:.2f} US$/t B.U.")
    print(f"    Custo Op.Var. MS (médio)   : {_custo_op_ms_medio:.2f} US$/t BSR  "
          f"← piso mínimo")
    print(f"    Custo total sínter (ref.)  : {_custo_sinter_referencia:.2f} US$/t BSR  "
          f"← referência ideal")

    _resultado(
        "PREÇO DO PÁTIO — Calibração",
        _passou_v4,
        f"{_preco_patio_v4:.2f} US$/t B.U.",
        f"≥ {_custo_op_ms_medio:.2f}  e  ≠ 0 se DELTA < 0",
        "",
        ("⚠️  Preencha PRECO_CIF_BU do PATIO com o CUSTO_SINTER "
         f"da aba KPIS_MS ({_custo_sinter_referencia:.2f} US$/t)."
         if not _passou_v4 else
         "Preço coerente com o custo de produção do sínter.")
    )

    if not _v4c:
        print(f"\n    ⚠️  ALERTA ADICIONAL: preço do pátio ({_preco_patio_v4:.2f}) "
              f"é mais que 2× o custo de referência ({_custo_sinter_referencia:.2f}). "
              f"Verifique se não há duplicidade de custo no input.")

    # ══════════════════════════════════════════════════════════════════════
    # VERIFICAÇÃO 5 — CAPACIDADE FÍSICA DO PÁTIO
    # O estoque final não pode ultrapassar ESTOQUE_MAX_PATIO.
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{SEP2}")
    print(f"  VERIFICAÇÃO 5 — Capacidade Física do Pátio")
    print(SEP2)

    # Estoque final real calculado (em BSR)
    _estoque_final_calc = ESTOQUE_INI + _massa_entrou_bsr_corr - _massa_saiu_bsr

    # Margem de utilização
    _margem_patio = ESTOQUE_MAX_PATIO - _estoque_final_calc
    _utilizacao   = (_estoque_final_calc / ESTOQUE_MAX_PATIO * 100
                     if ESTOQUE_MAX_PATIO > 0 else 0)
    _passou_v5    = _estoque_final_calc <= ESTOQUE_MAX_PATIO + _TOL_MASSA

    print(f"\n    Estoque inicial            : {ESTOQUE_INI:>12,.1f} t BSR")
    print(f"    + Entrada (estocagem)      : {_massa_entrou_bsr_corr:>12,.1f} t BSR")
    print(f"    - Saída   (retomada)       : {_massa_saiu_bsr:>12,.1f} t BSR")
    print(f"    = Estoque final calculado  : {_estoque_final_calc:>12,.1f} t BSR")
    print(f"    Capacidade máxima          : {ESTOQUE_MAX_PATIO:>12,.1f} t BSR")
    print(f"    Margem disponível          : {_margem_patio:>12,.1f} t BSR")
    print(f"    Utilização do pátio        : {_utilizacao:>11.1f} %")

    _resultado(
        "CAPACIDADE — Estoque final ≤ Cap. máxima",
        _passou_v5,
        f"{_estoque_final_calc:,.1f} t BSR  ({_utilizacao:.1f}%)",
        f"≤ {ESTOQUE_MAX_PATIO:,.1f} t BSR",
        "",
        ("⛔ Restrição Cap_Max_Patio_BSR violada — revisar "
         "ESTOQUE_MAX_PATIO ou reduzir DELTA_ESTOQUE no Excel."
         if not _passou_v5 else
         f"Margem de {_margem_patio:,.1f} t disponível no pátio.")
    )

    # ══════════════════════════════════════════════════════════════════════
    # VERIFICAÇÃO 6 — ATRIBUIÇÃO DO GAP FO vs CUSTO_GUSA AO PÁTIO
    # ══════════════════════════════════════════════════════════════════════
    # Garante que a diferença entre a Função Objetivo do solver e o
    # CUSTO_GUSA reportado no KPIS_AF é explicada INTEGRALMENTE pelo
    # custo das matérias-primas que geraram o sínter desviado ao pátio.
    #
    # RACIOCÍNIO FÍSICO-FINANCEIRO:
    #   A FO minimiza o custo total da usina no período — ela paga pelas
    #   MPs de TODO o sínter produzido, inclusive o que foi ao pátio.
    #   O CUSTO_GUSA reporta apenas o custo do gusa produzido HOJE.
    #   O sínter do pátio não gerou gusa hoje — é custo de período futuro.
    #   Após o ajuste do crédito do fino (÷ REND_SINTER), o gap_finos
    #   é zerado e sobra apenas o custo_sinter_patio.
    #   Esta verificação confirma que o gap residual é 100% explicado
    #   pelo custo do sínter estocado — sem nenhum componente espúrio.
    #
    # APROVAÇÃO: gap_residual ≈ custo_patio_calculado  (tolerância ±5%)
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{SEP2}")
    print(f"  VERIFICAÇÃO 6 — Atribuição do Gap FO vs CUSTO_GUSA ao Pátio")
    print(SEP2)

    # ── Recalcula o custo_unit_ms pós-solver (idêntico à Seção C) ─────────
    _cu_ms_v6 = {}
    for ms in MAQUINAS:
        _inv_v6 = sum(
            (sum(x_ms[mp][ms][af].value() or 0 for af in AFS)
             + (y_ms_patio[mp][ms].value() or 0))
            * preco_limpo.get(mp, 0)
            for mp in mps if clean_name(mp) != clean_name("Finos de NPO")
        )
        _finos_npo_v6 = sum(
            sum((x_dir[npo][af].value() or 0) for af in AFS)
            * Finos.get(npo, 0) * preco_limpo.get(npo, 0)
            * ((x_fino_npo[npo][ms].value() or 0)
               / (sum(x_fino_npo[npo][ms2].value() or 0
                      for ms2 in MAQUINAS) + 1e-10))
            for npo in npos_list
        )
        # custo_unit_ms BRUTO (sem crédito do fino) para calcular o gap real
        _cu_ms_v6[ms] = (
            (_inv_v6 + _finos_npo_v6) / PROD_FIXA_MS[ms]
            if PROD_FIXA_MS[ms] > 0 else 0
        ) + CUSTO_OP_VAR_MS[ms]

    # ── Custo BSR desviado ao pátio por MS ────────────────────────────────
    _bsr_patio_ms_v6 = {}
    for ms in MAQUINAS:
        _bsr_patio_ms_v6[ms] = sum(
            (y_ms_patio[mp][ms].value() or 0)
            * (1 - PERDA_FISICA[ms]
               if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO")
               else 1.0)
            * (1 - Umid[mp])
            * (1 - PPC[mp])
            for mp in mps if Aplicacao.get(mp) == "MS"
        )

    _custo_patio_calculado = sum(
        _bsr_patio_ms_v6[ms] * _cu_ms_v6[ms]
        for ms in MAQUINAS
    )

    # ── Gap real FO vs CUSTO_GUSA com crédito do fino aplicado ────────────
    _prod_gusa_v6 = sum(gusa[af].value() or 0 for af in AFS)
    _fo_abs_v6    = pl.value(modelo.objective)
    _fo_por_t_v6  = _fo_abs_v6 / _prod_gusa_v6 if _prod_gusa_v6 > 0 else 0

    # Custo_unit_ms com crédito (÷ REND) — igual ao que agora está na Seção C
    _cu_sinter_af_v6 = {
        ms: _cu_ms_v6[ms] / REND_SINTER[ms]
        if REND_SINTER[ms] > 0 else _cu_ms_v6[ms]
        for ms in MAQUINAS
    }
    _cg_com_credito = 0
    for af in AFS:
        _v_g_v6 = gusa[af].value() or 1

        _c_dir_v6 = (
            sum((x_dir[mp][af].value() or 0)
                * (1 - Finos.get(mp, 0)) * preco_limpo.get(mp, 0)
                for mp in mps if tipo_mp.get(mp) == "NPO")
            + sum((x_dir[mp][af].value() or 0) * preco_limpo.get(mp, 0)
                  for mp in mps if tipo_mp.get(mp) != "NPO")
        )

        _c_sin_v6 = sum(
            sum((x_ms[mp][ms][af].value() or 0)
                * (1 - PERDA_FISICA[ms]
                   if tipo_mp.get(mp) not in ("CARGA_ENERGETICA", "RECICLADO")
                   else 1.0)
                * (1 - Umid[mp])
                * (1 - (PPC_FINOS_EFETIVO
                        if clean_name(mp) == clean_name("Finos de NPO")
                        else PPC[mp]))
                * REND_SINTER[ms]
                * _cu_sinter_af_v6[ms]
                for mp in mps)
            for ms in MAQUINAS
        )

        _cu_patio_medio_v6 = (
            sum(_cu_ms_v6[ms] * PROD_FIXA_MS[ms] for ms in MAQUINAS)
            / sum(PROD_FIXA_MS.values())
        ) if sum(PROD_FIXA_MS.values()) > 0 else 0

        _c_ret_v6 = (
            (z_patio_af[af].value() or 0)
            * (1 - Umid["PATIO RETOMADO"]) * (1 - PPC["PATIO RETOMADO"])
            * _cu_patio_medio_v6
        )

        _c_qz_v6  = (x_quartzo[af].value() or 0) * 40.0
        _c_op_v6  = CUSTO_OP_VAR_AF[af] * _v_g_v6
        _cg_com_credito += _c_dir_v6 + _c_sin_v6 + _c_ret_v6 + _c_qz_v6 + _c_op_v6

    _cg_credito_por_t   = _cg_com_credito / _prod_gusa_v6 if _prod_gusa_v6 > 0 else 0
    _gap_residual_abs   = _fo_abs_v6 - _cg_com_credito
    _gap_residual_por_t = _gap_residual_abs / _prod_gusa_v6 if _prod_gusa_v6 > 0 else 0
    _custo_patio_por_t  = _custo_patio_calculado / _prod_gusa_v6 if _prod_gusa_v6 > 0 else 0

    if abs(_gap_residual_abs) > 1e-6:
        _pct_explicado = (_custo_patio_calculado / _gap_residual_abs) * 100
    else:
        _pct_explicado = 100.0

    _TOL_PCT   = 5.0
    _passou_v6 = abs(_pct_explicado - 100.0) <= _TOL_PCT

    # ── Impressão ──────────────────────────────────────────────────────────
    print(f"\n    FO solver                    : {_fo_por_t_v6:>10.4f} US$/t gusa")
    print(f"    CUSTO_GUSA (com crédito fino): {_cg_credito_por_t:>10.4f} US$/t gusa")
    print(f"    Gap residual                 : {_gap_residual_por_t:>10.4f} US$/t gusa")
    print(f"    {'─'*55}")
    print(f"    Decomposição do gap residual por MS:")
    for ms in MAQUINAS:
        _c_pat_ms_t = _bsr_patio_ms_v6[ms] * _cu_ms_v6[ms] / _prod_gusa_v6 if _prod_gusa_v6 > 0 else 0
        print(f"      {ms}: {_bsr_patio_ms_v6[ms]:>10,.1f} t BSR × "
              f"{_cu_ms_v6[ms]:.2f} US$/t = "
              f"{_c_pat_ms_t:.4f} US$/t gusa")
    print(f"    {'─'*55}")
    print(f"    Custo pátio calculado        : {_custo_patio_por_t:>10.4f} US$/t gusa")
    print(f"    % do gap explicado pelo pátio: {_pct_explicado:>10.2f} %")
    print(f"    Tolerância                   : {'100% ± ' + str(_TOL_PCT) + '%':>10}")

    if DELTA_ESTOQUE <= 0:
        print(f"\n    ⏭️  Não aplicável — sem sínter desviado ao pátio neste cenário.")
        _passou += 1
    else:
        _resultado(
            "ATRIBUIÇÃO — Gap 100% explicado pelo custo do pátio",
            _passou_v6,
            f"{_pct_explicado:.2f}% do gap = custo do pátio",
            f"100% ± {_TOL_PCT}%",
            "",
            (f"✅ O gap de {_gap_residual_por_t:.4f} US$/t é integralmente explicado "
             f"pelo custo das MPs que geraram os {DELTA_ESTOQUE:,.0f} t BSR estocados."
             if _passou_v6 else
             f"⚠️  Há {abs(_pct_explicado-100):.2f}% de gap não explicado pelo pátio. "
             f"Revisar custo_unit_ms ou REND_SINTER.")
        )

    # ══════════════════════════════════════════════════════════════════════
    # PLACAR FINAL
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{SEP}")
    _total = _passou + _falhou
    _icone_final = "✅" if _falhou == 0 else "❌"
    print(f"\n  {_icone_final}  PLACAR FINAL DA HOMOLOGAÇÃO DO PÁTIO")
    print(f"\n     Verificações aprovadas : {_passou} / {_total}")
    print(f"     Verificações reprovadas: {_falhou} / {_total}")

    if _falhou == 0:
        print(f"\n     ✅ Todas as verificações passaram.")
        print(f"        O cenário de pátio está matematicamente consistente.")
        print(f"        Pode prosseguir para o módulo de escrita reversa.")
    else:
        print(f"\n     ❌ {_falhou} verificação(ões) reprovada(s).")
        print(f"        Revise os itens marcados com ❌ acima antes de")
        print(f"        prosseguir para a escrita reversa e o relatório Excel.")

    print(f"\n{SEP}\n")

# ======================================================
# FIM DO MÓDULO 5.5 — HOMOLOGAÇÃO DO PÁTIO DE SÍNTER
# ======================================================

FileNotFoundError: Arquivo 'INPUT_MASTER_AF.xlsx' não encontrado.
Coloque o Excel na mesma pasta do notebook e tente novamente.